# Target binding for streaming video QA

Open this notebook on the RTX 4090 AutoDL container and choose **Restart Kernel → Run All**.
The pinned environment and model under `/root/autodl-tmp/hermes_causal_pilot` are verified before reuse.
This run has its own work directory, `/root/autodl-tmp/hermes_target_binding`.

**60 frozen clips, 300 questions, 1,800 selector outcomes.** The primary comparison uses the
269 questions with at least two complete four-frame windows. All windows lie within the latest
20 causal samples on the 0.5 FPS grid. One answer per window is reused across selectors that
choose it. No training, extra model, human labels, answer choices, or gold answers are used
to select windows. The original multiple-choice prompt remains the answer prompt.

| Selector | Rule |
|---|---|
| Recent | Newest four frames |
| Previous | Immediately preceding four frames, or recent if unavailable |
| Global relevance | Highest question-to-window relevance margin |
| Target only | Highest target-identification margin |
| Mean | Highest mean of target and target-specific evidence margins |
| Conjunction | Highest minimum of the same two margins |

**Primary contrast** is conjunction versus mean. Both use the same probes and candidates.
The report shows paired video-bootstrap intervals, wins and losses, window switches,
an answer oracle for the fixed candidate set, and a component-time estimate per policy.
The text-rich slice and other baseline contrasts are secondary.

The runner checks selected video sizes from four ZIP indexes before downloading. It requires
space for the selected members plus 4 GiB for working files. It uses HTTP byte ranges and
retains completed downloads and checkpointed questions. Leave `MAX_HOURS = 0.0` for the full
run. To resume, use the same notebook, GPU type and work directory. Transfers depend on
the container connection. Watch for `MEDIA_PLAN`, `VIDEO_READY`, and `VIDEO_BLOCK_DONE`.

At **TARGET_BINDING_COMPLETE**, download the evidence ZIP and save the executed notebook.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_target_binding')
REUSE = Path('/root/autodl-tmp/hermes_causal_pilot')
MAX_HOURS = 0.0                 # soft post-model budget; 0 disables; rerun resumes
REPO_URL = 'https://github.com/md-hameem/HERMES-reproduction.git'
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
MODEL_SHA = '74dd0bf867a4cda7950c17663794267c60cf4b40'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_TARGET_BINDING_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
assert MAX_HOURS >= 0
REPO = WORK/'repository'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'stage_runner.py'
print('Plan: 1,800 selector outcomes on 60 frozen clips')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')


## 1. Prepare pinned source, environment and model


In [ ]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    env.update(HF_HOME=str((REUSE if (REUSE/'cache/huggingface').is_dir() else WORK)/'cache/huggingface'), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    assert free >= 18, f'Need at least 18 GiB free on data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 15000, 'Use a GPU with at least 15 GiB VRAM'
    # Never print the environment or proxy values.


In [ ]:
SOURCE_HASHES = {'inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'scripts/build_failure_pilot.py': 'c81681e7f148702d07d1a2cc193a1ec32cf4f89778a911d92283088050ed7835', 'video_qa/base.py': '795cf98e8d1125ea934dc2c9744ba1bbd3b29701f17aae2d935f382c7343b2d5', 'video_qa/hermes_vqa.py': 'f25bac6cd19db7148e7fa39cc9d2e96fb40d1f14709778f2490023a65dddc95b', 'data/streamingbench/streamingbench_realtime.json': '6a2969953e11db44f5f8584d0d9fd3ec47c41dacf63807ef0003ccf446869f73'}
if not VALIDATE_ONLY:
    candidate_repo = REUSE/'repository'
    if (candidate_repo/'.git').is_dir():
        revision = probe(['git', 'rev-parse', 'HEAD'], ENV, candidate_repo)
        if revision.returncode == 0 and revision.stdout.strip() == BASE_REVISION and all(
            (candidate_repo/rel).is_file() and
            hashlib.sha256((candidate_repo/rel).read_bytes().replace(b'\r\n', b'\n')).hexdigest() == expected
            for rel, expected in SOURCE_HASHES.items()
        ):
            REPO = candidate_repo
            ENV['PYTHONDONTWRITEBYTECODE'] = '1'
            print('PINNED_SOURCE_REUSE_PASS', BASE_REVISION, flush=True)
if not VALIDATE_ONLY and REPO == WORK/'repository':
    if not (REPO/'.git').is_dir():
        assert not REPO.exists() or not list(REPO.iterdir()), 'Non-Git source directory has files; use a new WORK'
        REPO.mkdir(parents=True, exist_ok=True)
        logged(['git', 'init', REPO], 'clone.log', ENV)
        logged(['git', 'remote', 'add', 'origin', REPO_URL], 'clone.log', ENV, REPO)
    remote = probe(['git', 'remote', 'get-url', 'origin'], ENV, REPO)
    assert remote.returncode == 0 and remote.stdout.strip() == REPO_URL, 'Unexpected source repository origin'
    dirty = probe(['git', 'status', '--porcelain', '--untracked-files=no'], ENV, REPO)
    assert dirty.returncode == 0 and not dirty.stdout.strip(), 'Pinned checkout has local edits; preserve them and use a new WORK'
    if probe(['git', 'cat-file', '-e', BASE_REVISION+'^{commit}'], ENV, REPO).returncode:
        for attempt in range(4):
            try:
                logged(['git', 'fetch', '--filter=blob:none', '--depth', '1', 'origin', BASE_REVISION], 'clone.log', ENV, REPO)
                break
            except RuntimeError:
                if attempt == 3:
                    raise
                print('GITHUB_FETCH_RETRY', attempt+1, flush=True)
                time.sleep(3*(attempt+1))
    logged(['git', 'sparse-checkout', 'init', '--cone'], 'clone.log', ENV, REPO)
    logged(['git', 'sparse-checkout', 'set', 'inference', 'video_qa', 'scripts', 'data/streamingbench'], 'clone.log', ENV, REPO)
    for attempt in range(4):
        try:
            logged(['git', 'checkout', '--detach', BASE_REVISION], 'clone.log', ENV, REPO)
            break
        except RuntimeError:
            if attempt == 3:
                raise
            print('GITHUB_CHECKOUT_RETRY', attempt+1, flush=True)
            time.sleep(3*(attempt+1))
    assert probe(['git', 'rev-parse', 'HEAD'], ENV, REPO).stdout.strip() == BASE_REVISION
    for rel, expected in SOURCE_HASHES.items():
        assert hashlib.sha256((REPO/rel).read_bytes().replace(b'\r\n', b'\n')).hexdigest() == expected, 'Pinned source mismatch: '+rel
    print('PINNED_SOURCE_PASS', BASE_REVISION, flush=True)


In [ ]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
if not VALIDATE_ONLY:
    candidate_python = REUSE/'venv/bin/python'
    if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
        PYTHON = candidate_python
        print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)
    PIP_ENV = package_environment(ENV)
    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                             'https://pypi.tuna.tsinghua.edu.cn/simple'], PIP_ENV)
            if attempt.returncode:
                logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                        'https://mirrors.aliyun.com/pypi/simple'], 'setup.log', PIP_ENV)
        UV = [sys.executable, '-m', 'uv']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)
        def install(packages, extra=()):
            install_packages(UV, PYTHON, packages, ENV, extra)
        install(['torch @ https://download-r2.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl'])
        install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
                 'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
                 'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
                 'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops', 'psutil', 'pandas'])
        # Download GitHub source with turbo, then install the local archive using
        # direct package networking for any isolated build dependencies.
        tf_archive = WORK/'downloads'/f'transformers-{TF_SHA}.zip'
        tf_archive.parent.mkdir(parents=True, exist_ok=True)
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path
dest = Path(sys.argv[2]); temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3: raise
        time.sleep(3 * (attempt + 1))
"""
            logged([sys.executable, '-c', fetch_tf,
                    f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}', tf_archive],
                   'source_download.log', ENV)
        install([str(tf_archive)], ['--no-deps'])
    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO/'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)


In [ ]:
MODEL_DOWNLOAD = r"""
import json, sys
from pathlib import Path
from huggingface_hub import snapshot_download
from safetensors import safe_open
path = Path(snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], cache_dir=sys.argv[3],
    allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.vocab', 'tokenizer.*', 'merges.txt'],
    ignore_patterns=['onnx/*', '*.onnx', '*.bin', '*.msgpack'], max_workers=4))
assert path.name == sys.argv[2] and (path/'config.json').is_file()
files = list(path.glob('*.safetensors'))
assert files
for file in files:
    with safe_open(str(file), framework='pt', device='cpu') as handle:
        assert list(handle.keys())
Path(sys.argv[4]).write_text(json.dumps(str(path)), encoding='utf-8')
print('MODEL_REVISION_PASS', path.name)
"""
if not VALIDATE_ONLY:
    pointer = WORK/'model_path.json'
    failures = []
    for host in ('https://hf-mirror.com', 'https://huggingface.co'):
        try:
            logged([PYTHON, '-c', MODEL_DOWNLOAD, MODEL_ID, MODEL_SHA,
                    Path(ENV['HF_HOME'])/'hub', pointer], 'model_download.log', {**ENV, 'HF_ENDPOINT': host})
            break
        except RuntimeError as exc:
            failures.append(type(exc).__name__)
            print('Model endpoint failed; trying fallback', flush=True)
    else:
        raise RuntimeError('Model download failed on both endpoints; rerun to resume cached files')
    MODEL_PATH = Path(json.loads(pointer.read_text(encoding='utf-8')))
    assert MODEL_PATH.name == MODEL_SHA


## 2. Materialize trusted runner sources and audited reference data


In [ ]:
CONFIRMATION_BASE_SOURCE = r'''"""Run the frozen clip-disjoint StreamingBench comparison."""
import argparse
import hashlib
import json
import os
import random
import re
import statistics
import sys
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote
from zipfile import BadZipFile, ZipFile

try:
    from provenance import short_base as q
    from provenance import selection_base as selection
except ModuleNotFoundError as exc:
    if exc.name != "provenance":
        raise
    import autodl_short_query as q
    import autodl_construction_selection as selection

s, d, p = q.s, q.d, q.p
ARMS = ("compact_recent4", "latest_two_short_readout", "all_history_short_readout")
ARTIFACT = "HERMES_short_query_confirmation_evidence.zip"
SOURCE_PATHS = (*q.SOURCE_PATHS, "provenance/short_base.py", "provenance/selection_base.py")


def acquire_media(shards, work):
    """Read only listed ZIP members, check CRC and hash each realized input."""
    import requests

    output = work / "videos"
    output.mkdir(exist_ok=True)
    receipts = work / "provenance/media"
    receipts.mkdir(parents=True, exist_ok=True)
    assets = []
    for shard in shards:
        expected = [c["sample_id"] for c in shard["clips"]]
        receipt = receipts / f"{shard['shard']}.json"
        old = json.loads(receipt.read_text()) if receipt.exists() else {}
        valid = {}
        for clip in old.get("videos", []):
            path = output / f"sample_{clip['sample_id']}_real.mp4"
            if (clip["sample_id"] in expected and path.is_file() and path.stat().st_size == clip["bytes"]
                    and p.digest(path) == clip["sha256"]):
                valid[clip["sample_id"]] = dict(clip, path=str(path))
        received = old.get("transferred_bytes", 0)
        if set(valid) != set(expected):
            for host in ("huggingface.co", "hf-mirror.com"):
                url = f"https://{host}/datasets/mjuicem/StreamingBench/resolve/main/{quote(shard['archive'])}?download=true"
                try:
                    with requests.Session() as session:
                        accessor = p.HTTPRangeFile(url, shard["archive_bytes"], session)
                        try:
                            with ZipFile(accessor) as archive:
                                for clip in shard["clips"]:
                                    sid = clip["sample_id"]
                                    if sid in valid:
                                        continue
                                    item, = p.extract_members(archive, [dict(video_path=clip["video_basename"])], output)
                                    valid[sid] = item
                                    p.atomic_json(receipt, dict(shard=shard["shard"], archive=shard["archive"],
                                        transferred_bytes=received + accessor.bytes_received, videos=list(valid.values())))
                        finally:
                            received += accessor.bytes_received
                    break
                except (OSError, ValueError, BadZipFile, p.RangeUnsupported, requests.RequestException) as exc:
                    if host == "hf-mirror.com":
                        raise RuntimeError("Selected-member download failed on both endpoints") from exc
                    print(f"MEDIA_RANGE_RETRY {shard['shard']} {type(exc).__name__}", flush=True)
        if set(valid) != set(expected):
            raise ValueError("Frozen media set incomplete")
        entries = [valid[i] for i in expected]
        p.atomic_json(receipt, dict(shard=shard["shard"], archive=shard["archive"],
                                   transferred_bytes=received, videos=entries))
        assets.extend(entries)
    p.atomic_json(work / "asset_manifest.json", dict(mode="range_members_and_sha256", videos=assets))
    return {a["sample_id"]: a for a in assets}


def videos_for_manifest(manifest, repo):
    path = repo / manifest["annotation_path"]
    raw = path.read_bytes().replace(b"\r\n", b"\n")
    actual = hashlib.sha256(raw).hexdigest()
    if actual != manifest["annotation_sha256"]:
        raise ValueError(f"Annotation digest differs after newline normalization: {actual}")
    annotation = {p.sample_id(v): v for v in json.loads(raw)}
    videos = []
    for shard in manifest["shards"]:
        for clip in shard["clips"]:
            sid = clip["sample_id"]
            video = dict(annotation[sid], sample_id=sid)
            if (video["video_id"] != clip["video_id"] or len(video["conversations"]) != clip["question_count"]
                    or Path(video["video_path"]).name != clip["video_basename"]):
                raise ValueError("Frozen clip identity differs")
            if any(len(qa["choices"]) > 8 or qa["choices"].count(qa["answer"]) != 1 for qa in video["conversations"]):
                raise ValueError(f"Ambiguous multiple-choice gold for sample {sid}")
            videos.append(video)
    if len(videos) != 30 or len({v["sample_id"] for v in videos}) != 30:
        raise ValueError("Invalid confirmation video count")
    if sum(len(v["conversations"]) for v in videos) != 150:
        raise ValueError("Invalid confirmation question count")
    return videos


def plan_sample_grid(frame_timestamps, video):
    """Sample the latest frame at or before each 2-second grid point."""
    import math
    import numpy as np

    frame_timestamps = np.asarray(frame_timestamps, dtype=np.float64)
    if frame_timestamps.ndim != 2 or frame_timestamps.shape[1] != 2 or len(frame_timestamps) == 0:
        raise ValueError("Invalid source timestamps")
    relative = frame_timestamps - frame_timestamps[0, 0]
    if not np.isfinite(relative).all() or np.any(np.diff(relative[:, 0]) < 0):
        raise ValueError("Nonmonotone source timestamps")
    end = max(float(qa["end_time"]) for qa in video["conversations"])
    grid = np.arange(math.ceil(end * p.FPS)) / p.FPS
    if not len(grid) or grid[-1] > relative[-1, 1] + 1e-6:
        raise ValueError("Frozen query extends beyond the sampled source")
    indices = np.searchsorted(relative[:, 0], grid, side="right") - 1
    if indices[0] < 0 or np.any(np.diff(indices) <= 0):
        raise ValueError("Source cannot represent the frozen grid without repeated frames")
    return [dict(sampled_index=i, source_frame=int(j), start_s=float(relative[j, 0]),
                 end_s=float(relative[j, 1])) for i, j in enumerate(indices)]


def prepare_confirmation_features(model, video, work, identity):
    """Encode only the union of question windows while preserving chunk boundaries."""
    import math
    import torch
    from decord import VideoReader

    sid = video["sample_id"]
    directory = work / "features" / f"sample_{sid}"
    directory.mkdir(parents=True, exist_ok=True)
    marker, frame_path = directory / "complete.json", directory / "sampled_timestamps.csv"
    if marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if saved["identity"] == identity and p.digest(frame_path) == saved["frame_times_sha256"]:
            d.verify_feature_files(directory, saved)
            print(f"FEATURE_CACHE reuse sample={sid}", flush=True)
            return directory, saved
        raise d.RunConflict("Existing confirmation feature cache has a different identity")
    reader = VideoReader(video["video_path"], num_threads=1)
    times = plan_sample_grid(reader.get_frame_timestamp(list(range(len(reader)))), video)
    p.write_csv(frame_path, times)
    needed = set()
    for qa in video["conversations"]:
        end = math.ceil(float(qa["end_time"]) * p.FPS)
        needed.update(range(max(0, end - 20), end))
    chunks = []
    for original in p.chunk_plan(video, len(times)):
        group = [i for i in range(original["start"], original["stop"]) if i in needed]
        if not group:
            continue
        if group != list(range(group[0], group[-1] + 1)):
            raise ValueError("Non-contiguous selection inside an original chunk")
        p.report_progress("confirmation_features", sample_id=sid, sampled_start=group[0], sampled_stop=group[-1]+1)
        frames = torch.from_numpy(reader.get_batch([times[i]["source_frame"] for i in group]).asnumpy())
        pixels = model.processor.video_processor(frames.permute(0, 3, 1, 2), return_tensors="pt").pixel_values_videos
        features = model.get_video_features(pixels.to(model.device, model.dtype)).detach().cpu().contiguous()
        target = directory / f"chunk_{len(chunks):04d}.pt"
        temp = target.with_suffix(".tmp")
        torch.save(features, temp)
        os.replace(temp, target)
        chunks.append(dict(qa_index=original["qa_index"], start=group[0], stop=group[-1]+1, tokens=features.shape[1],
                           sha256=p.digest(target), file=target.name))
    saved = dict(identity=identity, sample_id=sid, chunks=chunks, frame_times_sha256=p.digest(frame_path),
                 sampling="0.5 FPS grid with source frame at or before grid timestamp")
    d.verify_feature_files(directory, saved)
    p.atomic_json(marker, saved)
    return directory, saved


def construct_block(model, analyzer, bank, meta, video, qi, history, recent, warmup=False):
    p.sync()
    before = time.perf_counter()
    p.reset(model)
    s.ingest(model, bank, history + recent, meta)
    original = s.snapshot(model)
    start = original["prompt"]
    logical_start = start + 196 * len(history)
    prefix = s.prefix_state(original, logical_start)
    p.sync()
    prep_time = time.perf_counter() - before
    qa = video["conversations"][qi]
    prompt = analyzer.format_mcqa_prompt(qa["question"], qa["choices"])
    sid = video["sample_id"]
    order = list(ARMS)
    rng = random.Random(p.SEED + 31 * sid + qi)
    rng.shuffle(order)
    rows = {}
    for arm in order:
        before = time.perf_counter()
        if arm == ARMS[0]:
            p.reset(model)
            groups = s.ingest(model, bank, recent, meta)
            encoded, stop = s.snapshot(model), start
            chosen = []
        elif not history:
            encoded, groups, stop, chosen = original, s.c.input_groups(recent, "original", meta), start, []
        else:
            chosen = list(range(max(0, len(history) - 2) if arm == ARMS[1] else 0, len(history)))
            encoded, groups, stop = selection.construction(model, bank, meta, prefix, recent, chosen, logical_start)
        p.sync()
        construction_time = time.perf_counter() - before
        before = time.perf_counter()
        state = q.make_short(encoded, start, stop) if stop > start else encoded
        p.sync()
        extraction_time = time.perf_counter() - before if stop > start else 0.0
        if warmup:
            q.position_trace(model, state, prompt, (0, 0))
        answer, answer_time, trace = q.position_trace(model, state, prompt, (0, 0))
        choice = analyzer.extract_characters_regex(answer)
        row = d.question_identity(video, qi, arm)
        row.update(pred_choice=choice, pred_answer=answer.replace("\n", ""), correct=int(choice == row["gold_choice"]),
                   selected_history_indices=[history[i] for i in chosen], recent_indices=recent,
                   recent_groups=groups, query_kv_bytes=q.cache_bytes(state), position_trace=trace,
                   answer_s=answer_time, construction_s=construction_time, extraction_s=extraction_time)
        rows[arm] = row
        model.clear_cache()
    if len({r["query_kv_bytes"] for r in rows.values()}) != 1:
        raise ValueError("Answer K/V budgets differ")
    expected_start = logical_start + 196 * len(recent)
    if any(rows[a]["position_trace"][0]["logical_start"] != expected_start for a in ARMS[1:]):
        raise ValueError("Contextual readout changed recent positions")
    if not history and len({r["pred_choice"] for r in rows.values()}) != 1:
        raise ValueError("Empty-history control differs")
    return [rows[a] for a in ARMS], dict(history_indices=history, recent_indices=recent,
                                      answer_order=order, shared_reference_preparation_s=prep_time)


def paired_interval(deltas, seed=2024, draws=10000):
    """Question-weighted point estimate and video-resampled percentile interval."""
    rng = random.Random(seed)
    videos = list(deltas.values())
    if not videos:
        raise ValueError("No paired videos")
    wins = [(sum(v), len(v)) for v in videos]
    values = []
    for _ in range(draws):
        choices = rng.choices(wins, k=len(wins))
        values.append(100 * sum(x[0] for x in choices) / sum(x[1] for x in choices))
    values.sort()
    total, count = (sum(x[i] for x in wins) for i in (0, 1))
    loo = [100 * (total - x) / (count - n) for x, n in wins if count > n]
    return dict(difference_pp=100 * total / count, ci_low_pp=values[int(.025 * draws)],
                ci_high_pp=values[min(draws - 1, int(.975 * draws))],
                leave_one_video_min_pp=min(loo) if loo else None, leave_one_video_max_pp=max(loo) if loo else None)


def export(rows, work, *, complete):
    p.write_csv(work / "exports/answers.csv", rows)
    groups, summary = {}, []
    for r in rows:
        groups.setdefault((r["sample_id"], r["qa_index"]), {})[r["method"]] = r
    for arm in ARMS:
        subset = [g[arm] for g in groups.values()]
        summary.append(dict(arm=arm, answers=len(subset), correct=sum(r["correct"] for r in subset),
                            median_query_kv_bytes=statistics.median(r["query_kv_bytes"] for r in subset),
                            median_construction_s=statistics.median(r["construction_s"] for r in subset),
                            median_extraction_s=statistics.median(r["extraction_s"] for r in subset),
                            median_answer_s=statistics.median(r["answer_s"] for r in subset)))
    p.write_csv(work / "exports/summary.csv", summary)
    lines = ["# Frozen short query-cache comparison", "", "| Arm | Correct | Answers | Median query KV bytes |",
             "|---|---:|---:|---:|"]
    lines.extend(f"| {r['arm']} | {r['correct']} | {r['answers']} | {r['median_query_kv_bytes']} |" for r in summary)
    lines.extend(["", "Clip-disjoint within StreamingBench. Original source overlap is unverified."])
    if complete:
        deltas = defaultdict(list)
        for (sid, _), g in groups.items():
            deltas[sid].append(g[ARMS[2]]["correct"] - g[ARMS[0]]["correct"])
        effects = paired_interval(deltas)
        values = [v for group in deltas.values() for v in group]
        p.atomic_json(work / "exports/paired_primary.json", dict(
            candidate=ARMS[2], baseline=ARMS[0], wins=values.count(1), losses=values.count(-1),
            videos=len(deltas), questions=len(values), **effects))
        lines.extend(["", f"Primary paired difference: {effects['difference_pp']:.2f} pp "
                      f"(95% video bootstrap {effects['ci_low_pp']:.2f}, {effects['ci_high_pp']:.2f})."])
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def execute(args):
    import torch
    import transformers

    if (args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128"
            or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available()):
        raise ValueError("Pinned model/runtime required")
    path = args.work / "reference.json"
    manifest = json.loads(path.read_text(encoding="utf-8"))
    videos = videos_for_manifest(manifest, args.repo)
    config = dict(manifest_sha256=p.digest(path), sources={n: p.digest(args.work/n) for n in SOURCE_PATHS},
                  model=p.MODEL_SHA, torch=torch.__version__, gpu=torch.cuda.get_device_name(0), arms=ARMS)
    fingerprint = d.fingerprint(config)
    config_path = args.work / "exports/run_config.json"
    if config_path.exists() and json.loads(config_path.read_text())["config_hash"] != fingerprint:
        raise d.RunConflict("Confirmation configuration changed; choose a new WORK")
    p.atomic_json(config_path, dict(**config, config_hash=fingerprint))
    assets = acquire_media(manifest["shards"], args.work)
    for video in videos:
        video["video_path"] = assets[video["sample_id"]]["path"]
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA

    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    torch.manual_seed(p.SEED)
    p.report_progress("loading_confirmation_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows, started, warmed = [], time.perf_counter(), False
    with torch.inference_mode():
        for i, video in enumerate(videos):
            if i and args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                break
            sid = video["sample_id"]
            source_sha = assets[sid]["sha256"]
            directory, meta = prepare_confirmation_features(
                model, video, args.work, d.fingerprint(dict(config=fingerprint, source=source_sha)))
            video["timestamps"] = p.read_csv(directory / "sampled_timestamps.csv")
            p.atomic_json(args.work / "provenance" / f"features_{sid}.json", dict(metadata=meta, timestamps=video["timestamps"]))
            bank = s.c.FeatureBank(directory, meta)
            if i == 0:
                p.preflight(model, video, args.work)
            for qi in range(len(video["conversations"])):
                h, r = s.split_frames(video, qi)
                key = d.fingerprint(dict(config=fingerprint, feature=d.fingerprint(meta), sample=sid, qa=qi, h=h, r=r))
                dest = args.work / "checkpoints" / f"sample_{sid}_q{qi}.json"
                if dest.exists():
                    saved = json.loads(dest.read_text())
                    if saved["key"] != key or d.fingerprint(saved["rows"]) != saved["rows_sha256"]:
                        raise d.RunConflict("Confirmation checkpoint differs")
                    block = saved["rows"]
                else:
                    block, prep = construct_block(model, analyzer, bank, meta, video, qi, h, r, warmup=not warmed)
                    warmed = True
                    p.atomic_json(args.work / "preparation" / dest.name, prep)
                    p.atomic_json(dest, dict(key=key, rows=block, rows_sha256=d.fingerprint(block)))
                rows.extend(block)
            del bank
            export(rows, args.work, complete=False)
            print(f"VIDEO_BLOCK_DONE {i+1}/30 questions={len(rows)//3}/150", flush=True)
    status = "complete" if len(rows) == 450 else "partial_time_budget"
    p.atomic_json(args.work / "exports/completion.json", dict(status=status, main_answers=len(rows),
                  main_questions=len(rows)//3, elapsed_post_model_s=time.perf_counter()-started, config_hash=fingerprint))
    export(rows, args.work, complete=status == "complete")
    p.report_progress(status, main_questions=len(rows)//3)
    package(args.work, status)
    print(f"SHORT_QUERY_CONFIRMATION_{status.upper()}", flush=True)


def package(work, status):
    previous = s.ARTIFACT
    try:
        s.ARTIFACT = ARTIFACT
        s.package(work, status)
    finally:
        s.ARTIFACT = previous


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for n in ("repo", "work", "model", "reuse"):
        parser.add_argument("--" + n, type=lambda value: Path(value).resolve(), required=True)
    parser.add_argument("--max-hours", type=float, default=2.0)
    args = parser.parse_args()
    d.validate_workspaces(args.work, args.reuse)
    if args.max_hours < 0:
        parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work / "exports/failure.json", dict(type=type(exc).__name__, detail=str(exc)[-1600:]))
            p.atomic_json(args.work / "exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(CONFIRMATION_BASE_SOURCE)
assert hashlib.sha256(CONFIRMATION_BASE_SOURCE.encode()).hexdigest() == '6fa34396ed012783b7a8af1ae9ab1e56d7cc224f37f1c02c65e63d112256de1d'
STAGE_BASE_SOURCE = r'''"""Factor history visibility during recent encoding and question/answer readout.

No production inference files are changed. Masking H during R construction is
validated against physically independent R construction at identical RoPE positions.
"""
import argparse
import contextlib
import copy
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZIP_DEFLATED,ZipFile

try:
    import stage_context_utils as c
except ModuleNotFoundError as exc:
    if exc.name != "stage_context_utils": raise
    import autodl_context_diagnostic as c

d,p=c.d,c.p
ARMS=("A_history_both","B_independent_recent","C_hide_history_readout","D_independent_hide_readout","compact_recent4")
ARTIFACT="HERMES_stage_attribution_evidence.zip"


def split_frames(video,qi,long_prefix=False):
    end=d.prefix_end(video,qi)
    selected=list(range(0 if long_prefix else max(0,end-20),end))
    recent=selected[-4:]
    history=selected[:-len(recent)] if recent else selected
    d.validate_frames(selected,video,qi,video["timestamps"])
    return history,recent


def cache_length(cache):
    if cache is None:return 0
    if hasattr(cache,"get_seq_length"):return int(cache.get_seq_length())
    return int(cache[0][0].shape[2])


def causal_mask(batch,q,past,dtype,device,blocked_start,blocked_stop):
    import torch
    if not 0<=blocked_start<=blocked_stop<=past:
        raise ValueError("History mask span must lie entirely in physical past cache")
    mask=torch.zeros((batch,1,q,past+q),dtype=dtype,device=device)
    minimum=torch.finfo(dtype).min
    mask[:,:,:,blocked_start:blocked_stop]=minimum
    future=torch.ones((q,q),dtype=torch.bool,device=device).triu(1)
    mask[:,:,:,past:].masked_fill_(future,minimum)
    return mask


class HistoryMask:
    """Apply one explicit causal+H mask to every LM call, including all decode steps."""
    def __init__(self,model,start,stop,force=False):
        self.model,self.start,self.stop,self.force=model,start,stop,force
        self.events=[];self.handle=None

    def hook(self,module,args,kwargs):
        if args:raise ValueError("Expected named LM inputs for audited mask")
        if kwargs.get("attention_mask") is not None:raise ValueError("Refuse to overwrite another attention mask")
        value=kwargs.get("inputs_embeds")
        if value is None:value=kwargs["input_ids"]
        batch,q=value.shape[:2]
        past=cache_length(kwargs.get("past_key_values"))
        mask=causal_mask(batch,q,past,self.model.dtype,self.model.device,self.start,self.stop)
        kwargs["attention_mask"]=mask
        self.events.append(dict(query_tokens=q,past_tokens=past,blocked_start=self.start,blocked_stop=self.stop,
                               shape=list(mask.shape),policy="causal plus hidden historical physical span"))
        return args,kwargs

    def __enter__(self):
        if self.stop>self.start or self.force:
            self.handle=self.model.language_model.register_forward_pre_hook(self.hook,with_kwargs=True)
        return self

    def __exit__(self,*exc):
        if self.handle is not None:self.handle.remove()
        return False


def snapshot(model):
    return dict(kv=[(k.clone(),v.clone()) for k,v in model.kv_cache],positions=[x.clone() for x in model._position_ids_cache],
                prompt=model.visual_start_idx)


def restore(model,state):
    model._layer_position_ids.clear()
    model.kv_cache=[(k.clone(),v.clone()) for k,v in state["kv"]]
    model._position_ids_cache=[x.clone() for x in state["positions"]]
    model.visual_start_idx=state["prompt"]
    model.conv_history=[]
    model.token_activity_cache=[None]*model.num_layers


def prefix_state(state,length):
    return dict(kv=[(k[:,:,:length].contiguous().clone(),v[:,:,:length].contiguous().clone()) for k,v in state["kv"]],
                positions=[x[:length].clone() for x in state["positions"]],prompt=state["prompt"])


def drop_history(state,start,stop):
    import torch
    n=state["kv"][0][0].shape[2]
    keep=torch.cat((torch.arange(start,device=state["positions"][0].device),torch.arange(stop,n,device=state["positions"][0].device)))
    return dict(kv=[(k.index_select(2,keep),v.index_select(2,keep)) for k,v in state["kv"]],
                positions=[x.index_select(0,keep) for x in state["positions"]],prompt=state["prompt"])


def assert_state(model,state):
    import torch
    for (k,v),(expected_k,expected_v),pos,expected_pos in zip(model.kv_cache,state["kv"],model._position_ids_cache,state["positions"]):
        torch.testing.assert_close(k,expected_k,atol=0,rtol=0)
        torch.testing.assert_close(v,expected_v,atol=0,rtol=0)
        torch.testing.assert_close(pos,expected_pos,atol=0,rtol=0)


def ingest(model,bank,indices,meta,logical_start=None):
    groups=c.input_groups(indices,"original",meta)
    for group in groups:
        if logical_start is None:
            p.ingest_features(model,bank.get(group),len(group))
        else:
            # Native independent reference: physical cache omits H while R's logical positions retain its gap.
            original=model._get_next_start_pos_per_layer
            try:
                model._get_next_start_pos_per_layer=lambda:[logical_start]*model.num_layers
                p.ingest_features(model,bank.get(group),len(group))
            finally:model._get_next_start_pos_per_layer=original
            logical_start+=len(group)*196
    return groups


def prepare_states(model,bank,meta,history,recent):
    import torch
    p.reset(model)
    full_groups=ingest(model,bank,history+recent,meta)
    a=snapshot(model)
    d.assert_fresh_cache(model,len(history)+len(recent))
    prompt=model.visual_start_idx
    stop=prompt+len(history)*196
    if history:
        restore(model,prefix_state(a,stop))
        with HistoryMask(model,prompt,stop) as mask:
            recent_groups=ingest(model,bank,recent,meta)
        b=snapshot(model)
        d.assert_fresh_cache(model,len(history)+len(recent))
        for (ak,av),(bk,bv) in zip(a["kv"],b["kv"]):
            torch.testing.assert_close(ak[:,:,:stop],bk[:,:,:stop],atol=0,rtol=0)
            torch.testing.assert_close(av[:,:,:stop],bv[:,:,:stop],atol=0,rtol=0)
        events=mask.events
    else:
        b=a;recent_groups=full_groups;events=[]
    p.reset(model)
    compact_groups=ingest(model,bank,recent,meta)
    compact=snapshot(model)
    d.assert_fresh_cache(model,len(recent))
    return dict(A=a,B=b,compact=compact),dict(history_stop=stop,prompt_tokens=prompt,full_groups=full_groups,
        independent_recent_groups=recent_groups,independent_encoding_mask_trace=events,compact_groups=compact_groups,
        state_bytes={name:sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in state["kv"]) for name,state in [("A",a),("B",b),("compact",compact)]},
        note="A/B/compact control snapshots are prepared once per question and held simultaneously; answer-arm timing is not standalone system cost")


def probe(model,state,ids,start,stop,force=False,steps=2,attentions=False):
    import torch
    restore(model,state)
    cache=model.kv_cache
    logical=int(state["positions"][0][-1])+1
    logits=[];weights=[]
    with HistoryMask(model,start,stop,force=force) as mask:
        try:
            for step in range(steps+1):
                current=ids if step==0 else ids[:,:1]
                n=current.shape[1]
                model._set_rotary_required_len(logical+n)
                for layer in range(model.num_layers):model._layer_position_ids[layer]=model._build_position_ids(logical,n,current.shape[0])
                output=model.language_model(input_ids=current,past_key_values=cache,position_ids=model._layer_position_ids[0],
                                            use_cache=True,output_attentions=attentions,return_dict=True)
                cache=output.past_key_values
                logits.append(output.logits.detach().clone())
                if attentions:weights.append(output.attentions)
                logical+=n
        finally:model._layer_position_ids.clear()
    return logits,weights,mask.events


def error_statistics(actual,expected,atol,rtol):
    import torch
    a,b=actual.float(),expected.float()
    if a.shape!=b.shape or not torch.isfinite(a).all() or not torch.isfinite(b).all():
        raise ValueError("Nonfinite or shape-mismatched preflight tensors")
    delta=(a-b).abs()
    return dict(elements=a.numel(),max_abs=float(delta.max()),rms=float(delta.square().mean().sqrt()),
                relative_l2=float(torch.linalg.vector_norm(a-b)/torch.linalg.vector_norm(b).clamp_min(1e-12)),
                exceeds_original_005_bound=int((delta>(.005+.005*b.abs())).sum()),
                exceeds_active_bound=int((delta>(atol+rtol*b.abs())).sum()))


class PrecisionProxy:
    """Override only public dtype; all state/methods stay on the original model."""
    def __init__(self,model,dtype):
        object.__setattr__(self,"_wrapped",model);object.__setattr__(self,"_dtype",dtype)
    def __getattr__(self,name):
        return object.__getattribute__(self,"_dtype") if name=="dtype" else getattr(object.__getattribute__(self,"_wrapped"),name)
    def __setattr__(self,name,value):setattr(object.__getattribute__(self,"_wrapped"),name,value)


@contextlib.contextmanager
def fp32_structural_reference(model):
    """Temporarily promote LM parameters, preserving original mixed buffer dtypes/values.

    Vision weights are never cast. Cached rotary buffers are restored exactly,
    rather than blindly calling half() on float32 frequency buffers afterward.
    """
    import torch
    from torch.nn.attention import sdpa_kernel,SDPBackend
    lm=model.language_model
    parameter_dtypes={name:value.dtype for name,value in lm.named_parameters()}
    buffers={name:value.clone() for name,value in lm.named_buffers()}
    rotary=[(layer.self_attn.rotary_emb,layer.self_attn.rotary_emb.max_seq_len_cached,
             getattr(layer.self_attn.rotary_emb,"_required_seq_len",0)) for layer in lm.model.layers]
    try:
        for value in lm.parameters():
            if value.is_floating_point() and value.dtype not in (torch.float16,torch.bfloat16,torch.float32):
                raise ValueError("Unexpected parameter dtype for reversible FP32 reference")
        model.clear_cache()
        for value in lm.parameters():
            if value.is_floating_point():value.data=value.data.float()
        # Both reference paths use the same high-precision math implementation.
        with sdpa_kernel(SDPBackend.MATH):yield PrecisionProxy(model,torch.float32)
    finally:
        for name,value in lm.named_parameters():value.data=value.data.to(parameter_dtypes[name])
        for name,value in buffers.items():
            parent,_,leaf=name.rpartition(".")
            setattr(lm.get_submodule(parent) if parent else lm,leaf,value)
        for module,length,required in rotary:
            module.max_seq_len_cached=length;module._required_seq_len=required
        model.clear_cache();model.kv_cache=None
        model._layer_position_ids.clear();model._position_ids_cache=[None]*model.num_layers
        model.token_activity_cache=[None]*model.num_layers


def _preflight_pass(model,bank,meta,ids,atol=.02,rtol=.005,kv_atol=.005,kv_rtol=.005,diagnostics=None,save=None):
    """Check physical masking against independent construction and compact readout."""
    import torch
    p.reset(model);sink=snapshot(model);s=sink["prompt"]
    states,detail=prepare_states(model,bank,meta,[0],[1])
    stop=s+196
    restore(model,sink)
    ingest(model,bank,[1],meta,logical_start=stop)
    independent=snapshot(model)
    construction_error=0.0
    comparisons=[]
    for layer,((fk,fv),(ik,iv)) in enumerate(zip(states["B"]["kv"],independent["kv"])):
        for kind,actual,expected in (("K",fk[:,:,stop:],ik[:,:,s:]),("V",fv[:,:,stop:],iv[:,:,s:])):
            comparisons.append(dict(layer=layer,kind=kind,**error_statistics(actual,expected,kv_atol,kv_rtol)))
    if diagnostics is not None:
        diagnostics.update(dtype=str(model.dtype),kv_atol=kv_atol,kv_rtol=kv_rtol,construction_by_layer=comparisons)
        if save:save()
    for layer,((fk,fv),(ik,iv)) in enumerate(zip(states["B"]["kv"],independent["kv"])):
        for kind,actual,expected in (("K",fk[:,:,stop:],ik[:,:,s:]),("V",fv[:,:,stop:],iv[:,:,s:])):
            torch.testing.assert_close(actual,expected,atol=kv_atol,rtol=kv_rtol,
                                       msg=lambda msg: f"Construction preflight layer {layer} {kind}: {msg}")
            construction_error=max(construction_error,float((actual-expected).abs().max()))
    records=[]
    plain,_,_=probe(model,states["A"],ids,0,0)
    explicit,_,all_events=probe(model,states["A"],ids,0,0,force=True)
    for name,left,right in [("all_visible_explicit_causal_matches_native",plain,explicit)]:
        maximum=0.0
        for a,b in zip(left,right):
            torch.testing.assert_close(a,b,atol=atol,rtol=rtol)
            assert torch.equal(a[:,-1].argmax(-1),b[:,-1].argmax(-1)),"Preflight argmax changed"
            maximum=max(maximum,float((a-b).abs().max()))
        records.append(dict(check=name,max_logit_error=maximum,argmax_equal=True))
    for label,state in (("contextual_recent",states["A"]),("independent_recent",states["B"])):
        masked,_,events=probe(model,state,ids,s,stop)
        compact,_,_=probe(model,drop_history(state,s,stop),ids,0,0)
        maximum=0.0
        for a,b in zip(masked,compact):
            torch.testing.assert_close(a,b,atol=atol,rtol=rtol)
            assert torch.equal(a[:,-1].argmax(-1),b[:,-1].argmax(-1)),"Masked/compact preflight argmax changed"
            maximum=max(maximum,float((a-b).abs().max()))
        assert len(events)==3 and [e["query_tokens"] for e in events][1:]==[1,1]
        records.append(dict(check=label+"_masked_matches_physically_removed_history",max_logit_error=maximum,
                            argmax_equal=True,query_and_decode_mask_calls=len(events)))
    _,weights,events=probe(model,states["B"],ids,s,stop,attentions=True)
    for step,layers in enumerate(weights):
        for attention in layers:
            assert not torch.any(attention[...,s:stop]),"Historical attention leaked"
            q=attention.shape[-2]
            assert not torch.any(torch.triu(attention[...,-q:],diagonal=1)),"Future attention leaked"
    # Query path must leave the stored control source unchanged.
    restore(model,states["A"])
    source=snapshot(model)
    probe(model,states["A"],ids,s,stop)
    for (a,b),(x,y) in zip(states["A"]["kv"],source["kv"]):
        torch.testing.assert_close(a,x,atol=0,rtol=0);torch.testing.assert_close(b,y,atol=0,rtol=0)
    model.clear_cache()
    return dict(status="pass",history_tokens=196,recent_tokens=196,prompt_tokens=s,
        independent_construction_max_kv_error=construction_error,comparisons=records,
        all_layer_hidden_history_attention_zero=True,all_layer_future_attention_zero=True,
        source_snapshots_unchanged=True,attention_checks_steps=len(events),layers=model.num_layers,
        tolerances=dict(kv_atol=kv_atol,kv_rtol=kv_rtol,logits_atol=atol,logits_rtol=rtol),
        scope="One two-frame visual prefix; explicit history mask at question prefill and two forced decode steps; not every production prefix")


def numerical_preflight(model,bank,meta,ids,atol=.02,rtol=.005,diagnostic_path=None):
    import torch
    diagnostics=dict(policy="v2_strict_fp32_structure_then_fp16_production",fp32={},production={})
    def save():
        if diagnostic_path is not None:p.atomic_json(diagnostic_path,diagnostics)
    if model.dtype in (torch.float16,torch.bfloat16):
        print("STAGE_PREFLIGHT strict FP32 structural reference (math attention)",flush=True)
        with fp32_structural_reference(model) as reference:
            structural=_preflight_pass(reference,bank,meta,ids,atol=.001,rtol=.001,kv_atol=.0001,kv_rtol=.0001,
                                       diagnostics=diagnostics["fp32"],save=save)
        diagnostics["fp32"]["result"]=structural;save()
        print("STAGE_PREFLIGHT restored production precision; checking FP16 numerical bounds",flush=True)
        result=_preflight_pass(model,bank,meta,ids,atol=atol,rtol=rtol,kv_atol=.01,kv_rtol=.01,
                               diagnostics=diagnostics["production"],save=save)
        result["fp32_structural_reference"]=structural
        result["production_precision_restored"]=True
    else:
        result=_preflight_pass(model,bank,meta,ids,atol=atol,rtol=rtol,diagnostics=diagnostics["production"],save=save)
    result["validation_policy"]=diagnostics["policy"]
    result["numerical_limits"]="FP32 checks mask and position equivalence. Production precision has a separate K/V tolerance. Zero hidden-history/future attention, argmax and per-question replay gates are mandatory."
    diagnostics["production"]["result"]=result;save()
    return result


def checkpoint_valid(value,key,identity,arm,history,recent):
    try:
        row=value["row"]
        return (value["identity"]==key and value["row_sha256"]==d.fingerprint(row) and all(row[k]==v for k,v in identity.items())
            and row["method"]==arm and row["history_indices"]==history and row["recent_indices"]==recent
            and ((row["status"]=="answered" and row["correct"]==int(row["pred_choice"]==row["gold_choice"]))
                 or (row["status"]=="skipped" and row["correct"] is None and bool(row["skip_reason"]))))
    except (KeyError,ValueError,TypeError):return False


def run_answer(model,analyzer,video,qi,arm,scope,states,detail,history,recent,feature_hash,config_hash,args,old_choice=None):
    import torch
    identity=d.question_identity(video,qi,arm)
    key=d.fingerprint(dict(config=config_hash,scope=scope,question=identity,history=history,recent=recent,features=feature_hash))
    dest=args.work/"checkpoints"/scope/arm/f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value=json.loads(dest.read_text(encoding="utf-8"))
        if not checkpoint_valid(value,key,identity,arm,history,recent):raise d.RunConflict("Stage checkpoint differs; preserve and use new WORK")
        if value["row"].get("reference_choice_matches") is False:
            raise ValueError("Previously saved replay mismatch remains unresolved; do not bypass the attribution gate")
        print(f"REUSE {scope} {arm} sample={video['sample_id']} q={qi}",flush=True)
        return value["row"]
    construction=arm in (ARMS[1],ARMS[3])
    hidden=arm in (ARMS[2],ARMS[3])
    state=states["compact" if arm==ARMS[4] else "B" if construction else "A"]
    s=state["prompt"];stop=s+len(history)*196
    p.report_progress("stage_answer",sample_id=video["sample_id"],qa_index=qi,arm=arm,scope=scope)
    row=dict(**identity,scope=scope,history_indices=history,recent_indices=recent,history_empty=not history,
        independent_construction=construction,history_readout_hidden=hidden,factor_cell=arm!=ARMS[4],
        history_physically_present=bool(history) and arm!=ARMS[4],status="answered",skip_reason="",
        feature_fingerprint=feature_hash,cache_lengths=[x[0].shape[2] for x in state["kv"]],
        recent_logical_start=s+(0 if arm==ARMS[4] else len(history)*196),
        recent_physical_start=s+(0 if arm==ARMS[4] else len(history)*196),
        actual_visual_indices=recent if arm==ARMS[4] else history+recent,mask_trace=[],
        query_logical_start=int(state["positions"][0][-1])+1,
        pred_choice="",pred_answer="",correct=None,answer_s=0.0,answer_wall_s=0.0,peak_allocated_gib=0.0,
        reference_choice=old_choice if old_choice is not None else "",reference_choice_matches="")
    started=time.perf_counter()
    restore(model,state)
    if model.conv_history:raise ValueError("Previous answer history leaked")
    random.seed(p.SEED);torch.manual_seed(p.SEED)
    torch.cuda.reset_peak_memory_stats()
    try:
        prompt=analyzer.format_mcqa_prompt(identity["question"],identity["choices"])
        qlen=len(model.processor.tokenizer(prompt["prompt"]).input_ids)
        context=model.language_model.config.max_position_embeddings
        if int(state["positions"][0][-1])+1+qlen+48>context:
            if scope=="main":raise ValueError("Main context limit exceeded")
            row.update(status="skipped",skip_reason="context_limit",correct=None)
        else:
            p.sync();before=time.perf_counter()
            with HistoryMask(model,s,stop if hidden else s) as mask:
                answer=model.question_answering(prompt,max_new_tokens=16)
            p.sync();row["answer_s"]=time.perf_counter()-before
            row["mask_trace"]=mask.events
            if hidden and history:
                assert mask.events and all(e["blocked_start"]==s and e["blocked_stop"]==stop for e in mask.events)
                assert all(e["query_tokens"]==1 for e in mask.events[1:])
            assert_state(model,state)
            choice=analyzer.extract_characters_regex(answer)
            row.update(pred_choice=choice,pred_answer=answer.replace("\n",""),correct=int(choice==identity["gold_choice"]),
                       peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
            if old_choice is not None:row["reference_choice_matches"]=choice==old_choice
    finally:model._layer_position_ids.clear();model.clear_cache()
    row["answer_wall_s"]=time.perf_counter()-started
    value=dict(identity=key,row=row,row_sha256=d.fingerprint(row))
    if not checkpoint_valid(value,key,identity,arm,history,recent):raise ValueError("Stage checkpoint failed validation")
    p.atomic_json(dest,value)
    print(f"STAGE_DONE {scope} {arm} sample={video['sample_id']} q={qi} correct={row['correct']} replay={row['reference_choice_matches']}",flush=True)
    if row["reference_choice_matches"] is False:
        raise ValueError("No-op/compact historical replay mismatch: evidence saved; stop attribution and audit numerics")
    return row


def export(rows,work):
    fields=["scope","sample_id","qa_index","method","question","choices","gold_choice","pred_choice","correct","status","skip_reason",
        "history_indices","recent_indices","actual_visual_indices","history_empty","factor_cell","history_physically_present",
        "independent_construction","history_readout_hidden","cache_lengths","query_logical_start",
        "recent_logical_start","mask_trace","reference_choice","reference_choice_matches","answer_s","answer_wall_s","peak_allocated_gib"]
    p.write_csv(work/"exports/predictions.csv",[{k:r.get(k,"") for k in fields} for r in rows])
    summary=[]
    for scope in ("main","long_204_q3"):
        for arm in (ARMS if scope=="main" else ARMS[:4]):
            subset=[r for r in rows if r["scope"]==scope and r["method"]==arm]
            if not subset:continue
            answers=[r for r in subset if r["status"]=="answered"]
            summary.append(dict(scope=scope,arm=arm,recorded=len(subset),answered=len(answers),correct=sum(r["correct"] for r in answers),
                history_empty_questions=sum(r["history_empty"] for r in answers)))
    p.write_csv(work/"exports/summary.csv",summary)
    pairs=[]
    keys={(r["scope"],r["sample_id"],r["qa_index"]):{} for r in rows}
    for r in rows:keys[r["scope"],r["sample_id"],r["qa_index"]][r["method"]]=r
    for a,b in [(ARMS[0],ARMS[1]),(ARMS[0],ARMS[2]),(ARMS[1],ARMS[3]),(ARMS[2],ARMS[3]),(ARMS[3],ARMS[4])]:
        for scope in ("main","long_204_q3"):
            matched=[(v[a],v[b]) for k,v in keys.items() if k[0]==scope and a in v and b in v and v[a]["status"]==v[b]["status"]=="answered"]
            if matched:pairs.append(dict(scope=scope,arm_a=a,arm_b=b,questions=len(matched),a_only_correct=sum(bool(x["correct"]) and not y["correct"] for x,y in matched),
                b_only_correct=sum(not x["correct"] and bool(y["correct"]) for x,y in matched),choice_changes=sum(x["pred_choice"]!=y["pred_choice"] for x,y in matched),
                history_empty_pairs=sum(x["history_empty"] for x,y in matched)))
    if pairs:p.write_csv(work/"exports/paired.csv",pairs)
    text=["# Stage attribution screen","","Inspected development questions; no original-method or population claim.",
        "","| Scope | Arm | Answered | Correct |","|---|---|---:|---:|"]
    text += [f"| {x['scope']} | {x['arm']} | {x['answered']} | {x['correct']} |" for x in summary]
    text += ["","A: H visible in encoding/readout. B: H masked during R encoding only. C: H masked at every question/decode call only. D: H masked at both stages. Compact: fresh recent-4 at compact positions.",
        "B/D use a physical H mask validated against independent cache construction at preserved logical positions; they are not complete DSCache implementations.",
        "H remains physically stored in A-D. C may retain historical information mixed indirectly into R. History-empty questions are null controls.",
        "Long 204/q3 results are separate from the main 40-question matrix. A and compact must replay earlier choices or execution stops.",
        "Shared cache preparation/snapshots are logged separately; per-answer timing/peak allocation includes shared control snapshots and is not standalone algorithm efficiency."]
    (work/"exports/report.md").write_text("\n".join(text)+"\n",encoding="utf-8")


def package(work,status):
    captured=work/"captured_logs";captured.mkdir(parents=True,exist_ok=True)
    for x in (work/"logs").glob("*"):
        if x.is_file():shutil.copy2(x,captured/x.name)
    if (work/"progress.json").exists():shutil.copy2(work/"progress.json",captured/"progress_snapshot.json")
    files=[]
    for area in ("exports","checkpoints","provenance","preparation","captured_logs"):
        files += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("stage_runner.py","stage_context_utils.py","context_input_utils.py","diagnostic_pilot_utils.py","reference.json","launch.json","asset_manifest.json","preflight.json","stage_preflight.json"):
        if (work/name).exists():files.append(work/name)
    manifest=[dict(path=x.relative_to(work).as_posix(),bytes=x.stat().st_size,sha256=p.digest(x)) for x in sorted(set(files))]
    p.atomic_json(work/"evidence_manifest.json",dict(status=status,files=manifest))
    target=work/ARTIFACT
    with ZipFile(target.with_suffix(".tmp"),"w",compression=ZIP_DEFLATED) as z:
        for x in manifest:z.write(work/x["path"],x["path"])
        z.write(work/"evidence_manifest.json","evidence_manifest.json")
    os.replace(target.with_suffix(".tmp"),target)
    print(f"EVIDENCE_ZIP={target}\nSHA256={p.digest(target)}",flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    reference=json.loads((args.work/"reference.json").read_text(encoding="utf-8"))
    d.check_reference(reference,args.repo)
    if args.model.name!=p.MODEL_SHA or torch.__version__!="2.10.0+cu128" or transformers.__version__!="4.45.0.dev0" or not torch.cuda.is_available():raise ValueError("Pinned model/CUDA environment required")
    config=dict(reference_sha256=p.digest(args.work/"reference.json"),sources={n:p.digest(args.work/n) for n in
        ("stage_runner.py","stage_context_utils.py","context_input_utils.py","diagnostic_pilot_utils.py")},arms=ARMS,
        model_revision=p.MODEL_SHA,torch=torch.__version__,transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),
        seed=p.SEED,main_window=20,recent_window=4,long_case=args.long_case,mask_policy="all layers, prefill and every decode; physical H retained",shared_control_snapshots=True)
    config_hash=d.fingerprint(config);path=args.work/"exports/run_config.json"
    if path.exists() and json.loads(path.read_text())["config_hash"]!=config_hash:raise d.RunConflict("Stage configuration changed; use new WORK")
    p.atomic_json(path,dict(**config,config_hash=config_hash))
    failure=args.work/"exports/failure.json"
    if failure.exists():
        dest=args.work/"logs"/f"previous_failure_{time.time_ns()}.json";dest.parent.mkdir(parents=True,exist_ok=True);os.replace(failure,dest)
    videos=copy.deepcopy(reference["videos"]);d.acquire_videos(videos,args.reuse,args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json",videos)
    sys.path.insert(0,str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8,os.cpu_count() or 1));torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    random.seed(p.SEED);np.random.seed(p.SEED);torch.manual_seed(p.SEED)
    p.report_progress("loading_stage_model")
    model,processor=load_model(model_path=str(args.model),kv_size=4000,streaming=True,sample_fps=p.FPS);model.eval()
    analyzer=HermesVQA(anno=[],save_dir=str(args.work),sample_fps=p.FPS,qa_model=model,qa_processor=processor)
    old={(r["sample_id"],r["qa_index"],r["method"]):r for r in reference["input_rows"]}
    rows=[];done=0;started=time.perf_counter();preflight_done=False;long_done=not args.long_case
    with torch.inference_mode():
        # Start with the already inspected 204 case, then preserve a deterministic video order.
        videos.sort(key=lambda v:(v["sample_id"]!=204,v["sample_id"]))
        for video in videos:
            directory,meta,_=d.acquire_features(model,video,args.reuse,args.work,d.fingerprint(dict(config=config_hash,video=video["video_sha256"])))
            bank=c.FeatureBank(directory,meta);feature_hash=d.fingerprint(meta)
            if not preflight_done:
                p.preflight(model,video,args.work)
                ids=processor.tokenizer("What is visible?",return_tensors="pt").input_ids.to(model.device)
                p.report_progress("stage_mask_position_preflight")
                p.atomic_json(args.work/"stage_preflight.json",numerical_preflight(model,bank,meta,ids,
                    diagnostic_path=args.work/"exports/stage_preflight_diagnostics.json"))
                preflight_done=True
            for qi in range(len(video["conversations"])):
                if done and args.max_hours and time.perf_counter()-started>args.max_hours*3600:break
                h,r=split_frames(video,qi)
                torch.cuda.reset_peak_memory_stats()
                before=time.perf_counter();states,detail=prepare_states(model,bank,meta,h,r);p.sync()
                detail.update(shared_preparation_s=time.perf_counter()-before,history_indices=h,recent_indices=r,
                              preparation_peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
                p.atomic_json(args.work/"preparation/main"/f"sample_{video['sample_id']}_q{qi}.json",detail)
                # Replay gates run first; no scientific factor comparison after a mismatch.
                for arm in (ARMS[0],ARMS[4],ARMS[1],ARMS[2],ARMS[3]):
                    previous=old[video["sample_id"],qi,"recent_20" if arm==ARMS[0] else "recent_4"] if arm in (ARMS[0],ARMS[4]) else None
                    rows.append(run_answer(model,analyzer,video,qi,arm,"main",states,detail,h,r,feature_hash,config_hash,args,
                                           previous["pred_choice"] if previous else None))
                del states
                if args.long_case and (video["sample_id"],qi)==(204,3):
                    h,r=split_frames(video,qi,True)
                    torch.cuda.reset_peak_memory_stats()
                    before=time.perf_counter();states,detail=prepare_states(model,bank,meta,h,r);p.sync()
                    detail.update(shared_preparation_s=time.perf_counter()-before,history_indices=h,recent_indices=r,
                                  preparation_peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
                    p.atomic_json(args.work/"preparation/long_204_q3.json",detail)
                    for arm in ARMS[:4]:
                        rows.append(run_answer(model,analyzer,video,qi,arm,"long_204_q3",states,detail,h,r,feature_hash,config_hash,args,
                                               reference["long_reference_choice"] if arm==ARMS[0] else None))
                    del states;long_done=True
                done+=1;export(rows,args.work)
                elapsed=time.perf_counter()-started
                print(f"QUESTION_BLOCK_DONE {done}/40 elapsed_min={elapsed/60:.2f} rough_remaining_min={(40-done)*elapsed/done/60:.2f}",flush=True)
            del bank
            if done<40 and args.max_hours and time.perf_counter()-started>args.max_hours*3600:break
    status="complete" if done==40 and long_done else "partial_time_budget"
    if status=="complete" and (sum(x["scope"]=="main" for x in rows)!=200 or any(x["status"]!="answered" for x in rows if x["scope"]=="main")):raise ValueError("Main stage matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json",dict(status=status,main_questions=done,main_planned=40,main_answers=sum(x["scope"]=="main" and x["status"]=="answered" for x in rows),
        long_case_done=long_done,long_case_enabled=bool(args.long_case),rows=len(rows),config_hash=config_hash,elapsed_post_model_s=time.perf_counter()-started,
        timing_scope="Includes preflight, verified feature work, shared control cache construction, answers and exports; excludes setup/download/model load/final packaging"))
    p.report_progress(status,main_questions=done);package(args.work,status)
    print("STAGE_ATTRIBUTION_COMPLETE" if status=="complete" else "STAGE_ATTRIBUTION_PARTIAL",flush=True)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    for n in ("repo","work","model","reuse"):parser.add_argument("--"+n,type=Path,required=True)
    parser.add_argument("--long-case",type=int,choices=(0,1),default=1)
    parser.add_argument("--max-hours",type=float,default=1.0)
    args=parser.parse_args()
    for n in ("repo","work","model","reuse"):setattr(args,n,getattr(args,n).resolve())
    d.validate_workspaces(args.work,args.reuse)
    if args.max_hours<0:parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:execute(args)
        except d.RunConflict:raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json",dict(status="failed",type=type(exc).__name__,detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json",dict(status="failed"));package(args.work,"failed_partial");raise


if __name__=="__main__":main()
'''
ast.parse(STAGE_BASE_SOURCE)
assert hashlib.sha256(STAGE_BASE_SOURCE.encode()).hexdigest() == 'd677ec07a0f0244cb219da0d049c2fb4b9d82d9658b5039a9ec6e4767a89c408'
CONTEXT_SOURCE = r'''"""Small fresh-state context/content and chunk-partition interventions.

All replacements are declared counterfactuals using original pre-query features.
Candidate-support probes are analyst-selected, not certified human oracles.
"""
import argparse
import copy
import hashlib
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

try:
    import context_input_utils as d
except ModuleNotFoundError as exc:
    if exc.name != "context_input_utils":
        raise
    import autodl_input_diagnostic as d

p = d.p
ARTIFACT = "HERMES_context_diagnostic_evidence.zip"
CASES = ((204,0),(203,2),(204,3),(209,0),(208,4),(202,0),(208,0))
ARMS = ("question_only", "recent_4_original", "recent_20_original", "full_prefix_original",
        "full_prefix_framewise", "recent_20_fixed4", "recent_20_opening_fill", "recent_20_local_fill",
        "full_prefix_fixed4", "full_prefix_opening_fill", "full_prefix_local_fill", "candidate_support")
SUPPORT_CANDIDATES = {(204,0):[6,7], (204,3):[69]}


def validate_indices(indices, video, qi):
    end = d.prefix_end(video,qi)
    if any(type(i) is not int or i < 0 or i >= end for i in indices):
        raise ValueError("Counterfactual contains an invalid/future sampled index")
    if indices != sorted(indices):
        raise ValueError("Input mapping must be nondecreasing; repeated fillers are explicit")
    query = float(video["conversations"][qi]["end_time"])
    if any(float(video["timestamps"][i]["start_s"]) > query+1e-6 for i in indices):
        raise ValueError("Source frame starts after query")


def arm_plan(video, qi, arm):
    if arm not in ARMS:
        raise ValueError("Unknown intervention arm")
    end = d.prefix_end(video,qi)
    recent = list(range(max(0,end-20),end))
    tail = list(range(max(0,end-4),end))
    if arm == "question_only":
        indices,partition = [],"original"
    elif arm == "candidate_support":
        indices,partition = list(SUPPORT_CANDIDATES.get((video["sample_id"],qi),[])),"fixed4"
    elif arm == "recent_4_original":
        indices,partition = tail,"original"
    else:
        indices = list(range(end)) if arm.startswith("full_prefix_") else recent
        partition = "original" if arm.endswith("_original") else "framewise" if arm.endswith("_framewise") else "fixed4"
        if arm.endswith("_fill"):
            filler = 0 if arm.endswith("_opening_fill") else tail[0]
            indices = [filler]*(len(indices)-len(tail)) + tail
    validate_indices(indices,video,qi)
    replacement = arm.endswith("_fill")
    return dict(arm=arm,sampled_indices=indices,partition=partition,counterfactual=replacement,
                protected_tail_slots=list(range(max(0,len(indices)-4),len(indices))) if replacement else [],
                candidate_only=arm == "candidate_support",
                skip_reason="candidate_not_defined" if arm == "candidate_support" and not indices else "",
                intervention_effective=replacement and indices != (list(range(end)) if arm.startswith("full_prefix_") else recent))


def input_groups(indices, partition, meta):
    if partition in ("fixed4","framewise"):
        size = 4 if partition == "fixed4" else 1
        return [indices[i:i+size] for i in range(0,len(indices),size)]
    slices = d.feature_slices(indices,meta["chunks"])
    by_file = {c["file"]:c for c in meta["chunks"]}
    return [list(range(by_file[c["file"]]["start"]+c["start"],by_file[c["file"]]["start"]+c["stop"])) for c in slices]


class FeatureBank:
    def __init__(self,directory,meta):
        import torch
        self.frames = {}
        for chunk in meta["chunks"]:
            value = torch.load(directory/chunk["file"],map_location="cpu",weights_only=True)
            count = chunk["stop"]-chunk["start"]
            if value.ndim != 3 or value.shape[:2] != (1,count*196) or value.dtype != torch.float16:
                raise ValueError("Feature layout differs from original per-frame token blocks")
            for offset,index in enumerate(range(chunk["start"],chunk["stop"])):
                if index in self.frames:
                    raise ValueError("Overlapping feature chunks")
                self.frames[index] = value[:,offset*196:(offset+1)*196]

    def get(self,indices):
        import torch
        return torch.cat([self.frames[i] for i in indices],dim=1).contiguous()


def valid_checkpoint(value,key,identity,plan):
    try:
        row = value["row"]
        return (value["identity"] == key and value["row_sha256"] == d.fingerprint(row)
            and all(row[k] == v for k,v in identity.items()) and row["plan"] == plan
            and ((row["status"] == "answered" and row["correct"] == int(row["pred_choice"] == identity["gold_choice"]))
              or (row["status"] == "skipped" and row["correct"] is None and bool(row["skip_reason"]))))
    except (KeyError,TypeError,ValueError):
        return False


def run_arm(model,analyzer,video,qi,arm,bank,meta,config_hash,args):
    import torch
    plan = arm_plan(video,qi,arm)
    identity = d.question_identity(video,qi,arm)
    key = d.fingerprint(dict(config=config_hash,question=identity,plan=plan,features=d.fingerprint(meta)))
    dest = args.work/"checkpoints"/arm/f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value = json.loads(dest.read_text(encoding="utf-8"))
        if not valid_checkpoint(value,key,identity,plan):
            raise d.RunConflict("Checkpoint identity/correctness mismatch; preserve and choose new WORK")
        print(f"REUSE {arm} sample={video['sample_id']} q={qi}",flush=True)
        return value["row"]
    indices = plan["sampled_indices"]
    p.report_progress("context_arm",sample_id=video["sample_id"],qa_index=qi,arm=arm,input_slots=len(indices))
    row = dict(**identity,plan=plan,status="skipped",skip_reason=plan["skip_reason"],pred_answer="",pred_choice="",correct=None,
               visual_tokens=len(indices)*196,source_frame_indices=[int(video["timestamps"][i]["source_frame"]) for i in indices],
               source_start_seconds=[float(video["timestamps"][i]["start_s"]) for i in indices],
               input_identity=d.fingerprint(dict(features=d.fingerprint(meta),indices=indices)),
               ingest_s=0.0,answer_s=0.0,wall_s=0.0,peak_allocated_gib=0.0,cache_lengths_before_answer=[])
    started = time.perf_counter()
    if not row["skip_reason"]:
        random.seed(p.SEED)
        torch.manual_seed(p.SEED)
        p.reset(model)
        try:
            d.assert_fresh_cache(model,0)
            prompt = analyzer.format_mcqa_prompt(identity["question"],identity["choices"])
            qtokens = len(model.processor.tokenizer(prompt["prompt"]).input_ids)
            row["prompt_tokens"] = model.visual_start_idx
            row["question_tokens"] = qtokens
            row["context_limit"] = model.language_model.config.max_position_embeddings
            reason = d.context_reason(len(indices),model.visual_start_idx,qtokens,row["context_limit"],args.max_visual_tokens,
                                      "native_prefix" if arm.startswith("full_prefix_") else arm)
            if not reason and arm.startswith("full_prefix_"):
                base = sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in model.kv_cache)
                needed = base/model.visual_start_idx*(len(indices)*196+qtokens+48)
                if torch.cuda.mem_get_info()[0] < needed*1.5+2*1024**3:
                    reason = "estimated_vram_limit"
            if reason:
                row["skip_reason"] = reason
            else:
                groups = input_groups(indices,plan["partition"],meta)
                row["ingestion_groups"] = groups
                p.sync()
                torch.cuda.reset_peak_memory_stats()
                before = time.perf_counter()
                for group in groups:
                    p.ingest_features(model,bank.get(group),len(group))
                p.sync()
                row["ingest_s"] = time.perf_counter()-before
                d.assert_fresh_cache(model,len(indices))
                row["cache_lengths_before_answer"] = model._get_cache_seq_len_per_layer()
                row["protected_tail_token_starts"] = [model.visual_start_idx+i*196 for i in plan["protected_tail_slots"]]
                before = time.perf_counter()
                answer = model.question_answering(prompt,max_new_tokens=16)
                p.sync()
                prediction = analyzer.extract_characters_regex(answer)
                row.update(status="answered",pred_answer=answer.replace("\n",""),pred_choice=prediction,
                           correct=int(prediction == identity["gold_choice"]),answer_s=time.perf_counter()-before,
                           peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
        except torch.cuda.OutOfMemoryError:
            if not arm.startswith("full_prefix_"):
                raise
            row.update(status="skipped",skip_reason="cuda_oom",correct=None)
        finally:
            model._layer_position_ids.clear()
            model.clear_cache()
    row["wall_s"] = time.perf_counter()-started
    value = dict(identity=key,row=row,row_sha256=d.fingerprint(row))
    if not valid_checkpoint(value,key,identity,plan):
        raise ValueError("New arm checkpoint failed validation")
    p.atomic_json(dest,value)
    print(f"ARM_DONE {arm} sample={video['sample_id']} q={qi} status={row['status']} correct={row['correct']} reason={row['skip_reason']}",flush=True)
    return row


def export(rows,reference,work):
    fields = ["sample_id","qa_index","method","question","choices","gold_answer","gold_choice","status","skip_reason","pred_choice","correct",
              "plan","input_identity","cache_lengths_before_answer","protected_tail_token_starts","ingest_s","answer_s","wall_s","peak_allocated_gib"]
    p.write_csv(work/"exports/predictions.csv",[{k:r.get(k,"") for k in fields} for r in rows])
    summary = []
    for arm in ARMS:
        cases = [r for r in rows if r["method"] == arm]
        answers = [r for r in cases if r["status"] == "answered"]
        summary.append(dict(arm=arm,recorded=len(cases),answered=len(answers),skipped=len(cases)-len(answers),correct=sum(r["correct"] for r in answers)))
    p.write_csv(work/"exports/summary.csv",summary)
    old = {(r["sample_id"],r["qa_index"],r["method"]):r for r in reference["input_rows"]}
    replay = []
    arm_mapping = {"question_only":"question_only","recent_4_original":"recent_4","recent_20_original":"recent_20","full_prefix_original":"native_prefix"}
    for row in rows:
        if row["method"] in arm_mapping:
            prior = old[row["sample_id"],row["qa_index"],arm_mapping[row["method"]]]
            comparable = row["status"] == prior["status"] == "answered"
            replay.append(dict(sample_id=row["sample_id"],qa_index=row["qa_index"],arm=row["method"],comparable=comparable,
                               previous_choice=prior["pred_choice"],new_choice=row["pred_choice"],
                               choice_matches=row["pred_choice"] == prior["pred_choice"] if comparable else ""))
    p.write_csv(work/"exports/replay_checks.csv",replay)
    by_key = {(r["sample_id"],r["qa_index"],r["method"]):r for r in rows}
    pairs = []
    contrasts = [("full_prefix_original","full_prefix_framewise"),("full_prefix_original","full_prefix_fixed4"),
                 ("recent_20_original","recent_20_fixed4"),("recent_20_fixed4","recent_20_opening_fill"),
                 ("recent_20_fixed4","recent_20_local_fill"),("full_prefix_fixed4","full_prefix_opening_fill"),
                 ("full_prefix_fixed4","full_prefix_local_fill")]
    for sid,qi in CASES:
        for a,b in contrasts:
            if (sid,qi,a) not in by_key or (sid,qi,b) not in by_key:
                continue
            x,y = by_key[sid,qi,a],by_key[sid,qi,b]
            answered = x["status"] == y["status"] == "answered"
            fixed_content = x["plan"]["sampled_indices"] == y["plan"]["sampled_indices"]
            fixed_positions = len(x["plan"]["sampled_indices"]) == len(y["plan"]["sampled_indices"])
            same_tail = x["plan"]["sampled_indices"][-4:] == y["plan"]["sampled_indices"][-4:]
            if b.endswith("_fill") and (not fixed_positions or not same_tail):
                raise ValueError("Counterfactual changed protected tail length/source positions")
            pairs.append(dict(sample_id=sid,qa_index=qi,arm_a=a,arm_b=b,both_answered=answered,
                same_content=fixed_content,same_length=fixed_positions,choice_changed=x["pred_choice"] != y["pred_choice"] if answered else "",
                same_tail_sources=same_tail,replacement_effective=y["plan"]["intervention_effective"],
                a_correct=x["correct"],b_correct=y["correct"],
                interpretation=("Identical-input null control" if fixed_content and x["plan"]["partition"] == y["plan"]["partition"]
                    else "Chunking/numerical control" if fixed_content else "Counterfactual earlier-content replacement; repeated filler is a confound")))
    if pairs:
        p.write_csv(work/"exports/contrasts.csv",pairs)
    text = ["# Context/partition diagnostic", "", "Seven selected development cases; not an algorithm evaluation or population accuracy estimate.",
            "", "| Arm | Answered | Skipped | Correct |", "|---|---:|---:|---:|"]
    text += [f"| {r['arm']} | {r['answered']} | {r['skipped']} | {r['correct']} |" for r in summary]
    mismatches = sum(r["choice_matches"] is False for r in replay if r["comparable"])
    text += ["",f"Historical replay choice mismatches: {mismatches}. Inspect mismatches before interpreting interventions.",
        "Inspect exact-content chunking controls before attributing differences to semantics.",
        "Replacement arms keep last-four source frames and their token positions fixed at equal input length; they repeat pre-query fillers, which are not certified neutral.",
        "When there are no earlier slots, replacement arms are identical-input controls, not independent evidence.",
        "Candidate support at 204/q0 [6,7] and 204/q3 [69] comes from analyst-reviewed saved sheets; no human oracle certification is claimed.",
        "Gold is used only for scoring. All answers reset state and use no compression/past answers. Resource-limited full prefixes are skipped without truncation.",
        "Question wall times exclude model/setup/shared preparation. No publication-ready improvement is implied."]
    (work/"exports/report.md").write_text("\n".join(text)+"\n",encoding="utf-8")


def export_review(video,qi,work):
    from decord import VideoReader
    from PIL import Image,ImageDraw
    reader = VideoReader(video["video_path"],num_threads=1)
    directory = work/"review"/f"sample_{video['sample_id']}_q{qi}"
    directory.mkdir(parents=True,exist_ok=True)
    end = d.prefix_end(video,qi)
    for start in range(0,end,12):
        target = directory/f"page_{start//12+1:02d}.jpg"
        if target.exists():
            continue
        chosen = list(range(start,min(start+12,end)))
        frames = reader.get_batch([int(video["timestamps"][i]["source_frame"]) for i in chosen]).asnumpy()
        canvas = Image.new("RGB",(1024,520),"white")
        draw = ImageDraw.Draw(canvas)
        draw.text((4,4),f"sample={video['sample_id']} q={qi} | ORIGINAL PREFIX: no filler frames | support review pending",fill="black")
        for n,(frame,index) in enumerate(zip(frames,chosen)):
            picture = Image.fromarray(frame)
            picture.thumbnail((256,140))
            x,y = n%4*256,25+n//4*164
            canvas.paste(picture,(x,y))
            draw.text((x+3,y+143),f"sampled {index} | {video['timestamps'][index]['start_s']}s",fill="black")
        canvas.save(target,quality=85)
    for index in SUPPORT_CANDIDATES.get((video["sample_id"],qi),[]):
        target = directory/f"candidate_sampled_{index:04d}_original.png"
        if not target.exists():
            source = int(video["timestamps"][index]["source_frame"])
            Image.fromarray(reader[source].asnumpy()).save(target)


def package(work,status):
    captured = work/"captured_logs"
    captured.mkdir(exist_ok=True,parents=True)
    for source in (work/"logs").glob("*"):
        if source.is_file():
            shutil.copy2(source,captured/source.name)
    if (work/"progress.json").is_file():
        shutil.copy2(work/"progress.json",captured/"progress_snapshot.json")
    files = []
    for area in ("exports","checkpoints","provenance","review","captured_logs"):
        files += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("context_runner.py","context_input_utils.py","diagnostic_pilot_utils.py","reference.json","launch.json","asset_manifest.json","preflight.json"):
        if (work/name).exists():
            files.append(work/name)
    manifest = [dict(path=x.relative_to(work).as_posix(),bytes=x.stat().st_size,sha256=p.digest(x)) for x in sorted(set(files))]
    p.atomic_json(work/"evidence_manifest.json",dict(status=status,files=manifest))
    target = work/ARTIFACT
    with ZipFile(target.with_suffix(".tmp"),"w",compression=ZIP_DEFLATED) as z:
        for row in manifest:
            z.write(work/row["path"],row["path"])
        z.write(work/"evidence_manifest.json","evidence_manifest.json")
    os.replace(target.with_suffix(".tmp"),target)
    print(f"EVIDENCE_ZIP={target}\nSHA256={p.digest(target)}",flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    reference = json.loads((args.work/"reference.json").read_text(encoding="utf-8"))
    d.check_reference(reference,args.repo)
    if args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128" or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available():
        raise ValueError("Pinned model/CUDA environment required")
    config = dict(reference_sha256=p.digest(args.work/"reference.json"),sources={name:p.digest(args.work/name) for name in
        ("context_runner.py","context_input_utils.py","diagnostic_pilot_utils.py")},cases=CASES,arms=ARMS,
        max_visual_tokens=args.max_visual_tokens,seed=p.SEED,model_revision=p.MODEL_SHA,
        torch=torch.__version__,transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),
        no_compression=True,fresh_state=True,candidate_support="Analyst sheet candidates, not certified oracle")
    config_hash = d.fingerprint(config)
    path = args.work/"exports/run_config.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8"))["config_hash"] != config_hash:
        raise d.RunConflict("Run configuration changed; use a new WORK directory")
    p.atomic_json(path,dict(**config,config_hash=config_hash))
    previous_failure = args.work/"exports/failure.json"
    if previous_failure.exists():
        destination = args.work/"logs"/f"previous_failure_{time.time_ns()}.json"
        destination.parent.mkdir(exist_ok=True,parents=True)
        os.replace(previous_failure,destination)
    videos = copy.deepcopy(reference["videos"])
    d.acquire_videos(videos,args.reuse,args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json",videos)
    sys.path.insert(0,str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8,os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_model")
    model,processor = load_model(model_path=str(args.model),kv_size=4000,streaming=True,sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[],save_dir=str(args.work),sample_fps=p.FPS,qa_model=model,qa_processor=processor)
    by_id = {v["sample_id"]:v for v in videos}
    rows,assets = [],{}
    started = time.perf_counter()
    completed = 0
    with torch.inference_mode():
        p.preflight(model,by_id[204],args.work)
        for index,(sid,qi) in enumerate(CASES):
            if index and args.max_hours and time.perf_counter()-started > args.max_hours*3600:
                break
            video = by_id[sid]
            if sid not in assets:
                directory,meta,_ = d.acquire_features(model,video,args.reuse,args.work,d.fingerprint(dict(config=config_hash,video=video["video_sha256"])))
                assets[sid] = (FeatureBank(directory,meta),meta)
            bank,meta = assets[sid]
            order = list(ARMS)
            random.Random(p.SEED+sid*10+qi).shuffle(order)
            for arm in order:
                rows.append(run_arm(model,analyzer,video,qi,arm,bank,meta,config_hash,args))
            export_review(video,qi,args.work)
            export(rows,reference,args.work)
            completed += 1
            elapsed = time.perf_counter()-started
            print(f"CASE_DONE {completed}/{len(CASES)} elapsed_min={elapsed/60:.2f} rough_remaining_min={(len(CASES)-completed)*elapsed/completed/60:.2f}",flush=True)
    status = "complete" if completed == len(CASES) else "partial_time_budget"
    if status == "complete":
        if len(rows) != len(CASES)*len(ARMS) or any(r["status"] != "answered" for r in rows if r["method"] in ARMS[:3]):
            raise ValueError("Core replay control matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json",dict(status=status,case_blocks=completed,planned_blocks=len(CASES),
        rows=len(rows),answered=sum(r["status"] == "answered" for r in rows),config_hash=config_hash,
        elapsed_post_model_s=time.perf_counter()-started,timing_scope="Post-model preflight, verified feature work, arms, review/export; excludes setup/download/model load/final packaging"))
    p.report_progress(status,case_blocks=completed)
    package(args.work,status)
    print("CONTEXT_COMPLETE" if status == "complete" else "CONTEXT_PARTIAL",flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo","work","model","reuse"):
        parser.add_argument("--"+name,type=Path,required=True)
    parser.add_argument("--max-visual-tokens",type=int,default=16384)
    parser.add_argument("--max-hours",type=float,default=.75)
    args = parser.parse_args()
    for name in ("repo","work","model","reuse"):
        setattr(args,name,getattr(args,name).resolve())
    d.validate_workspaces(args.work,args.reuse)
    if args.max_hours < 0 or args.max_visual_tokens < 3920:
        parser.error("Invalid budget/context cap")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json",dict(status="failed",type=type(exc).__name__,detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json",dict(status="failed"))
            package(args.work,"failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(CONTEXT_SOURCE)
assert hashlib.sha256(CONTEXT_SOURCE.encode()).hexdigest() == '5f75057770b517add6a7d735474d671875bfc3a8f2d14899c82f4dfdacfc6d31'
INPUT_SOURCE = r'''"""Matched fresh-state input controls. Heavy libraries are imported only at runtime.

The companion notebook embeds this runner, the tested pilot utilities and the
audited historical reference. Historical artifacts are data, never executed.
"""
import argparse
import copy
import hashlib
import html
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

try:
    import diagnostic_pilot_utils as p
except ModuleNotFoundError as exc:
    if exc.name != "diagnostic_pilot_utils":
        raise
    import autodl_failure_pilot as p

METHODS = ("question_only", "recent_4", "recent_20", "native_prefix", "support_oracle")
ARTIFACT = "HERMES_input_diagnostic_evidence.zip"
PRIORITY = {(204, 0), (202, 0), (208, 0), (212, 3), (218, 3), (204, 3), (219, 3)}


class RunConflict(ValueError):
    """Existing scientific outputs must not be replaced by a different run."""


def fingerprint(value):
    return hashlib.sha256(p.stable(value).encode()).hexdigest()


def validate_workspaces(work, reuse):
    work, reuse = Path(work).resolve(), Path(reuse).resolve()
    if work == reuse or work.is_relative_to(reuse) or reuse.is_relative_to(work):
        raise ValueError("WORK and reusable pilot directory must be separate non-nested locations")


def check_reference(reference, repo):
    for name, expected in reference["source_sha256"].items():
        actual = hashlib.sha256((repo / name).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
        if actual != expected:
            raise ValueError(f"Pinned source differs: {name}")
    annotation = json.loads((repo / "data/streamingbench/streamingbench_realtime.json").read_text(encoding="utf-8"))
    by_id = {str(v["video_id"]): v for v in annotation}
    for video in reference["videos"]:
        original = by_id[str(video["video_id"])]
        if original["conversations"] != video["conversations"] or p.sample_id(original) != video["sample_id"]:
            raise ValueError("Historical annotation identity mismatch")


def prefix_end(video, qi):
    return math.ceil(float(video["conversations"][qi]["end_time"]) * p.FPS)


def validate_frames(indices, video, qi, timestamps):
    end = prefix_end(video, qi)
    if any(type(i) is not int for i in indices) or indices != sorted(set(indices)):
        raise ValueError("Frame indices must be sorted unique integers")
    if any(i < 0 or i >= end or i >= len(timestamps) for i in indices):
        raise ValueError("Selected frame outside original query prefix")
    query = float(video["conversations"][qi]["end_time"])
    if any(float(timestamps[i]["start_s"]) > query + 1e-6 for i in indices):
        raise ValueError("Sampled frame starts after query time")
    return indices


def validate_oracles(items, videos, max_frames=20):
    if not isinstance(items, list):
        raise ValueError("Support JSON must be a list")
    by_id = {v["sample_id"]: v for v in videos}
    result = {}
    for item in items:
        sid, qi = item["sample_id"], item["qa_index"]
        if type(sid) is not int or type(qi) is not int or sid not in by_id:
            raise ValueError("Unknown oracle sample or invalid index type")
        video = by_id[sid]
        if not 0 <= qi < len(video["conversations"]) or (sid, qi) in result:
            raise ValueError("Invalid/duplicate oracle question")
        if item.get("verified") is not True or not str(item.get("notes", "")).strip():
            raise ValueError("Oracle requires verified=true and support review notes")
        indices = item["sampled_indices"]
        if not isinstance(indices, list) or not 1 <= len(indices) <= max_frames:
            raise ValueError("Oracle must contain 1..20 sampled frame indices")
        validate_frames(indices, video, qi, video["timestamps"])
        result[(sid, qi)] = item
    return result


def selected_frames(method, video, qi, oracles, native_enabled):
    end = prefix_end(video, qi)
    if method == "question_only":
        return [], None
    if method in ("recent_4", "recent_20"):
        count = 4 if method == "recent_4" else 20
        return list(range(max(0, end - count), end)), None
    if method == "native_prefix":
        return (list(range(end)), None) if native_enabled else ([], "disabled")
    oracle = oracles.get((video["sample_id"], qi))
    return (oracle["sampled_indices"], None) if oracle else ([], "not_requested")


def context_reason(frame_count, prompt_tokens, question_tokens, context_limit, native_cap, method):
    # question_answering uses greedy max_new_tokens=16; 32 extra positions are a safety reserve.
    needed = prompt_tokens + frame_count * 196 + question_tokens + 16 + 32
    if needed > context_limit:
        return "context_limit"
    if method == "native_prefix" and frame_count * 196 > native_cap:
        return "configured_native_cap"
    return None


def feature_slices(indices, chunks):
    """Return chunk-local runs without mixing or duplicating original frame identities."""
    covered, slices = [], []
    for item in chunks:
        chosen = [i for i in indices if item["start"] <= i < item["stop"]]
        for i in chosen:
            covered.append(i)
            if slices and slices[-1]["file"] == item["file"] and slices[-1]["stop"] == i - item["start"]:
                slices[-1]["stop"] += 1
            else:
                slices.append(dict(file=item["file"], start=i-item["start"], stop=i-item["start"]+1))
    if covered != indices:
        raise ValueError("Feature chunks do not cover selected frames exactly once")
    return slices


def read_features(directory, item):
    import torch
    value = torch.load(directory / item["file"], map_location="cpu", weights_only=True)
    if value.ndim != 3 or value.shape[0] != 1 or value.shape[1] % 196 or value.dtype != torch.float16:
        raise ValueError("Unexpected feature layout/dtype; never guess frame boundaries")
    return value[:, item["start"]*196:item["stop"]*196].contiguous()


def acquire_videos(videos, reuse, work):
    # Only exact trusted historical video hashes can seed resumption/reuse.
    entries, missing = [], []
    for v in videos:
        sid = v["sample_id"]
        candidates = [work / "videos" / f"sample_{sid}_real.mp4", reuse / "videos" / f"sample_{sid}_real.mp4"]
        found = next((x for x in candidates if x.is_file() and p.digest(x) == v["video_sha256"]), None)
        if found is None:
            missing.append(v)
        else:
            entries.append(dict(sample_id=sid, path=str(found.resolve()), bytes=found.stat().st_size,
                                sha256=v["video_sha256"]))
    if not missing:
        assets = dict(mode="verified_existing_videos", videos=entries)
    else:
        downloaded = p.prepare_assets(missing, work)
        assets = dict(mode="verified_existing_plus_downloaded", videos=entries+downloaded["videos"],
                      download_details={k:v for k,v in downloaded.items() if k != "videos"})
    by_id = {x["sample_id"]: x for x in assets["videos"]}
    for v in videos:
        item = by_id[v["sample_id"]]
        if item["sha256"] != v["video_sha256"] or p.digest(item["path"]) != v["video_sha256"]:
            raise ValueError("Video differs from audited source")
        v["video_path"] = item["path"]
    p.atomic_json(work / "asset_manifest.json", assets)


def verify_feature_files(directory, meta):
    for c in meta["chunks"]:
        if Path(c["file"]).name != c["file"] or c["tokens"] != (c["stop"]-c["start"])*196:
            raise ValueError("Invalid feature chunk metadata")
        path = directory / c["file"]
        if not path.is_file() or p.digest(path) != c["sha256"]:
            raise ValueError(f"Missing/corrupt feature: {path}")


def acquire_features(model, video, reuse, work, identity):
    sid = video["sample_id"]
    old_dir = reuse / "features" / f"sample_{sid}"
    expected = video["feature_meta"]
    if (old_dir / "complete.json").is_file():
        try:
            old = json.loads((old_dir / "complete.json").read_text(encoding="utf-8"))
            if old != expected or p.read_csv(old_dir / "sampled_timestamps.csv") != video["timestamps"]:
                raise ValueError("Old feature metadata differs from audited archive")
            verify_feature_files(old_dir, expected)
            meta, directory, origin = old, old_dir, "verified_historical_cache"
            print(f"FEATURE_REUSE_VERIFIED sample={sid}", flush=True)
        except (OSError, ValueError, KeyError) as exc:
            print(f"FEATURE_REUSE_REJECTED sample={sid} reason={type(exc).__name__}; regenerating separately", flush=True)
            directory = None
    else:
        directory = None
    if directory is None:
        directory = work / "features" / f"sample_{sid}"
        meta = p.prepare_features(model, video, work, identity)
        verify_feature_files(directory, meta)
        origin = "diagnostic_cache_verified"
        if p.read_csv(directory / "sampled_timestamps.csv") != video["timestamps"]:
            raise ValueError("Regenerated source sampling differs from original pilot")
        original_layout = [{k:c[k] for k in ("qa_index", "start", "stop", "tokens")} for c in expected["chunks"]]
        if [{k:c[k] for k in ("qa_index", "start", "stop", "tokens")} for c in meta["chunks"]] != original_layout:
            raise ValueError("Regenerated feature chunk boundaries differ")
    saved = dict(origin=origin, metadata=meta, directory=str(directory.resolve()),
                 timestamp_rows=video["timestamps"], feature_fingerprint=fingerprint(meta))
    p.atomic_json(work / "provenance" / f"features_{sid}.json", saved)
    return directory, meta, saved


def question_identity(video, qi, method):
    qa = video["conversations"][qi]
    return dict(sample_id=video["sample_id"], video_id=str(video["video_id"]), qa_index=qi,
                method=method, question=qa["question"], choices=qa["choices"],
                gold_answer=qa["answer"], gold_choice="ABCDEFGH"[qa["choices"].index(qa["answer"])],
                query_time_s=qa["end_time"], task=qa.get("task", ""))


def checkpoint_valid(value, expected, identity, frames):
    try:
        if value["checkpoint_identity"] != expected or value["row_sha256"] != fingerprint(value["row"]):
            return False
        row = value["row"]
        if any(row[k] != v for k,v in identity.items()) or row["sampled_indices"] != frames:
            return False
        if row["status"] == "answered":
            return row["correct"] == int(row["pred_choice"] == row["gold_choice"])
        return row["status"] == "skipped" and bool(row["skip_reason"]) and row["correct"] is None
    except (KeyError, TypeError, ValueError):
        return False


def assert_fresh_cache(model, frames):
    import torch
    expected = model.visual_start_idx + frames * 196
    if model.conv_history or model._get_cache_seq_len_per_layer() != [expected]*model.num_layers:
        raise ValueError("Fresh control retained history or has an unexpected cache length")
    for positions in model._position_ids_cache:
        torch.testing.assert_close(positions, torch.arange(expected, device=model.device), atol=0, rtol=0)


def run_question(model, analyzer, video, qi, method, directory, meta, config_hash, oracles, args):
    import torch
    identity = question_identity(video, qi, method)
    frames, reason = selected_frames(method, video, qi, oracles, args.native)
    validate_frames(frames, video, qi, video["timestamps"])
    check_id = fingerprint(dict(config=config_hash, feature=fingerprint(meta), question=identity, frames=frames))
    dest = args.work / "checkpoints" / method / f"sample_{video['sample_id']}_q{qi}.json"
    if dest.is_file():
        value = json.loads(dest.read_text(encoding="utf-8"))
        if not checkpoint_valid(value, check_id, identity, frames):
            raise RunConflict(f"Existing checkpoint mismatched/corrupt; preserve and use new WORK: {dest}")
        print(f"REUSE {method} sample={video['sample_id']} q={qi}", flush=True)
        return value["row"]
    p.report_progress("question", sample_id=video["sample_id"], qa_index=qi, method=method, selected_frames=len(frames))
    row = dict(**identity, status="skipped", skip_reason=reason or "", sampled_indices=frames,
               source_frame_indices=[int(video["timestamps"][i]["source_frame"]) for i in frames],
               sampled_start_seconds=[float(video["timestamps"][i]["start_s"]) for i in frames],
               pred_answer="", pred_choice="", correct=None, visual_tokens=len(frames)*196,
               ingest_s=0.0, answer_s=0.0, wall_s=0.0, peak_allocated_gib=0.0,
               fresh_state=True, no_compression=True, feature_fingerprint=fingerprint(meta))
    start = time.perf_counter()
    if not reason:
        random.seed(p.SEED)
        torch.manual_seed(p.SEED)
        p.reset(model)
        assert_fresh_cache(model, 0)
        prompt = analyzer.format_mcqa_prompt(identity["question"], identity["choices"])
        qlen = len(model.processor.tokenizer(prompt["prompt"]).input_ids)
        reason = context_reason(len(frames), model.visual_start_idx, qlen,
                                model.language_model.config.max_position_embeddings, args.native_max_tokens, method)
        if not reason and method == "native_prefix":
            # Conservative KV estimate plus 2 GiB workspace; hard cap remains separate.
            base_bytes = sum(k.numel()*k.element_size()+v.numel()*v.element_size() for k,v in model.kv_cache)
            kv_estimate = base_bytes / model.visual_start_idx * (len(frames)*196+qlen+48)
            free, _ = torch.cuda.mem_get_info()
            if free < kv_estimate*1.5 + 2*1024**3:
                reason = "estimated_vram_limit"
        if reason:
            row["skip_reason"] = reason
        else:
            torch.cuda.reset_peak_memory_stats()
            try:
                p.sync()
                before = time.perf_counter()
                for item in feature_slices(frames, meta["chunks"]):
                    value = read_features(directory, item)
                    p.ingest_features(model, value, item["stop"]-item["start"])
                    del value
                p.sync()
                row["ingest_s"] = time.perf_counter()-before
                assert_fresh_cache(model, len(frames))
                row["cache_lengths_before_answer"] = model._get_cache_seq_len_per_layer()
                before = time.perf_counter()
                # Gold is used only AFTER generation. Same prompt/parser/16-token greedy policy as pilot.
                answer = model.question_answering(prompt, max_new_tokens=16)
                p.sync()
                pred = analyzer.extract_characters_regex(answer)
                row.update(status="answered", pred_answer=answer.replace("\n", ""), pred_choice=pred,
                           correct=int(pred == identity["gold_choice"]), answer_s=time.perf_counter()-before,
                           peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3)
            except torch.cuda.OutOfMemoryError:
                if method != "native_prefix":
                    raise
                row.update(status="skipped", skip_reason="cuda_oom", correct=None)
                print("NATIVE_PREFIX_OOM: case skipped explicitly; other controls continue", flush=True)
            finally:
                p.reset(model)
        # Release even a cap/context-skipped fresh state.
        model.clear_cache()
    row["wall_s"] = time.perf_counter()-start
    value = dict(checkpoint_identity=check_id, row=row, row_sha256=fingerprint(row))
    if not checkpoint_valid(value, check_id, identity, frames):
        raise ValueError("New checkpoint failed identity validation")
    p.atomic_json(dest, value)
    print(f"QUESTION_DONE {method} sample={video['sample_id']} q={qi} status={row['status']} correct={row['correct']} reason={row['skip_reason']}", flush=True)
    return row


def export(rows, reference, work):
    fields = list(question_identity(reference["videos"][0], 0, "question_only")) + [
        "status", "skip_reason", "sampled_indices", "source_frame_indices", "sampled_start_seconds", "pred_answer", "pred_choice", "correct",
        "visual_tokens", "ingest_s", "answer_s", "wall_s", "peak_allocated_gib", "fresh_state", "no_compression", "feature_fingerprint", "cache_lengths_before_answer"]
    p.write_csv(work / "exports/predictions.csv", [{k:r.get(k, "") for k in fields} for r in rows])
    summary = []
    for method in METHODS:
        subset = [r for r in rows if r["method"] == method]
        answered = [r for r in subset if r["status"] == "answered"]
        summary.append(dict(method=method, planned=40, recorded=len(subset), answered=len(answered),
                            skipped=len(subset)-len(answered), correct=sum(r["correct"] for r in answered),
                            accuracy_pct=100*sum(r["correct"] for r in answered)/len(answered) if answered else "",
                            measured_wall_s=sum(r["wall_s"] for r in answered)))
    p.write_csv(work / "exports/summary.csv", summary)
    old = {(int(r["sample_id"]), int(r["qa_index"]), r["method"]): r for r in reference["predictions"]}
    paired = []
    for method in METHODS:
        new = [r for r in rows if r["method"] == method and r["status"] == "answered"]
        for baseline in ("fifo", "hermes", "causal"):
            pairs = [(r, old[(r["sample_id"], r["qa_index"], baseline)]) for r in new]
            paired.append(dict(method=method, historical_baseline=baseline, matched_questions=len(pairs),
                diagnostic_correct=sum(r["correct"] for r,b in pairs),
                historical_correct=sum(int(b["correct"]) for r,b in pairs),
                diagnostic_only_correct=sum(r["correct"] and not int(b["correct"]) for r,b in pairs),
                historical_only_correct=sum(not r["correct"] and int(b["correct"]) for r,b in pairs),
                comparison_scope="Exact shared identities; historical session; changed input/history; descriptive development comparison"))
    p.write_csv(work / "exports/paired_historical.csv", paired)
    text = ["# Fresh-state input diagnostic", "", "Selected development cases; no population or original-method claim.", "",
            "| Method | Answered | Skipped | Correct | Accuracy on answered |", "|---|---:|---:|---:|---:|"]
    for s in summary:
        accuracy = f"{s['accuracy_pct']:.1f}%" if s["answered"] else "N/A"
        text.append(f"| {s['method']} | {s['answered']} | {s['skipped']} | {s['correct']} | {accuracy} |")
    text += ["", "Native-prefix/oracle rows may cover different subsets: use paired_historical.csv for exact-subset comparisons.",
        "All controls reset per question; no eviction or past-answer history. Recent/oracle inputs compact selected frames chronologically.",
        "Historical comparisons reuse old predictions; they do not establish same-session efficiency or isolate memory policy alone.",
        "Oracle is privileged human support, not a deployable method. No oracle runs without verified annotations.",
        "Question wall time includes reset/feature reading/ingestion/answer; excludes setup and shared feature preparation.",
        "Review sheets show original sampled frames, not proof of support annotation. Source videos and feature tensors are excluded from ZIP."]
    (work / "exports/report.md").write_text("\n".join(text)+"\n", encoding="utf-8")
    return summary


def initialize_review(videos, work):
    sheet = work / "review/support_review.csv"
    if not sheet.exists():
        rows = []
        for video in videos:
            for qi, qa in enumerate(video["conversations"]):
                rows.append(dict(sample_id=video["sample_id"], qa_index=qi,
                    priority="first" if (video["sample_id"], qi) in PRIORITY else "later",
                    question=qa["question"], query_time_s=qa["end_time"], review_status="pending",
                    support_intervals_seconds_json="", verified_sampled_indices_json="",
                    sampled_support_sufficient="", ambiguity="", cause="unclassified", notes=""))
        p.write_csv(sheet, rows)
    content = ["<!doctype html><meta charset='utf-8'><title>Priority support review</title>",
               "<h1>Original sampled prefixes: support still needs review</h1>",
               "<p>Indices are sampled-frame indices. All pages stop at the original query prefix. "
               "Gold answers are review labels, never provided as model inputs. Fill support_review.csv separately.</p>"]
    for video in videos:
        for qi, qa in enumerate(video["conversations"]):
            if (video["sample_id"], qi) not in PRIORITY:
                continue
            content += [f"<h2>sample {video['sample_id']} / q{qi}</h2>",
                        f"<p>{html.escape(qa['question'])} | query {qa['end_time']} s</p>",
                        f"<p>Choices: {html.escape(str(qa['choices']))} | gold: {html.escape(str(qa['answer']))}</p>"]
            for page in range(math.ceil(prefix_end(video, qi)/12)):
                name = f"sample_{video['sample_id']}_q{qi}/page_{page+1:02d}.jpg"
                content.append(f"<p><a href='{name}'>Page {page+1}</a></p>")
    (work/"review/index.html").write_text("\n".join(content), encoding="utf-8")


def review_pages(video, work):
    from decord import VideoReader
    from PIL import Image, ImageDraw
    reader = None
    for qi, qa in enumerate(video["conversations"]):
        if (video["sample_id"], qi) not in PRIORITY:
            continue
        directory = work / "review" / f"sample_{video['sample_id']}_q{qi}"
        expected = math.ceil(prefix_end(video, qi)/12)
        if all((directory/f"page_{i+1:02d}.jpg").is_file() for i in range(expected)):
            continue
        directory.mkdir(parents=True, exist_ok=True)
        if reader is None:
            reader = VideoReader(video["video_path"], num_threads=1)
        for start in range(0, prefix_end(video, qi), 12):
            p.report_progress("priority_review_frames", sample_id=video["sample_id"], qa_index=qi, sampled_start=start)
            indices = list(range(start, min(start+12, prefix_end(video, qi))))
            source = [int(video["timestamps"][i]["source_frame"]) for i in indices]
            frames = reader.get_batch(source).asnumpy()
            canvas = Image.new("RGB", (1024, 520), "white")
            draw = ImageDraw.Draw(canvas)
            draw.text((4, 4), f"sample={video['sample_id']} q={qi} query={qa['end_time']}s | ORIGINAL SAMPLED PREFIX | support unreviewed", fill="black")
            for n, (frame, i) in enumerate(zip(frames, indices)):
                image = Image.fromarray(frame)
                image.thumbnail((256, 140))
                x, y = n%4*256, 25+n//4*164
                canvas.paste(image, (x,y))
                draw.text((x+3,y+143), f"sampled {i} | {video['timestamps'][i]['start_s']}s", fill="black")
            canvas.save(directory/f"page_{start//12+1:02d}.jpg", quality=85)


def package(work, status):
    captured = work / "captured_logs"
    captured.mkdir(parents=True, exist_ok=True)
    for path in (work / "logs").glob("*"):
        if path.is_file():
            shutil.copy2(path, captured/path.name)
    if (work/"progress.json").is_file():
        shutil.copy2(work/"progress.json", captured/"progress_snapshot.json")
    paths = []
    for area in ("exports", "checkpoints", "provenance", "review", "captured_logs"):
        paths += [x for x in (work/area).rglob("*") if x.is_file() and ".ipynb_checkpoints" not in x.parts]
    for name in ("diagnostic_runner.py", "diagnostic_pilot_utils.py", "reference.json", "launch.json",
                 "asset_manifest.json", "preflight.json", "support_oracle.json"):
        if (work/name).is_file():
            paths.append(work/name)
    manifest = [dict(path=x.relative_to(work).as_posix(), bytes=x.stat().st_size, sha256=p.digest(x)) for x in sorted(set(paths))]
    p.atomic_json(work/"evidence_manifest.json", dict(status=status, files=manifest))
    output = work / ARTIFACT
    with ZipFile(output.with_suffix(".tmp"), "w", compression=ZIP_DEFLATED) as z:
        for item in manifest:
            z.write(work/item["path"], item["path"])
        z.write(work/"evidence_manifest.json", "evidence_manifest.json")
    os.replace(output.with_suffix(".tmp"), output)
    print(f"EVIDENCE_ZIP={output}\nSHA256={p.digest(output)}", flush=True)


def execute(args):
    import numpy as np
    import torch
    import transformers
    args.work, args.repo, args.reuse = args.work.resolve(), args.repo.resolve(), args.reuse.resolve()
    validate_workspaces(args.work, args.reuse)
    if args.model.name != p.MODEL_SHA:
        raise ValueError("Model snapshot revision differs")
    reference = json.loads((args.work / "reference.json").read_text(encoding="utf-8"))
    check_reference(reference, args.repo)
    videos = copy.deepcopy(reference["videos"])
    initialize_review(videos, args.work)
    oracles = validate_oracles(json.loads((args.work/"support_oracle.json").read_text(encoding="utf-8")), videos)
    if torch.__version__ != "2.10.0+cu128" or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available():
        raise RuntimeError("Pinned GPU environment required")
    config = dict(reference_sha256=p.digest(args.work/"reference.json"), runner_sha256=p.digest(Path(__file__)),
                  utils_sha256=p.digest(Path(p.__file__)), model_revision=p.MODEL_SHA, sample_fps=p.FPS,
                  native=args.native, native_max_tokens=args.native_max_tokens,
                  support=sorted(oracles.values(), key=lambda x:(x["sample_id"],x["qa_index"])),
                  torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0),
                  methods=METHODS, seed=p.SEED, fresh_state=True, max_new_tokens=16)
    config_hash = fingerprint(config)
    previous = args.work/"exports/run_config.json"
    if previous.is_file() and json.loads(previous.read_text(encoding="utf-8"))["config_hash"] != config_hash:
        raise RunConflict("Configuration differs from existing run; preserve it and choose a new WORK directory")
    p.atomic_json(previous, dict(**config, config_hash=config_hash))
    failure = args.work/"exports/failure.json"
    if failure.exists():
        target = args.work/"logs"/f"previous_failure_{time.time_ns()}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        os.replace(failure, target)
    p.report_progress("verifying_or_downloading_videos")
    acquire_videos(videos, args.reuse, args.work)
    p.atomic_json(args.work/"exports/replay_annotations.json", videos)
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows = []
    started = time.perf_counter()
    complete = True
    with torch.inference_mode():
        p.preflight(model, videos[0], args.work)
        for vi, video in enumerate(videos):
            if vi and args.max_hours and time.perf_counter()-started > args.max_hours*3600:
                complete = False
                break
            feature_id = fingerprint(dict(config=config_hash, video=video["video_sha256"]))
            directory, meta, provenance = acquire_features(model, video, args.reuse, args.work, feature_id)
            for qi in range(len(video["conversations"])):
                order = list(METHODS)
                random.Random(p.SEED+video["sample_id"]*10+qi).shuffle(order)
                for method in order:
                    rows.append(run_question(model, analyzer, video, qi, method, directory, meta, config_hash, oracles, args))
            if args.review:
                review_pages(video, args.work)
            summary = export(rows, reference, args.work)
            elapsed = time.perf_counter()-started
            print(f"VIDEO_DONE {vi+1}/8 elapsed_min={elapsed/60:.2f} rough_remaining_min={(7-vi)*elapsed/(vi+1)/60:.2f}", flush=True)
    status = "complete" if complete else "partial_time_budget"
    mandatory = [r for r in rows if r["method"] in METHODS[:3]]
    if complete and (len(mandatory) != 120 or any(r["status"] != "answered" for r in mandatory)):
        raise ValueError("Mandatory 120-answer control matrix incomplete")
    p.atomic_json(args.work/"exports/completion.json", dict(status=status, rows=len(rows),
                  mandatory_answered=sum(r["status"] == "answered" for r in mandatory),
                  config_hash=config_hash, elapsed_post_model_s=time.perf_counter()-started,
                  timing_scope="Includes preflight, shared feature work, per-question controls and review export; excludes setup/download/model loading"))
    p.report_progress(status, recorded=len(rows))
    package(args.work, status)
    print("DIAGNOSTIC_COMPLETE" if complete else "DIAGNOSTIC_PARTIAL", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--"+name, type=Path, required=True)
    parser.add_argument("--native", type=int, default=1, choices=(0,1))
    parser.add_argument("--native-max-tokens", type=int, default=8192)
    parser.add_argument("--review", type=int, default=1, choices=(0,1))
    parser.add_argument("--max-hours", type=float, default=1.0)
    args = parser.parse_args()
    if args.max_hours < 0 or args.native_max_tokens < 196:
        parser.error("Invalid budget/context cap")
    # Reject dangerous output locations before the monitor or failure packager can write there.
    validate_workspaces(args.work, args.reuse)
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except RunConflict:
            # Do not replace an older completion/config with this rejected attempt.
            raise
        except Exception as exc:
            p.atomic_json(args.work/"exports/failure.json", dict(status="failed", type=type(exc).__name__, detail=str(exc)[-1500:]))
            p.atomic_json(args.work/"exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(INPUT_SOURCE)
assert hashlib.sha256(INPUT_SOURCE.encode()).hexdigest() == 'c152696ec0401a0afe08c7406a559aa37b3914b77fd6a7c1f25ed782cff3a521'
UTILS_SOURCE = r'''"""Resumable AutoDL scorer screen. Bootstrap is embedded in the companion notebook.

Only experimental instances are patched. Official inference files remain intact.
Heavy imports are deferred so selection, checkpoint and download logic can be
verified on a CPU host without installing the GPU environment.
"""

import argparse
import ast
import copy
import csv
import hashlib
import html
import io
import json
import math
import os
import random
import re
import subprocess
import sys
import threading
import time
import types
from pathlib import Path
from zipfile import ZipFile


MODEL_ID = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
MODEL_SHA = "74dd0bf867a4cda7950c17663794267c60cf4b40"
SHARD_NAME = "Real-Time Visual Understanding_201-250.zip"
SHARD_BYTES = 2230277873
SHARD_SHA = "b7946e8b678fbf704b9816fa708e6febb77f5c37a7ad884084adf858f5d20dbf"
METHODS = ("fifo", "hermes", "causal")
FPS = 0.5
SEED = 2024
_PROGRESS = None


def stable(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
    os.replace(temp, path)


def write_csv(path, rows):
    if not rows:
        raise ValueError("Cannot publish an empty result table")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp, path)


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


class ProgressMonitor:
    """Report liveness separately from completed work, including blocked I/O."""
    def __init__(self, work, interval=30):
        self.work, self.interval = Path(work), interval
        self.work.mkdir(parents=True, exist_ok=True)
        self.started = time.monotonic()
        self.changed = self.started
        self.lock = threading.Lock()
        self.stop = threading.Event()
        self.state = {"stage": "starting", "detail": "No completed work yet"}
        self.thread = threading.Thread(target=self._heartbeat, daemon=True)

    def _write(self, heartbeat=False):
        # Caller holds the lock, so update and heartbeat cannot race the temp file.
        value = {**self.state, "elapsed_s": round(time.monotonic() - self.started, 1),
                 "seconds_since_stage_update": round(time.monotonic() - self.changed, 1),
                 "heartbeat_unix_time": time.time(),
                 "note": "Heartbeat shows process liveness; it does not prove download or GPU progress"}
        try:
            atomic_json(self.work / "progress.json", value)
        except OSError as exc:
            # A Windows reader can temporarily prevent atomic replacement.
            # Progress reporting must not terminate a scientific computation;
            # a subsequent heartbeat retries while stdout remains available.
            print(f"PROGRESS_FILE_RETRY error_type={type(exc).__name__}", flush=True)
        label = "HEARTBEAT" if heartbeat else "STAGE"
        print(label + " " + stable(value), flush=True)

    def update(self, stage, **detail):
        with self.lock:
            self.state = {"stage": stage, **detail}
            self.changed = time.monotonic()
            self._write()

    def _heartbeat(self):
        while not self.stop.wait(self.interval):
            with self.lock:
                self._write(heartbeat=True)

    def __enter__(self):
        global _PROGRESS
        _PROGRESS = self
        self.update("starting", detail="Selecting diagnostic videos")
        self.thread.start()
        return self

    def __exit__(self, kind, error, traceback):
        global _PROGRESS
        self.stop.set()
        self.thread.join(timeout=5)
        if kind is not None:
            self.update("failed", error_type=kind.__name__)
        _PROGRESS = None


def report_progress(stage, **detail):
    if _PROGRESS is not None:
        _PROGRESS.update(stage, **detail)


def sample_id(video):
    return int(re.search(r"sample_(\d+)", video["video_path"])[1])


def choose_screen(repo, per_outcome=2):
    from build_failure_pilot import select_cases
    rows = read_csv(repo / "results/failure_pilot/paired_inventory.csv")
    pool = []
    for row in rows:
        if 201 <= int(row["sample_id"]) <= 250:
            pool.append({**row, "duplicate_gold_options": row["duplicate_gold_options"] == "True"})
    quotas = {k: per_outcome for k in ("hermes_only", "fifo_only", "both_wrong", "both_correct")}
    selected = select_cases(pool, seed=SEED, quotas=quotas)
    annotations = json.loads((repo / "data/streamingbench/streamingbench_realtime.json").read_text(encoding="utf-8"))
    ids = {r["video_id"] for r in selected}
    by_id = {str(v["video_id"]): v for v in annotations if str(v["video_id"]) in ids}
    # Calibrate on one HERMES-only and one FIFO-only target before other cases.
    ordered = [selected[group * per_outcome + index]["video_id"]
               for index in range(per_outcome) for group in range(4)]
    videos = [by_id[vid] for vid in ordered]
    if len(videos) != per_outcome * 4:
        raise ValueError("Screen selection and annotation IDs differ")
    return selected, videos


class RangeUnsupported(RuntimeError):
    pass


class HTTPRangeFile(io.RawIOBase):
    """Seekable bounded HTTP reader for ZIP/ZIP64 central directories and members.

    A server returning 200 to a Range request is rejected BEFORE its body is read.
    ZipFile verifies each extracted member's CRC. Whole-archive SHA is only
    verified when the full-download fallback is used.
    """
    def __init__(self, url, size, session, block_size=8 * 1024**2):
        self.url, self.size, self.session = url, size, session
        self.pos, self.block_size = 0, block_size
        self.buffer_start, self.buffer = -1, b""
        self.bytes_received = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        if pos < 0:
            raise ValueError("Negative seek")
        self.pos = pos
        return pos

    def read(self, count=-1):
        count = max(0, self.size - self.pos) if count is None or count < 0 else min(count, max(0, self.size - self.pos))
        chunks = []
        while count:
            if not self.buffer_start <= self.pos < self.buffer_start + len(self.buffer):
                start = self.pos
                end = min(self.size - 1, start + max(self.block_size, min(count, self.block_size)) - 1)
                for attempt in range(3):
                    try:
                        from urllib.parse import urlsplit
                        report_progress("dataset_range_request", endpoint=urlsplit(self.url).hostname,
                                        request_bytes=end - start + 1, range_start=start,
                                        received_mib=round(self.bytes_received / 1024**2, 2), attempt=attempt + 1)
                        with self.session.get(self.url, headers={"Range": f"bytes={start}-{end}",
                                                               "Accept-Encoding": "identity"},
                                              stream=True, timeout=(30, 120)) as response:
                            expected = f"bytes {start}-{end}/{self.size}"
                            if response.status_code != 206 or response.headers.get("Content-Range") != expected:
                                raise RangeUnsupported(f"Range unavailable or size mismatch, HTTP {response.status_code}")
                            data = response.content
                            if len(data) != end - start + 1:
                                raise OSError("Truncated HTTP range")
                            self.bytes_received += len(data)
                            report_progress("dataset_range_received", received_mib=round(self.bytes_received / 1024**2, 2))
                            self.buffer_start, self.buffer = start, data
                        break
                    except RangeUnsupported:
                        raise
                    except Exception as exc:
                        report_progress("dataset_range_retry", attempt=attempt + 1, error_type=type(exc).__name__)
                        if attempt == 2:
                            raise
                        time.sleep(2 * (attempt + 1))
            offset = self.pos - self.buffer_start
            take = min(count, len(self.buffer) - offset)
            chunks.append(self.buffer[offset:offset + take])
            self.pos += take
            count -= take
        return b"".join(chunks)


def extract_members(archive, videos, directory):
    directory.mkdir(parents=True, exist_ok=True)
    members = {}
    for item in archive.infolist():
        if item.is_dir() or Path(item.filename).name.startswith("._"):
            continue
        match = re.search(r"sample_(\d+)", item.filename)
        if match and item.filename.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            sid = int(match[1])
            if sid in members:
                raise ValueError(f"Ambiguous source members for sample {sid}")
            members[sid] = item
    metadata = []
    for video in videos:
        sid = sample_id(video)
        item = members[sid]
        report_progress("extracting_video", sample_id=sid, expected_mib=round(item.file_size / 1024**2, 2))
        # The archive name is never used as an extraction path.
        dest = directory / f"sample_{sid}_real{Path(item.filename).suffix.lower()}"
        temp = dest.with_suffix(dest.suffix + ".tmp")
        with archive.open(item) as source, temp.open("wb") as target:
            while True:
                block = source.read(4 * 1024**2)
                if not block:
                    break
                target.write(block)
        if temp.stat().st_size != item.file_size:
            raise ValueError(f"Truncated extracted video {sid}")
        os.replace(temp, dest)
        metadata.append(dict(sample_id=sid, path=str(dest.resolve()), bytes=item.file_size,
                             member=item.filename, zip_crc32=item.CRC, sha256=digest(dest)))
        print(f"VIDEO_READY sample={sid} MiB={item.file_size/1024**2:.1f}", flush=True)
    return metadata


def download_full(urls, path):
    import requests
    if path.is_file() and path.stat().st_size == SHARD_BYTES and digest(path) == SHARD_SHA:
        return
    for url in urls:
        for attempt in range(3):
            try:
                start = path.stat().st_size if path.exists() else 0
                if start >= SHARD_BYTES:
                    path.unlink()
                    start = 0
                report_progress("dataset_full_download", cached_mib=round(start / 1024**2, 2))
                with requests.get(url, headers={"Range": f"bytes={start}-", "Accept-Encoding": "identity"},
                                  stream=True, timeout=(30, 180)) as response:
                    response.raise_for_status()
                    if response.status_code == 206:
                        if not response.headers.get("Content-Range", "").startswith(f"bytes {start}-"):
                            raise ValueError("Wrong resume offset")
                        mode = "ab"
                    else:
                        mode = "wb"
                    last = time.monotonic()
                    with path.open(mode) as handle:
                        for block in response.iter_content(4 * 1024**2):
                            handle.write(block)
                            if time.monotonic() - last > 30:
                                report_progress("dataset_full_download", downloaded_mib=round(path.stat().st_size / 1024**2, 2))
                                print(f"Archive {path.stat().st_size/1024**3:.2f}/2.08 GiB", flush=True)
                                last = time.monotonic()
                if path.stat().st_size == SHARD_BYTES and digest(path) == SHARD_SHA:
                    return
                raise ValueError("Full archive size/hash mismatch")
            except Exception as exc:
                print(f"Download retry {attempt+1}: {type(exc).__name__}", flush=True)
                time.sleep(3 * (attempt + 1))
    raise RuntimeError("Dataset download failed on mirror and official endpoint; rerun to resume")


def prepare_assets(videos, work):
    import requests
    directory = work / "videos"
    marker = work / "asset_manifest.json"
    if marker.exists():
        report_progress("verifying_cached_videos", videos=len(videos))
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if {x["sample_id"] for x in saved["videos"]} == {sample_id(v) for v in videos}:
            if all(Path(x["path"]).is_file() and Path(x["path"]).stat().st_size == x["bytes"]
                   and digest(x["path"]) == x["sha256"] for x in saved["videos"]):
                print("Reusing SHA-verified selected videos", flush=True)
                return saved
    from urllib.parse import quote
    urls = [f"https://{host}/datasets/mjuicem/StreamingBench/resolve/main/{quote(SHARD_NAME)}?download=true"
            for host in ("hf-mirror.com", "huggingface.co")]
    for url in urls:
        try:
            from urllib.parse import urlsplit
            report_progress("opening_remote_zip", endpoint=urlsplit(url).hostname, selected_videos=len(videos))
            with requests.Session() as session:
                remote = HTTPRangeFile(url, SHARD_BYTES, session)
                with ZipFile(remote) as archive:
                    metadata = extract_members(archive, videos, directory)
            saved = dict(mode="range_selected_members", videos=metadata, transferred_bytes=remote.bytes_received,
                         archive_expected_sha256=SHARD_SHA, whole_archive_sha256_verified=False,
                         integrity="ZIP member CRC verified; observed video SHA256 saved for resumption",
                         source=url)
            atomic_json(marker, saved)
            return saved
        except Exception as exc:
            print(f"Selective ZIP unavailable ({type(exc).__name__}); trying next endpoint", flush=True)
    print("Falling back to the single 2.08 GiB archive; extracting only selected videos", flush=True)
    archive_path = work / "downloads" / SHARD_NAME
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    download_full(urls, archive_path)
    with ZipFile(archive_path) as archive:
        metadata = extract_members(archive, videos, directory)
    saved = dict(mode="full_archive", videos=metadata, archive_expected_sha256=SHARD_SHA,
                 whole_archive_sha256_verified=True, source=urls)
    atomic_json(marker, saved)
    return saved


def chunk_plan(video, sampled_frames):
    cursor, chunks = 0, []
    for i, qa in enumerate(video["conversations"]):
        end = math.ceil(float(qa["end_time"]) * FPS)
        if end < cursor or end > sampled_frames:
            raise ValueError(f"Invalid prefix bound for sample {sample_id(video)}, question {i}")
        while cursor < end:
            stop = min(cursor + 16, end)
            chunks.append(dict(qa_index=i, start=cursor, stop=stop))
            cursor = stop
    return chunks


def causal_scores(model, input_ids, past_key_values=None):
    """Fresh layerwise causal propagation through actual Qwen2 decoder layers.

    The temporary DynamicCache has separate lists; Qwen2 appends via torch.cat
    and never writes original K/V tensors. The source cache remains unchanged.
    Explicit masks allow different physical cache lengths per layer.
    """
    import torch
    from transformers import DynamicCache
    past = model.kv_cache if past_key_values is None else past_key_values
    cache = DynamicCache.from_legacy_cache(tuple((k, v) for k, v in past))
    hidden = model.get_input_embeddings()(input_ids)
    batch, q_len = input_ids.shape
    starts = model._get_next_start_pos_per_layer()
    model._set_rotary_required_len(max(starts) + q_len)
    original_hooks = dict(model._layer_position_ids)
    weights = []
    try:
        for i, layer in enumerate(model.language_model.model.layers):
            pos = model._build_position_ids(starts[i], q_len, batch)
            model._layer_position_ids[i] = pos
            past_len = past[i][0].shape[2]
            mask = torch.zeros((batch, 1, q_len, past_len + q_len), device=hidden.device, dtype=hidden.dtype)
            future = torch.ones((q_len, q_len), device=hidden.device, dtype=torch.bool).triu(1)
            mask[:, :, :, past_len:].masked_fill_(future, torch.finfo(hidden.dtype).min)
            result = layer(hidden, attention_mask=mask, position_ids=pos, past_key_value=cache,
                           output_attentions=True, use_cache=False)
            hidden = result[0]
            weights.append(result[1])
            if result[1] is None or result[1].shape[-1] != past_len + q_len:
                raise ValueError("Native decoder did not return expected attention")
    finally:
        model._layer_position_ids.clear()
        model._layer_position_ids.update(original_hooks)
    return weights


def reference_scores(model, input_ids):
    import torch
    from transformers import DynamicCache
    starts = model._get_next_start_pos_per_layer()
    q_len, batch = input_ids.shape[1], input_ids.shape[0]
    previous = dict(model._layer_position_ids)
    try:
        for i in range(model.num_layers):
            model._layer_position_ids[i] = model._build_position_ids(starts[i], q_len, batch)
        model._set_rotary_required_len(max(starts) + q_len)
        cache = DynamicCache.from_legacy_cache(tuple((k.clone(), v.clone()) for k, v in model.kv_cache))
        output = model.language_model.model(input_ids=input_ids, past_key_values=cache,
                                            position_ids=model._layer_position_ids[0],
                                            use_cache=False, output_attentions=True,
                                            attention_mask=None, return_dict=True)
        return list(output.attentions)
    finally:
        model._layer_position_ids.clear()
        model._layer_position_ids.update(previous)


def ingest_features(model, features, frames):
    import torch
    from inference.reindex_1d import contiguous_kv
    features = features.to(device=model.device, dtype=model.dtype)
    starts = model._get_next_start_pos_per_layer()
    batch, length = features.shape[:2]
    model._set_rotary_required_len(max(starts) + length)
    try:
        for i in range(model.num_layers):
            model._layer_position_ids[i] = model._build_position_ids(starts[i], length, batch)
        output = model.language_model(inputs_embeds=features, past_key_values=model.kv_cache,
                                      use_cache=True, return_dict=True,
                                      position_ids=model._layer_position_ids[0],
                                      cache_position=None, attention_mask=None)
        model.kv_cache = contiguous_kv(output.past_key_values)
        model._append_position_ids(starts, length)
        model.last_encoded_frames = frames
        model.total_processed_frames += frames
    finally:
        model._layer_position_ids.clear()


def fifo_compress(model):
    import torch
    from inference.reindex_1d import compute_cos_sin_for_positions, rotary_delta, apply_rotary_delta_to_keys_only
    lengths = model._get_cache_seq_len_per_layer()
    if max(lengths) <= model.kv_size:
        return
    threshold = model.language_model.config.max_position_embeddings - 1024
    compact = max(int(p[-1]) for p in model._position_ids_cache) > threshold
    new = []
    for i, (keys, values) in enumerate(model.kv_cache):
        keep = torch.cat((torch.arange(model.visual_start_idx, device=model.device),
                          torch.arange(max(model.visual_start_idx, lengths[i] - model.kv_size), lengths[i], device=model.device)))
        old = model._position_ids_cache[i][keep]
        pos = torch.arange(len(keep), device=model.device) if compact else old
        k = keys.index_select(2, keep)
        if compact:
            co, si = compute_cos_sin_for_positions(model.language_model, len(old), old, keys.dtype, model.device)
            cn, sn = compute_cos_sin_for_positions(model.language_model, len(pos), pos, keys.dtype, model.device)
            cd, sd = rotary_delta(co, si, cn, sn)
            k = apply_rotary_delta_to_keys_only(k, cd, sd)
        new.append((k.contiguous(), values.index_select(2, keep).contiguous()))
        model._position_ids_cache[i] = pos.clone()
    model.kv_cache = new
    model.token_activity_cache = [None] * model.num_layers


def reset(model):
    model.clear_cache()
    model._layer_position_ids.clear()
    model._position_ids_cache = [None] * model.num_layers
    model.token_activity_cache = [None] * model.num_layers
    model.total_processed_frames = 0
    model.encode_init_prompt()


def sync():
    import torch
    torch.cuda.synchronize()


def prepare_features(model, video, work, identity):
    import numpy as np
    import torch
    from decord import VideoReader
    from PIL import Image, ImageDraw
    directory = work / "features" / f"sample_{sample_id(video)}"
    report_progress("preparing_video_features", sample_id=sample_id(video))
    marker = directory / "complete.json"
    if marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if saved["identity"] == identity and all((directory / x["file"]).is_file()
                and digest(directory / x["file"]) == x["sha256"] for x in saved["chunks"]):
            print(f"FEATURE_CACHE reuse sample={sample_id(video)}", flush=True)
            return saved
    directory.mkdir(parents=True, exist_ok=True)
    reader = VideoReader(video["video_path"], num_threads=1)
    fps = round(reader.get_avg_fps())
    step = int(fps / FPS)
    if step < 1:
        raise ValueError("Invalid source fps")
    indices = list(range(0, len(reader), step))
    plan = chunk_plan(video, len(indices))
    timestamps = reader.get_frame_timestamp(indices).tolist()
    frame_rows = [dict(sampled_index=i, source_frame=frame, start_s=float(ts[0]), end_s=float(ts[1]))
                  for i, (frame, ts) in enumerate(zip(indices, timestamps))]
    write_csv(directory / "sampled_timestamps.csv", frame_rows)
    chunks, total_encoder_s = [], 0.0
    for index, item in enumerate(plan):
        sync()
        start = time.perf_counter()
        frames = torch.from_numpy(reader.get_batch(indices[item["start"]:item["stop"]]).asnumpy())
        pixels = model.processor.video_processor(frames.permute(0, 3, 1, 2), return_tensors="pt").pixel_values_videos
        features = model.get_video_features(pixels.to(model.device, model.dtype)).detach().cpu().contiguous()
        sync()
        total_encoder_s += time.perf_counter() - start
        dest = directory / f"chunk_{index:04d}.pt"
        temp = dest.with_suffix(".tmp")
        torch.save(features, temp)
        os.replace(temp, dest)
        chunks.append(dict(**item, file=dest.name, sha256=digest(dest), tokens=features.shape[1]))
        report_progress("preparing_video_features", sample_id=sample_id(video), chunks_done=index + 1,
                        chunks_total=len(plan))
    # Contact sheets are sparse review aids, not evidence annotations.
    review_dir = work / "review"
    review_dir.mkdir(exist_ok=True)
    for qi, qa in enumerate(video["conversations"]):
        report_progress("making_contact_sheets", sample_id=sample_id(video), qa_index=qi)
        end = math.ceil(float(qa["end_time"]) * FPS)
        chosen = sorted(set(np.linspace(0, end - 1, min(12, end), dtype=int).tolist()))
        thumbnails = reader.get_batch([indices[i] for i in chosen]).asnumpy()
        canvas = Image.new("RGB", (4 * 256, 3 * 164), "white")
        draw = ImageDraw.Draw(canvas)
        for n, (frame, si) in enumerate(zip(thumbnails, chosen)):
            im = Image.fromarray(frame)
            im.thumbnail((256, 142))
            x, y = (n % 4) * 256, (n // 4) * 164
            canvas.paste(im, (x, y))
            draw.text((x + 3, y + 144), f"sampled {si} | {timestamps[si][0]:.2f}s", fill="black")
        canvas.save(review_dir / f"sample_{sample_id(video)}_q{qi}.jpg", quality=80)
    saved = dict(identity=identity, sample_id=sample_id(video), source_video_sha256=digest(video["video_path"]),
                 source_fps_rounded=fps, sampling_step=step, chunks=chunks,
                 preparation_s=total_encoder_s, sampled_frames_available=len(indices),
                 note="Preparation includes chunk decode/preprocess/vision/transfer; excludes contact sheets and serialization")
    atomic_json(marker, saved)
    return saved


def checkpoint_valid(path, expected_hash, video, expected_method=None):
    if not path.exists():
        return False
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
        if value.get("config_hash") != expected_hash or value.get("status") != "complete":
            return False
        if expected_method is not None and value.get("method") != expected_method:
            return False
        rows = value["predictions"]
        if len(rows) != len(video["conversations"]):
            return False
        for i, (row, qa) in enumerate(zip(rows, video["conversations"])):
            gold = "ABCDEFGH"[qa["choices"].index(qa["answer"])]
            if not (row["qa_index"] == i and row["video_id"] == str(video["video_id"])
                    and row["question"] == qa["question"] and row["choices"] == qa["choices"]
                    and row["gold_choice"] == gold and row["gold_answer"] == qa["answer"]
                    and row["correct"] == int(row["pred_choice"] == gold)):
                return False
        return sha_predictions(rows) == value["prediction_sha256"]
    except (OSError, ValueError, KeyError, TypeError):
        return False


def sha_predictions(rows):
    return hashlib.sha256(stable(rows).encode()).hexdigest()


def run_method(model, analyzer, video, method, feature_meta, work, config_hash):
    import torch
    output = work / "checkpoints" / method / f"sample_{sample_id(video)}.json"
    report_progress("method_replay", method=method, sample_id=sample_id(video))
    if checkpoint_valid(output, config_hash, video, method):
        print(f"REUSE method={method} sample={sample_id(video)}", flush=True)
        return json.loads(output.read_text(encoding="utf-8"))
    random.seed(SEED)
    torch.manual_seed(SEED)
    reset(model)
    original = model._compute_attention_scores_manually
    if method == "causal":
        model._compute_attention_scores_manually = types.MethodType(causal_scores, model)
    feature_dir = work / "features" / f"sample_{sample_id(video)}"
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    events, rows = [], []
    try:
        for i, qa in enumerate(video["conversations"]):
            report_progress("method_replay", method=method, sample_id=sample_id(video), qa_index=i)
            for item in feature_meta["chunks"]:
                if item["qa_index"] != i:
                    continue
                features = torch.load(feature_dir / item["file"], map_location="cpu", weights_only=True)
                sync()
                before = time.perf_counter()
                ingest_features(model, features, item["stop"] - item["start"])
                sync()
                ingest_s = time.perf_counter() - before
                before = time.perf_counter()
                if method == "fifo":
                    fifo_compress(model)
                else:
                    model.predict_and_compress()
                sync()
                events.append(dict(qa_index=i, sampled_start=item["start"], sampled_stop=item["stop"],
                                   ingest_s=ingest_s, compression_s=time.perf_counter() - before,
                                   cache_lengths=model._get_cache_seq_len_per_layer()))
            gold = "ABCDEFGH"[qa["choices"].index(qa["answer"])]
            sync()
            before = time.perf_counter()
            answer = analyzer.video_close_qa(qa["question"], qa["choices"], gold)
            sync()
            rows.append(dict(video_id=str(video["video_id"]), sample_id=sample_id(video), qa_index=i,
                             query_time_s=qa["end_time"], task=qa["task"], question=qa["question"],
                             choices=qa["choices"], gold_answer=qa["answer"], gold_choice=gold,
                             pred_answer=answer["pred_answer"], pred_choice=answer["pred_choice"],
                             correct=int(answer["acc"]), answer_s=time.perf_counter() - before))
        sync()
        value = dict(status="complete", config_hash=config_hash, method=method,
                     predictions=rows, prediction_sha256=sha_predictions(rows), events=events,
                     replay_wall_s=time.perf_counter() - start,
                     peak_allocated_gib=torch.cuda.max_memory_allocated() / 1024**3,
                     peak_reserved_gib=torch.cuda.max_memory_reserved() / 1024**3)
        atomic_json(output, value)
        if not checkpoint_valid(output, config_hash, video, method):
            raise ValueError("Newly written checkpoint failed independent identity check")
        print(f"METHOD_DONE {method} sample={sample_id(video)} correct={sum(r['correct'] for r in rows)}/{len(rows)} replay_s={value['replay_wall_s']:.1f}", flush=True)
        return value
    finally:
        model._compute_attention_scores_manually = original
        model.clear_cache()


def preflight(model, video, work):
    """Prove feature ingestion and fresh scores agree with original/native paths."""
    import torch
    report_progress("gpu_preflight", sample_id=sample_id(video))
    from decord import VideoReader
    reader = VideoReader(video["video_path"], num_threads=1)
    step = int(round(reader.get_avg_fps()) / FPS)
    indices = list(range(0, len(reader), step))[:16]
    if len(indices) != 16:
        raise ValueError("Preflight needs 16 sampled frames")
    frames = torch.from_numpy(reader.get_batch(indices).asnumpy())
    reset(model)
    original = model.get_video_features
    captured = []
    def capture(pixels):
        features = original(pixels)
        captured.append(features.detach().cpu())
        return features
    model.get_video_features = capture
    try:
        model.encode_video_chunk(frames)
    finally:
        model.get_video_features = original
    cache = tuple((k.clone(), v.clone()) for k, v in model.kv_cache)
    positions = [p.clone() for p in model._position_ids_cache]
    reset(model)
    ingest_features(model, captured[0], len(frames))
    for (ka, va), (kb, vb), pa, pb in zip(cache, model.kv_cache, positions, model._position_ids_cache):
        torch.testing.assert_close(ka, kb, atol=0, rtol=0)
        torch.testing.assert_close(va, vb, atol=0, rtol=0)
        torch.testing.assert_close(pa, pb, atol=0, rtol=0)
    del cache
    comparisons = []
    for stage in ("before_pruning", "after_position_sensitive_pruning"):
        if stage != "before_pruning":
            # Constrain selection to force real pruning and summary positions.
            keep = []
            for i, length in enumerate(model._get_cache_seq_len_per_layer()):
                n = 255 if i >= model.long_term_threshold else 256
                keep.append(torch.cat((torch.arange(model.visual_start_idx, device=model.device),
                                       torch.arange(length - n, length, device=model.device))))
            model.apply_kv_cache_pruning_strict(keep)
        ids = model.processor.tokenizer("Describe objects and earlier events in the scene.", return_tensors="pt").input_ids.to(model.device)
        previous = tuple((k.clone(), v.clone()) for k, v in model.kv_cache)
        before_positions = [p.clone() for p in model._position_ids_cache]
        before_activity = [p.clone() if p is not None else None for p in model.token_activity_cache]
        actual = causal_scores(model, ids)
        expected = reference_scores(model, ids)
        max_error = 0.0
        for a, b in zip(actual, expected):
            torch.testing.assert_close(a, b, atol=2e-3, rtol=2e-3)
            max_error = max(max_error, float((a - b).abs().max()))
            # Explicit causality must give future probe tokens no mass.
            q = ids.shape[1]
            assert not torch.any(torch.triu(a[..., -q:], diagonal=1))
        for (ka, va), (kb, vb), pa, pb in zip(previous, model.kv_cache, before_positions, model._position_ids_cache):
            torch.testing.assert_close(ka, kb, atol=0, rtol=0)
            torch.testing.assert_close(va, vb, atol=0, rtol=0)
            torch.testing.assert_close(pa, pb, atol=0, rtol=0)
        assert all(a is None and b is None or a is not None and torch.equal(a, b)
                   for a, b in zip(before_activity, model.token_activity_cache))
        comparisons.append(dict(stage=stage, max_attention_error=max_error,
                                cache_lengths=model._get_cache_seq_len_per_layer()))
        del actual, expected, previous
    result = dict(status="pass", feature_ingestion="bitwise_equal_KV_and_positions",
                  causal_scorer_comparisons=comparisons, source_cache_unchanged=True)
    atomic_json(work / "preflight.json", result)
    model.clear_cache()
    print("PREFLIGHT_PASS: feature reuse, native attention, causality and cache immutability", flush=True)
    return result


def export_results(results, selected, work, config):
    targets = {r["case_id"] for r in selected}
    flat = [dict(method=x["method"], **r) for x in results for r in x["predictions"]]
    write_csv(work / "exports/predictions.csv", flat)
    summary, pairs = [], []
    for method in METHODS:
        runs = [x for x in results if x["method"] == method]
        rows = [r for x in runs for r in x["predictions"]]
        focus = [r for r in rows if f"sample_{r['sample_id']:03d}_q{r['qa_index']}" in targets]
        summary.append(dict(method=method, videos=len(runs), questions=len(rows),
                            correct=sum(r["correct"] for r in rows),
                            accuracy_pct=100 * sum(r["correct"] for r in rows) / len(rows),
                            selected_targets=len(focus), target_correct=sum(r["correct"] for r in focus),
                            total_replay_s=sum(x["replay_wall_s"] for x in runs),
                            peak_allocated_gib=max(x["peak_allocated_gib"] for x in runs)))
    indexed = {(r["method"], r["video_id"], r["qa_index"]): r for r in flat}
    for row in flat:
        if row["method"] != "hermes":
            continue
        f, c = [indexed[(m, row["video_id"], row["qa_index"])] for m in ("fifo", "causal")]
        pairs.append(dict(sample_id=row["sample_id"], video_id=row["video_id"], qa_index=row["qa_index"],
                          task=row["task"], hermes_correct=row["correct"], fifo_correct=f["correct"],
                          causal_correct=c["correct"], hermes_choice=row["pred_choice"],
                          fifo_choice=f["pred_choice"], causal_choice=c["pred_choice"],
                          is_selected_target=f"sample_{row['sample_id']:03d}_q{row['qa_index']}" in targets))
    write_csv(work / "exports/summary.csv", summary)
    write_csv(work / "exports/paired_questions.csv", pairs)
    atomic_json(work / "exports/run_config.json", config)
    lines = ["# AutoDL causal-scorer screen", "", "Exploratory outcome-selected development reuse. No confirmatory accuracy claim.", "",
             "Shared decode/vision feature preparation is outside method replay timings. These are not full-system streaming latencies.", "",
             "| Method | Correct / QA | Target correct | Replay seconds |", "|---|---:|---:|---:|"]
    for r in summary:
        lines.append(f"| {r['method']} | {r['correct']}/{r['questions']} | {r['target_correct']}/{r['selected_targets']} | {r['total_replay_s']:.1f} |")
    lines += ["", "Evidence intervals, token survival and visual causes remain unannotated. Contact sheets are sparse review aids.",
              "A fresh causal scorer is a reproduction diagnostic. This run does not establish an original algorithm."]
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return summary


def package(work, status):
    import shutil
    from zipfile import ZIP_DEFLATED
    if (work / "progress.json").exists():
        atomic_json(work / "exports/progress_snapshot.json", json.loads((work / "progress.json").read_text(encoding="utf-8")))
    # The notebook writes the live subprocess log concurrently. Freeze copies
    # before hashing, otherwise archive entries can race their manifest hashes.
    for source in (work / "logs").rglob("*"):
        if source.is_file():
            target = work / "captured_logs" / source.relative_to(work / "logs")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    files = []
    for area in ("exports", "checkpoints", "review", "captured_logs"):
        files.extend(p for p in (work / area).rglob("*") if p.is_file())
    files.extend(p for name in ("preflight.json", "asset_manifest.json", "selection.json", "pilot_runner.py", "launch.json")
                 if (p := work / name).is_file())
    files.extend(p for p in (work / "features").rglob("*.csv"))
    files.extend(p for p in (work / "features").rglob("complete.json"))
    manifest = [dict(path=p.relative_to(work).as_posix(), bytes=p.stat().st_size, sha256=digest(p))
                for p in sorted(set(files))]
    atomic_json(work / "evidence_manifest.json", dict(status=status, files=manifest))
    output = work / "HERMES_causal_scorer_pilot_evidence.zip"
    temp = output.with_suffix(".tmp")
    with ZipFile(temp, "w", compression=ZIP_DEFLATED) as archive:
        for item in manifest:
            archive.write(work / item["path"], item["path"])
        archive.write(work / "evidence_manifest.json", "evidence_manifest.json")
    os.replace(temp, output)
    print(f"EVIDENCE_ZIP={output}\nSHA256={digest(output)}", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo", type=Path, required=True)
    parser.add_argument("--work", type=Path, required=True)
    parser.add_argument("--model", type=Path, required=True)
    parser.add_argument("--per-outcome", type=int, default=2, choices=[1, 2, 3])
    parser.add_argument("--max-hours", type=float, default=2.0)
    args = parser.parse_args()
    with ProgressMonitor(args.work):
        execute(args)


def execute(args):
    if args.model.name != MODEL_SHA:
        raise ValueError("Use the pinned model snapshot directory named by MODEL_SHA")
    work, repo = args.work.resolve(), args.repo.resolve()
    sys.path.insert(0, str(repo / "scripts"))
    sys.path.insert(0, str(repo))
    selected, videos = choose_screen(repo, args.per_outcome)
    report_progress("dataset_preparation", selected_videos=len(videos))
    work.mkdir(parents=True, exist_ok=True)
    review_sheet = work / "review/evidence_review.csv"
    if review_sheet.exists() and [r["case_id"] for r in read_csv(review_sheet)] != [r["case_id"] for r in selected]:
        raise ValueError("Existing review sheet has different cases; use a new WORK directory for a changed screen size")
    previous_failure = work / "exports/failure.json"
    if previous_failure.exists():
        history = work / "logs" / f"previous_failure_{time.time_ns()}.json"
        history.parent.mkdir(parents=True, exist_ok=True)
        os.replace(previous_failure, history)
    atomic_json(work / "selection.json", dict(cases=selected, source="201-250 inspected development shard",
                                             per_outcome=args.per_outcome, seed=SEED))
    assets = prepare_assets(videos, work)
    asset_by_id = {x["sample_id"]: x for x in assets["videos"]}
    for v in videos:
        v["video_path"] = asset_by_id[sample_id(v)]["path"]
    atomic_json(work / "exports/replay_annotations.json", videos)
    if not review_sheet.exists():
        write_csv(review_sheet, [dict(case_id=r["case_id"], review_status="pending",
                                    support_intervals_seconds_json="", support_visible_at_original_sampling="",
                                    failure_cause="unclassified", notes="") for r in selected])
    # Pin inference sources and assets in every checkpoint identity.
    report_progress("importing_gpu_libraries")
    import numpy as np
    import torch
    import transformers
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required")
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    source_files = ["inference/llavaov_hermes.py", "inference/abstract_hermes.py", "inference/reindex_1d.py",
                    "video_qa/base.py", "video_qa/hermes_vqa.py", "scripts/build_failure_pilot.py"]
    config = dict(model_id=MODEL_ID, model_revision=MODEL_SHA, kv_size=4000, sample_fps=FPS, seed=SEED,
                  source_sha256={p: digest(repo / p) for p in source_files},
                  runner_sha256=hashlib.sha256(Path(__file__).read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
                  torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0),
                  methods=METHODS, shared_features=True, feature_cache_dtype="float16",
                  assets={str(x["sample_id"]): x["sha256"] for x in assets["videos"]},
                  conversations=[dict(video_id=v["video_id"], conversations=v["conversations"]) for v in videos],
                  selected_case_ids=[r["case_id"] for r in selected],
                  note="Outcome-selected development screen; full-history replay; no population inference")
    config_hash = hashlib.sha256(stable(config).encode()).hexdigest()
    config["config_hash"] = config_hash
    config["soft_budget_hours_this_invocation"] = args.max_hours
    atomic_json(work / "exports/run_config.json", config)
    report_progress("loading_model", model_revision=MODEL_SHA)
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=FPS)
    if model.language_model.config._attn_implementation != "sdpa":
        raise RuntimeError("Current HERMES control must use its original SDPA/manual-score path")
    analyzer = HermesVQA(anno=[], save_dir=str(work), sample_fps=FPS, qa_model=model, qa_processor=processor)
    results, completed_videos = [], 0
    started = time.perf_counter()
    summary = []
    try:
        with torch.inference_mode():
            preflight(model, videos[0], work)
            for index, video in enumerate(videos):
                # Stop only between complete three-method video blocks.
                if index >= 2 and args.max_hours > 0 and time.perf_counter() - started >= args.max_hours * 3600:
                    print("TIME_BUDGET_STOP: valid partial screen; rerun to resume", flush=True)
                    break
                feature_identity = hashlib.sha256(stable(dict(
                    model_revision=MODEL_SHA, source=config["source_sha256"],
                    source_video=asset_by_id[sample_id(video)]["sha256"],
                    conversations=video["conversations"], sample_fps=FPS,
                    runner=config["runner_sha256"], torch=torch.__version__)).encode()).hexdigest()
                meta = prepare_features(model, video, work, feature_identity)
                order = list(METHODS)
                random.Random(SEED + index).shuffle(order)
                for method in order:
                    results.append(run_method(model, analyzer, video, method, meta, work, config_hash))
                completed_videos += 1
                report_progress("video_block_complete", completed_videos=completed_videos, planned_videos=len(videos))
                summary = export_results(results, selected, work, config)
                elapsed = time.perf_counter() - started
                print(f"BLOCK_DONE {completed_videos}/{len(videos)} elapsed_min={elapsed/60:.1f} remaining_estimate_min={(len(videos)-completed_videos)*elapsed/completed_videos/60:.1f}", flush=True)
        status = "complete" if completed_videos == len(videos) else "partial_time_budget"
        atomic_json(work / "exports/completion.json", dict(status=status, videos=completed_videos,
                    planned_videos=len(videos), methods=list(METHODS), summary=summary,
                    config_hash=config_hash, elapsed_s=time.perf_counter() - started))
        report_progress(status, completed_videos=completed_videos, planned_videos=len(videos))
        package(work, status)
        print("PILOT_COMPLETE" if status == "complete" else "PILOT_PARTIAL", flush=True)
    except Exception as exc:
        atomic_json(work / "exports/failure.json", dict(status="failed", type=type(exc).__name__,
                    detail=str(exc)[-1500:], completed_video_blocks=completed_videos))
        atomic_json(work / "exports/completion.json", dict(status="failed", config_hash=config_hash,
                    completed_video_blocks=completed_videos, planned_videos=len(videos)))
        package(work, "failed_partial")
        raise


if __name__ == "__main__":
    main()
'''
ast.parse(UTILS_SOURCE)
assert hashlib.sha256(UTILS_SOURCE.encode()).hexdigest() == 'f91d6d839198e8ebfcf597a8fff1738bcccd89f575c9eb18677711c310977db3'
SHORT_BASE_SOURCE = r'''"""Compare position-preserving physical history removal with masked readout."""
import argparse
import copy
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path

try:
    from provenance import matched_base as m
except ModuleNotFoundError as exc:
    if exc.name != "provenance":
        raise
    import autodl_matched_boundary as m

s, d, p = m.s, m.d, m.p
ARMS = ("full_contextual", "masked_contextual", "short_contextual", "compact_recent4")
ARTIFACT = "HERMES_short_query_evidence.zip"
SOURCE_PATHS = (*m.SOURCE_PATHS, "provenance/matched_base.py")


def cache_bytes(state):
    return sum(t.numel() * t.element_size() for pair in state["kv"] for t in pair)


def make_short(state, start, stop):
    """Remove H without reindexing RoPE or changing any retained keys/values."""
    import torch

    short = s.drop_history(state, start, stop)
    for full_pos, short_pos in zip(state["positions"], short["positions"], strict=True):
        torch.testing.assert_close(short_pos, torch.cat((full_pos[:start], full_pos[stop:])), atol=0, rtol=0)
        if short_pos[-1] != full_pos[-1]:
            raise ValueError("Physical removal changed the next logical position")
    if s.cache_length(short["kv"]) != s.cache_length(state["kv"]) - (stop - start):
        raise ValueError("Unexpected physical short cache length")
    return short


def position_trace(model, state, prompt, mask_span):
    """Use the unchanged QA wrapper and capture actual prefill/decode positions."""
    import torch

    s.restore(model, state)
    base = s.cache_length(state["kv"])
    logical = int(state["positions"][0][-1]) + 1
    calls = []

    def record(_module, _args, kwargs):
        calls.append((s.cache_length(kwargs["past_key_values"]), kwargs["position_ids"].detach()))

    handle = model.language_model.register_forward_pre_hook(record, with_kwargs=True)
    try:
        with s.HistoryMask(model, *mask_span) as mask:
            p.sync()
            before = time.perf_counter()
            answer = model.question_answering(prompt, max_new_tokens=16)
            p.sync()
            elapsed = time.perf_counter() - before
    finally:
        handle.remove()
    if not calls or len(calls) > 17:
        raise ValueError("Missing or oversized query trace")
    trace = []
    for step, (past, positions) in enumerate(calls):
        n = positions.shape[-1]
        expected = torch.arange(logical, logical + n, device=positions.device).view(1, -1)
        if past != base or (step > 0 and n != 1) or not torch.equal(positions, expected):
            raise ValueError("Answer wrapper used incorrect cache lengths or logical positions")
        trace.append(dict(past_tokens=past, logical_start=logical, query_tokens=n))
        base += n
        logical += n
    if mask_span[0] != mask_span[1]:
        if len(mask.events) != len(trace) or any(
            (e["blocked_start"], e["blocked_stop"]) != mask_span
            or (e["past_tokens"], e["query_tokens"]) != (t["past_tokens"], t["query_tokens"])
            for e, t in zip(mask.events, trace, strict=True)
        ):
            raise ValueError("Incomplete answer mask")
    s.assert_state(model, state)
    return answer, elapsed, trace


def preflight(model, bank, meta, ids, work):
    import torch

    m.numerical_preflight(model, bank, meta, ids, work / "exports/matched_preflight.json")
    report = {}

    def compare(target, strict):
        states, detail = m.prepare_states(target, bank, meta, [0], [1])
        full = states["A"]
        start, stop = detail["prompt_tokens"], detail["history_stop"]
        short = make_short(full, start, stop)
        masked, _, _ = s.probe(target, full, ids, start, stop)
        removed, _, _ = s.probe(target, short, ids, 0, 0)
        rows = []
        for step, (a, b) in enumerate(zip(masked, removed, strict=True)):
            rows.append(dict(step=step, argmax_equal=torch.equal(a[:, -1].argmax(-1), b[:, -1].argmax(-1)),
                             **s.error_statistics(a, b, .001 if strict else .02, .001 if strict else .005)))
        report[str(target.dtype)] = rows
        p.atomic_json(work / "exports/short_preflight.json", report)
        if strict:
            for a, b in zip(masked, removed, strict=True):
                torch.testing.assert_close(a, b, atol=.001, rtol=.001)
            if not all(r["argmax_equal"] for r in rows):
                raise ValueError("FP32 short query changed preflight argmax")

    with s.fp32_structural_reference(model) as reference:
        compare(reference, True)
    compare(model, False)
    print("SHORT_QUERY_PREFLIGHT_PASS", flush=True)


def compare_rows(rows):
    groups = {}
    for row in rows:
        groups.setdefault((row["sample_id"], row["qa_index"]), {})[row["method"]] = row
    pairs = []
    for left, right in (("short_contextual", "masked_contextual"), ("short_contextual", "compact_recent4"),
                        ("short_contextual", "full_contextual")):
        matched = [(g[left], g[right]) for g in groups.values()]
        byte_deltas = [b["kv_bytes"] - a["kv_bytes"] for a, b in matched]
        time_deltas = [b["answer_median_s"] - a["answer_median_s"] for a, b in matched]
        equal_work = [delta for (a, b), delta in zip(matched, time_deltas, strict=True)
                      if [len(t) for t in a["traces"]] == [len(t) for t in b["traces"]]]
        pairs.append(dict(left=left, right=right, questions=len(matched),
                          left_only_correct=sum(a["correct"] > b["correct"] for a, b in matched),
                          right_only_correct=sum(a["correct"] < b["correct"] for a, b in matched),
                          choice_changes=sum(a["pred_choice"] != b["pred_choice"] for a, b in matched),
                          median_kv_bytes_saved=statistics.median(byte_deltas),
                          median_paired_answer_s_saved=statistics.median(time_deltas),
                          equal_decode_count_questions=len(equal_work),
                          median_equal_decode_answer_s_saved=statistics.median(equal_work) if equal_work else None))
    return pairs


def export(rows, work):
    p.write_csv(work / "exports/answers.csv", rows)
    pairs = compare_rows(rows)
    p.write_csv(work / "exports/paired.csv", pairs)
    summary = []
    for arm in ARMS:
        group = [row for row in rows if row["method"] == arm]
        summary.append(dict(arm=arm, answered=len(group), correct=sum(r["correct"] for r in group),
                            median_kv_bytes=statistics.median(r["kv_bytes"] for r in group),
                            median_answer_s=statistics.median(r["answer_median_s"] for r in group)))
    p.write_csv(work / "exports/summary.csv", summary)
    lines = ["# Short query cache development run", "",
             "| Arm | Correct | Answered | Median query KV bytes | Median answer seconds |",
             "|---|---:|---:|---:|---:|"]
    lines.extend(f"| {r['arm']} | {r['correct']} | {r['answered']} | {r['median_kv_bytes']} | {r['median_answer_s']:.4f} |" for r in summary)
    lines.extend(["", "Query KV bytes count retained K/V tensors. Answer timing excludes restore and cache construction.",
                  "The same 40 development questions are used. Timing does not measure total live memory or whole-system latency."])
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_block(model, analyzer, video, qi, states, detail, short, feature_hash, old, repeats, warmup=False):
    start, stop = detail["prompt_tokens"], detail["history_stop"]
    sid = video["sample_id"]
    inputs = {
        ARMS[0]: (states["A"], (0, 0)),
        ARMS[1]: (states["A"], (start, stop)),
        ARMS[2]: (short, (0, 0)),
        ARMS[3]: (states["compact"], (0, 0)),
    }
    identity = d.question_identity(video, qi, ARMS[0])
    prompt = analyzer.format_mcqa_prompt(identity["question"], identity["choices"])
    choices, texts, times, traces = ({a: [] for a in ARMS} for _ in range(4))
    orders = []
    rng = random.Random(p.SEED + sid * 31 + qi)
    for rep in range(-int(warmup), repeats):
        order = list(ARMS)
        rng.shuffle(order)
        if rep >= 0:
            orders.append(order)
        for arm in order:
            state, span = inputs[arm]
            answer, elapsed, trace = position_trace(model, state, prompt, span)
            choice = analyzer.extract_characters_regex(answer)
            if rep >= 0:
                choices[arm].append(choice)
                texts[arm].append(answer.replace("\n", ""))
                times[arm].append(elapsed)
                traces[arm].append(trace)
    rows = []
    for arm in ARMS:
        state, _ = inputs[arm]
        expected = old.get((sid, qi, arm))
        row = d.question_identity(video, qi, arm)
        row.update(pred_choice=choices[arm][0], pred_answer=texts[arm][0],
                   correct=int(choices[arm][0] == identity["gold_choice"]),
                   choices_by_repeat=choices[arm], answer_seconds=times[arm], traces=traces[arm],
                   answer_median_s=statistics.median(times[arm]),
                   physical_tokens=s.cache_length(state["kv"]), kv_bytes=cache_bytes(state),
                   query_logical_start=traces[arm][0][0]["logical_start"], feature_fingerprint=feature_hash,
                   historical_choice=expected, historical_matches=choices[arm][0] == expected if expected is not None else None)
        rows.append(row)
    if rows[1]["query_logical_start"] != rows[2]["query_logical_start"]:
        raise ValueError("Short and masked query starts disagree")
    return rows, orders


def package(work, status):
    name = s.ARTIFACT
    try:
        s.ARTIFACT = ARTIFACT
        s.package(work, status)
    finally:
        s.ARTIFACT = name


def execute(args):
    import numpy as np
    import torch
    import transformers

    if (args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128"
            or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available()):
        raise ValueError("Pinned model/CUDA environment required")
    path = args.work / "reference.json"
    reference = json.loads(path.read_text(encoding="utf-8"))
    d.check_reference(reference, args.repo)
    old = {(r["sample_id"], r["qa_index"], r["arm"]): r["choice"] for r in reference["baseline_choices"]}
    if len(old) != 120:
        raise ValueError("Incomplete reference choices")
    config = dict(reference_sha256=p.digest(path), previous_archive_sha256=reference["matched_archive_sha256"],
                  sources={n: p.digest(args.work / n) for n in SOURCE_PATHS}, arms=ARMS, repeats=args.repeats,
                  gpu=torch.cuda.get_device_name(0), model_revision=p.MODEL_SHA, torch=torch.__version__,
                  transformers=transformers.__version__, seed=p.SEED)
    config_hash = d.fingerprint(config)
    config_path = args.work / "exports/run_config.json"
    if config_path.exists() and json.loads(config_path.read_text())["config_hash"] != config_hash:
        raise d.RunConflict("Short query configuration changed; use a new WORK")
    p.atomic_json(config_path, dict(**config, config_hash=config_hash))
    failure = args.work / "exports/failure.json"
    if failure.exists():
        previous = args.work / "logs" / f"previous_failure_{time.time_ns()}.json"
        previous.parent.mkdir(exist_ok=True)
        failure.replace(previous)
    videos = copy.deepcopy(reference["videos"])
    d.acquire_videos(videos, args.reuse, args.work)
    p.atomic_json(args.work / "exports/replay_annotations.json", videos)
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA

    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_short_query_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows, done, started, warmed = [], 0, time.perf_counter(), False
    videos.sort(key=lambda v: (v["sample_id"] != 204, v["sample_id"]))
    with torch.inference_mode():
        for video in videos:
            directory, meta, _ = d.acquire_features(model, video, args.reuse, args.work,
                d.fingerprint(dict(config=config_hash, video=video["video_sha256"])))
            bank = s.c.FeatureBank(directory, meta)
            feature_hash = d.fingerprint(meta)
            if video["sample_id"] == 204:
                p.preflight(model, video, args.work)
                ids = processor.tokenizer("What is visible?", return_tensors="pt").input_ids.to(model.device)
                preflight(model, bank, meta, ids, args.work)
            for qi in range(len(video["conversations"])):
                if done and args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                    break
                sid = video["sample_id"]
                h, r = s.split_frames(video, qi)
                key = d.fingerprint(dict(config=config_hash, feature=feature_hash, sample=sid, qa=qi, h=h, r=r))
                checkpoint = args.work / "checkpoints" / f"sample_{sid}_q{qi}.json"
                if checkpoint.exists():
                    saved = json.loads(checkpoint.read_text(encoding="utf-8"))
                    if saved["key"] != key or d.fingerprint(saved["rows"]) != saved["rows_sha256"]:
                        raise d.RunConflict("Short query checkpoint differs; use a new WORK")
                    block = saved["rows"]
                else:
                    p.report_progress("short_query_question", sample_id=sid, qa_index=qi)
                    p.sync()
                    before = time.perf_counter()
                    states, detail = m.prepare_states(model, bank, meta, h, r)
                    p.sync()
                    preparation_s = time.perf_counter() - before
                    start, stop = detail["prompt_tokens"], detail["history_stop"]
                    before = time.perf_counter()
                    short = make_short(states["A"], start, stop)
                    p.sync()
                    materialize_s = time.perf_counter() - before
                    block, orders = run_block(model, analyzer, video, qi, states, detail, short, feature_hash,
                                              old, args.repeats, warmup=not warmed)
                    warmed = True
                    p.atomic_json(args.work / "preparation" / f"sample_{sid}_q{qi}.json", dict(
                        **detail, history_indices=h, recent_indices=r, short_materialize_s=materialize_s,
                        shared_preparation_s=preparation_s, timing_orders=orders))
                    p.atomic_json(checkpoint, dict(key=key, rows=block, rows_sha256=d.fingerprint(block)))
                    model.clear_cache()
                    del states, short
                if any(row["historical_matches"] is False or len(set(row["choices_by_repeat"])) != 1 for row in block):
                    raise ValueError("Baseline replay or within-arm repeat changed; inspect saved checkpoint")
                if not h and len({row["pred_choice"] for row in block}) != 1:
                    raise ValueError("Empty-history null control disagreed")
                rows.extend(block)
                done += 1
                export(rows, args.work)
                print(f"QUESTION_BLOCK_DONE {done}/40 elapsed_min={(time.perf_counter()-started)/60:.2f}", flush=True)
            del bank
            if args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                break
    status = "complete" if done == 40 else "partial_time_budget"
    if len(rows) != 4 * done:
        raise ValueError("Short query matrix incomplete")
    p.atomic_json(args.work / "exports/completion.json", dict(status=status, main_questions=done,
        main_answers=len(rows), timed_calls=len(rows)*args.repeats, config_hash=config_hash,
        elapsed_post_model_s=time.perf_counter()-started))
    p.report_progress(status, main_questions=done)
    package(args.work, status)
    print(f"SHORT_QUERY_{status.upper()}", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--" + name, type=Path, required=True)
    parser.add_argument("--repeats", type=int, choices=(1, 2, 3), default=2)
    parser.add_argument("--max-hours", type=float, default=1.0)
    args = parser.parse_args()
    for name in ("repo", "work", "model", "reuse"):
        setattr(args, name, getattr(args, name).resolve())
    d.validate_workspaces(args.work, args.reuse)
    if args.max_hours < 0:
        parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work / "exports/failure.json", dict(type=type(exc).__name__, detail=str(exc)[-1500:]))
            p.atomic_json(args.work / "exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(SHORT_BASE_SOURCE)
assert hashlib.sha256(SHORT_BASE_SOURCE.encode()).hexdigest() == 'fb4260f5f6048e4155607d111914526caec381e38526de18068850b74a9c6819'
MATCHED_BASE_SOURCE = r'''"""Isolate history visibility during recent encoding with matched forward shapes."""
import argparse
import copy
import json
import os
import random
import sys
import time
from pathlib import Path

try:
    from provenance import stage_base as s
except ModuleNotFoundError as exc:
    if exc.name != "provenance":
        raise
    import autodl_stage_attribution as s

d, p = s.d, s.p
ARMS = s.ARMS
ARTIFACT = "HERMES_matched_boundary_evidence.zip"
SOURCE_PATHS = ("stage_runner.py", "provenance/stage_base.py", "stage_context_utils.py",
                "context_input_utils.py", "diagnostic_pilot_utils.py")


def prepare_states(model, bank, meta, history, recent):
    """Share the original H prefix with previous B/D, then match all R groups."""
    import torch

    p.reset(model)
    original_groups = s.ingest(model, bank, history + recent, meta)
    original = s.snapshot(model)
    prompt = original["prompt"]
    stop = prompt + len(history) * 196
    if history:
        prefix = s.prefix_state(original, stop)
        del original
        s.restore(model, prefix)
        with s.HistoryMask(model, prompt, prompt, force=True) as visible_mask:
            visible_groups = s.ingest(model, bank, recent, meta)
        visible = s.snapshot(model)
        s.restore(model, prefix)
        with s.HistoryMask(model, prompt, stop) as mask:
            hidden_groups = s.ingest(model, bank, recent, meta)
        hidden = s.snapshot(model)
        trace = mask.events
        visible_trace = visible_mask.events
        if len(trace) != len(hidden_groups) or len(visible_trace) != len(visible_groups):
            raise ValueError("Incomplete construction mask trace")
        if not trace or visible_groups != hidden_groups:
            raise ValueError("Recent encoding groups are not matched")
        for a, b in zip(visible_trace, trace, strict=True):
            if a != dict(b, blocked_stop=prompt):
                raise ValueError("Construction masks differ beyond history visibility")
        for i, ((vk, vv), (hk, hv)) in enumerate(zip(visible["kv"], hidden["kv"])):
            torch.testing.assert_close(visible["positions"][i], hidden["positions"][i], atol=0, rtol=0)
            for left, right in ((vk, hk), (vv, hv)):
                torch.testing.assert_close(left[:, :, :stop], right[:, :, :stop], atol=0, rtol=0)
    else:
        visible = hidden = original
        visible_groups = hidden_groups = original_groups
        visible_trace = trace = []
    d.assert_fresh_cache(model, len(history) + len(recent))
    p.reset(model)
    compact_groups = s.ingest(model, bank, recent, meta)
    compact = s.snapshot(model)
    d.assert_fresh_cache(model, len(recent))
    return dict(A=visible, B=hidden, compact=compact), dict(
        prompt_tokens=prompt, history_stop=stop, original_groups=original_groups,
        visible_recent_groups=visible_groups, hidden_recent_groups=hidden_groups,
        compact_groups=compact_groups, visible_construction_mask_trace=visible_trace, hidden_construction_mask_trace=trace,
        original_group_crosses_boundary=any(set(g).intersection(history) and set(g).intersection(recent)
                                            for g in original_groups),
    )


def flip_history(state, start, stop, length):
    changed = s.prefix_state(state, length)
    for keys, values in changed["kv"]:
        keys[:, :, start:stop].neg_()
        values[:, :, start:stop].neg_()
    return changed


def numerical_preflight(model, bank, meta, ids, path):
    """Keep the audited FP32 physical reference and FP16 same-shape checks."""
    import torch

    report = dict(policy="fp32_physical_reference_and_fp16_same_shape_invariance",
                  fp32={}, fp16=dict(construction=[], logits=[]))

    def save():
        p.atomic_json(path, report)

    if model.dtype != torch.float16:
        raise ValueError("Expected the pinned FP16 model")
    print("MATCHED_PREFLIGHT FP32 physical reference", flush=True)
    with s.fp32_structural_reference(model) as reference:
        report["fp32"]["result"] = s._preflight_pass(
            reference, bank, meta, ids, atol=.001, rtol=.001, kv_atol=.0001, kv_rtol=.0001,
            diagnostics=report["fp32"], save=save,
        )
    save()

    print("MATCHED_PREFLIGHT FP16 same-shape invariance", flush=True)
    states, detail = prepare_states(model, bank, meta, [0], [1])
    start, stop = detail["prompt_tokens"], detail["history_stop"]
    hidden = states["B"]
    s.restore(model, flip_history(hidden, start, stop, stop))
    with s.HistoryMask(model, start, stop):
        s.ingest(model, bank, [1], meta)
    changed = s.snapshot(model)
    for layer, ((k, v), (ck, cv)) in enumerate(zip(hidden["kv"], changed["kv"])):
        for kind, a, b in (("K", k, ck), ("V", v, cv)):
            report["fp16"]["construction"].append(dict(
                layer=layer, kind=kind, **s.error_statistics(a[:, :, stop:], b[:, :, stop:], 0, 0)))
    save()
    if any(record["max_abs"] != 0 for record in report["fp16"]["construction"]):
        raise ValueError("Hidden history changed FP16 recent K/V")

    def check_logits(label, left, right):
        for step, (a, b) in enumerate(zip(left, right, strict=True)):
            equal = torch.equal(a[:, -1].argmax(-1), b[:, -1].argmax(-1))
            report["fp16"]["logits"].append(dict(
                comparison=label, step=step, argmax_equal=equal, **s.error_statistics(a, b, .02, .005)))
            save()
            if not equal:
                raise ValueError(f"FP16 preflight argmax changed: {label}, step {step}")

    # Use the same original-partition state as the audited native/explicit control.
    p.reset(model)
    s.ingest(model, bank, [0, 1], meta)
    original = s.snapshot(model)
    native, _, _ = s.probe(model, original, ids, 0, 0)
    explicit, _, _ = s.probe(model, original, ids, 0, 0, force=True)
    check_logits("native_vs_all_visible_mask", native, explicit)
    for label, reference_state in (("original_contextual", original), ("matched_independent", hidden)):
        masked, _, _ = s.probe(model, reference_state, ids, start, stop)
        compact, _, _ = s.probe(model, s.drop_history(reference_state, start, stop), ids, 0, 0)
        check_logits(label + "_physical_removal", masked, compact)
    for state in (states["A"], hidden):
        masked, _, events = s.probe(model, state, ids, start, stop)
        perturbed, _, _ = s.probe(model, flip_history(state, start, stop, s.cache_length(state["kv"])),
                                  ids, start, stop)
        for a, b in zip(masked, perturbed, strict=True):
            torch.testing.assert_close(a, b, atol=0, rtol=0)
        if len(events) != 3 or [e["query_tokens"] for e in events][1:] != [1, 1]:
            raise ValueError("Missing FP16 prefill/decode mask checks")
    _, weights, _ = s.probe(model, hidden, ids, start, stop, attentions=True)
    if len(weights) != 3 or any(len(layers) != model.num_layers for layers in weights):
        raise ValueError("Incomplete FP16 attention trace")
    for layers in weights:
        for attention in layers:
            if torch.any(attention[..., start:stop]) or torch.any(torch.triu(attention[..., -attention.shape[-2]:], 1)):
                raise ValueError("FP16 attention reached hidden history or future tokens")
    s.assert_state(model, hidden)
    model.clear_cache()
    report.update(status="pass", fp16_masked_readout_exact=True, fp16_hidden_and_future_attention_zero=True)
    save()


def export(rows, old, work):
    s.export(rows, work)
    current = {(r["sample_id"], r["qa_index"], r["method"]): r for r in rows}
    comparisons = [
        ("primary_C_matched_vs_D", "current", 2, "current", 3),
        ("A_matched_vs_B", "current", 0, "current", 1),
        ("C_matched_vs_original_C", "current", 2, "previous", 2),
        ("A_matched_vs_original_A", "current", 0, "previous", 0),
        ("C_matched_vs_compact_recent4", "current", 2, "current", 4),
        ("C_matched_vs_A_matched_readout", "current", 2, "current", 0),
    ]
    details, summary = [], []
    for label, left_source, left_arm, right_source, right_arm in comparisons:
        left_rows = current if left_source == "current" else old
        right_rows = current if right_source == "current" else old
        matched = []
        for sid, qi, method in sorted(current):
            if method != ARMS[left_arm] or (sid, qi, ARMS[right_arm]) not in right_rows:
                continue
            a, b = left_rows[sid, qi, method], right_rows[sid, qi, ARMS[right_arm]]
            matched.append(dict(contrast=label, sample_id=sid, qa_index=qi, history_empty=a["history_empty"],
                                left_choice=a["pred_choice"], right_choice=b["pred_choice"],
                                left_correct=a["correct"], right_correct=b["correct"]))
        details.extend(matched)
        summary.append(dict(contrast=label, questions=len(matched),
                            left_only_correct=sum(r["left_correct"] > r["right_correct"] for r in matched),
                            right_only_correct=sum(r["left_correct"] < r["right_correct"] for r in matched),
                            choice_changes=sum(r["left_choice"] != r["right_choice"] for r in matched)))
    p.write_csv(work / "exports/matched_pairs.csv", summary)
    p.write_csv(work / "exports/matched_question_pairs.csv", details)
    report_path = work / "exports/report.md"
    base_report = report_path.read_text(encoding="utf-8").replace(
        "Long 204/q3 results are separate from the main 40-question matrix. A and compact must replay earlier choices or execution stops.",
        "B, D and compact recent-4 must replay earlier choices. This run uses the 40 main questions.",
    )
    with report_path.open("w", encoding="utf-8") as handle:
        handle.write(base_report)
        handle.write("\n## Matched construction contrasts\n\n")
        handle.write("A and C here split recent encoding at the H/R boundary and use an explicit causal mask. "
                     "The primary contrast is C versus D with identical recent groups and hidden history at readout.\n\n")
        handle.write("| Contrast | Questions | Left wins | Right wins | Choice changes |\n|---|---:|---:|---:|---:|\n")
        for row in summary:
            handle.write("| " + " | ".join(map(str, row.values())) + " |\n")


def package(work, status):
    original_name = s.ARTIFACT
    try:
        s.ARTIFACT = ARTIFACT
        s.package(work, status)
    finally:
        s.ARTIFACT = original_name


def execute(args):
    import numpy as np
    import torch
    import transformers

    reference = json.loads((args.work / "reference.json").read_text(encoding="utf-8"))
    d.check_reference(reference, args.repo)
    old = {(r["sample_id"], r["qa_index"], r["method"]): r for r in reference["stage_rows"]}
    if len(old) != 200 or args.long_case:
        raise ValueError("Expected the 40-question matched-boundary experiment only")
    if (args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128"
            or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available()):
        raise ValueError("Pinned model/CUDA environment required")
    config = dict(reference_sha256=p.digest(args.work / "reference.json"),
                  sources={name: p.digest(args.work / name) for name in SOURCE_PATHS},
                  previous_stage_archive_sha256=reference["stage_archive_sha256"],
                  model_revision=p.MODEL_SHA, torch=torch.__version__, transformers=transformers.__version__,
                  gpu=torch.cuda.get_device_name(0), seed=p.SEED, arms=ARMS, main_window=20, recent_window=4,
                  construction="Shared native H prefix; identical R groups; H visibility is the only construction intervention",
                  primary_contrast="C_matched_vs_D", exact_replay_arms=[ARMS[1], ARMS[3], ARMS[4]])
    config_hash = d.fingerprint(config)
    config_path = args.work / "exports/run_config.json"
    if config_path.exists() and json.loads(config_path.read_text())["config_hash"] != config_hash:
        raise d.RunConflict("Matched-boundary config changed; use a new WORK")
    p.atomic_json(config_path, dict(**config, config_hash=config_hash))
    failure = args.work / "exports/failure.json"
    if failure.exists():
        previous_failure = args.work / "logs" / f"previous_failure_{time.time_ns()}.json"
        previous_failure.parent.mkdir(exist_ok=True)
        failure.replace(previous_failure)
    videos = copy.deepcopy(reference["videos"])
    d.acquire_videos(videos, args.reuse, args.work)
    p.atomic_json(args.work / "exports/replay_annotations.json", videos)
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA

    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    random.seed(p.SEED)
    np.random.seed(p.SEED)
    torch.manual_seed(p.SEED)
    p.report_progress("loading_matched_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows, done, started = [], 0, time.perf_counter()
    videos.sort(key=lambda v: (v["sample_id"] != 204, v["sample_id"]))
    with torch.inference_mode():
        for video in videos:
            directory, meta, _ = d.acquire_features(model, video, args.reuse, args.work,
                d.fingerprint(dict(config=config_hash, video=video["video_sha256"])))
            bank, feature_hash = s.c.FeatureBank(directory, meta), d.fingerprint(meta)
            if video["sample_id"] == 204:
                p.preflight(model, video, args.work)
                ids = processor.tokenizer("What is visible?", return_tensors="pt").input_ids.to(model.device)
                p.report_progress("matched_mask_position_preflight")
                numerical_preflight(model, bank, meta, ids, args.work / "exports/matched_preflight.json")
                print("MATCHED_PREFLIGHT_PASS", flush=True)
            for qi in range(len(video["conversations"])):
                if done and args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                    break
                h, r = s.split_frames(video, qi)
                torch.cuda.reset_peak_memory_stats()
                before = time.perf_counter()
                states, detail = prepare_states(model, bank, meta, h, r)
                p.sync()
                detail.update(history_indices=h, recent_indices=r, shared_preparation_s=time.perf_counter() - before,
                              preparation_peak_allocated_gib=torch.cuda.max_memory_allocated() / 1024**3)
                sid = video["sample_id"]
                p.atomic_json(args.work / "preparation/main" / f"sample_{sid}_q{qi}.json", detail)
                block = []
                for i in (1, 3, 4, 0, 2):
                    arm = ARMS[i]
                    previous = old[sid, qi, arm]["pred_choice"] if i in (1, 3, 4) else None
                    block.append(s.run_answer(model, analyzer, video, qi, arm, "main", states, detail,
                                              h, r, feature_hash, config_hash, args, previous))
                if not h and len({row["pred_choice"] for row in block}) != 1:
                    raise ValueError("Empty-history null control disagreed")
                rows.extend(block)
                del states
                done += 1
                export(rows, old, args.work)
                print(f"QUESTION_BLOCK_DONE {done}/40 elapsed_min={(time.perf_counter() - started) / 60:.2f}", flush=True)
            del bank
            if args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                break
    status = "complete" if done == 40 else "partial_time_budget"
    if len(rows) != 5 * done or any(row["status"] != "answered" for row in rows):
        raise ValueError("Matched-boundary matrix incomplete")
    p.atomic_json(args.work / "exports/completion.json", dict(status=status, main_questions=done,
        main_answers=len(rows), exact_replays=3 * done, config_hash=config_hash,
        elapsed_post_model_s=time.perf_counter() - started))
    p.report_progress(status, main_questions=done)
    package(args.work, status)
    print(f"MATCHED_BOUNDARY_{status.upper()}", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--" + name, type=Path, required=True)
    parser.add_argument("--long-case", type=int, choices=(0,), default=0)
    parser.add_argument("--max-hours", type=float, default=1.0)
    args = parser.parse_args()
    for name in ("repo", "work", "model", "reuse"):
        setattr(args, name, getattr(args, name).resolve())
    d.validate_workspaces(args.work, args.reuse)
    if args.max_hours < 0:
        parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work / "exports/failure.json", dict(type=type(exc).__name__, detail=str(exc)[-1500:]))
            p.atomic_json(args.work / "exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(MATCHED_BASE_SOURCE)
assert hashlib.sha256(MATCHED_BASE_SOURCE.encode()).hexdigest() == '73abd3549caf36524fbdf4a9e1cf6e0a28f852762aed690125ef5f05f308c495'
SELECTION_BASE_SOURCE = r'''"""Test bounded, visual-only history selection for recent-cache construction."""
import argparse
import copy
import json
import math
import os
import random
import statistics
import sys
import time
from pathlib import Path

try:
    from provenance import short_base as q
except ModuleNotFoundError as exc:
    if exc.name != "provenance":
        raise
    import autodl_short_query as q

s, d, p = q.s, q.d, q.p
ARMS = ("selected_two", "latest_two", "attention_mass_two", "no_history", "all_history", "compact_recent4")
ARTIFACT = "HERMES_construction_selection_evidence.zip"
SOURCE_PATHS = (*q.SOURCE_PATHS, "provenance/short_base.py")


def attention_stats(logits, values):
    """Sufficient statistics for one attention key group."""
    import torch

    return torch.logsumexp(logits, dim=-1), logits.softmax(dim=-1) @ values


def merge_stats(left, right):
    import torch

    left_z, left_mean = left
    right_z, right_mean = right
    total = torch.logaddexp(left_z, right_z)
    return total, ((left_z - total).exp().unsqueeze(-1) * left_mean
                   + (right_z - total).exp().unsqueeze(-1) * right_mean)


def select_groups(base, older, count=2):
    """Greedily preserve the full first-layer output; break ties by recency."""
    import torch

    target = base
    for group in older:
        target = merge_stats(target, group)
    masses = [float((z - target[0]).exp().mean()) for z, _ in older]
    selected, gains = [], []
    running = base
    for _ in range(min(len(older), count)):
        candidates = [i for i in range(len(older)) if i not in selected]
        errors = torch.stack([
            (merge_stats(running, older[i])[1] - target[1]).square().mean() for i in candidates
        ]).tolist()
        index, error = min(zip(candidates, errors, strict=True), key=lambda x: (x[1], -x[0]))
        selected.append(index)
        gains.append(error)
        running = merge_stats(running, older[index])
    mass_order = sorted(range(len(older)), key=lambda i: (masses[i], i), reverse=True)[:count]
    def residual(indices):
        result = base
        for index in indices:
            result = merge_stats(result, older[index])
        return float((result[1] - target[1]).square().mean())

    return selected, mass_order, dict(
        attention_mass=masses, selected_residual_mse=gains,
        no_history_residual_mse=residual([]), mass_residual_mse=residual(mass_order),
        latest_residual_mse=residual(range(max(0, len(older) - count), len(older))),
    )


def score_history(model, bank, prefix, recent, count):
    """Project the recent visual tokens once and score only real, pre-query history."""
    import torch
    from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb, repeat_kv

    layer = model.language_model.model.layers[0]
    attention = layer.self_attn
    x = layer.input_layernorm(bank.get(recent).to(device=model.device, dtype=model.dtype))
    batch, n, _ = x.shape

    def project(projection, heads):
        return projection(x).view(batch, n, heads, attention.head_dim).transpose(1, 2)

    query = project(attention.q_proj, attention.num_heads)
    key = project(attention.k_proj, attention.num_key_value_heads)
    value = project(attention.v_proj, attention.num_key_value_heads)
    start = int(prefix["positions"][0][-1]) + 1
    model._set_rotary_required_len(start + n)
    cos, sin = attention.rotary_emb(value, seq_len=start + n)
    query, key = apply_rotary_pos_emb(query, key, cos, sin, model._build_position_ids(start, n, batch))
    pk, pv = prefix["kv"][0]
    groups = attention.num_key_value_groups
    query = query.float() / math.sqrt(attention.head_dim)
    first = prefix["prompt"]
    past_key = repeat_kv(pk, groups).float()
    past_value = repeat_kv(pv, groups).float()
    recent_key = repeat_kv(key, groups).float()
    recent_value = repeat_kv(value, groups).float()
    current = (query @ recent_key.transpose(-1, -2)).masked_fill(
        torch.ones((n, n), device=query.device, dtype=torch.bool).triu(1), float("-inf"))
    sink = query @ past_key[:, :, :first].transpose(-1, -2)
    base = attention_stats(torch.cat((sink, current), dim=-1),
                           torch.cat((past_value[:, :, :first], recent_value), dim=2))
    older = []
    for offset in range(first, pk.shape[2], 196):
        k = past_key[:, :, offset:offset + 196]
        v = past_value[:, :, offset:offset + 196]
        older.append(attention_stats(query @ k.transpose(-1, -2), v))
    return select_groups(base, older, count)


def slice_prefix(prefix, frames):
    import torch

    start = prefix["prompt"]
    device = prefix["positions"][0].device
    indices = [torch.arange(start, device=device)]
    indices.extend(torch.arange(start + i * 196, start + (i + 1) * 196, device=device) for i in sorted(frames))
    keep = torch.cat(indices)
    return dict(prompt=start, kv=[(k.index_select(2, keep), v.index_select(2, keep)) for k, v in prefix["kv"]],
                positions=[x.index_select(0, keep) for x in prefix["positions"]])


def construction(model, bank, meta, prefix, recent, selected, logical_start):
    """Encode recent frames once with physically selected H at unchanged positions."""
    support = slice_prefix(prefix, selected)
    s.restore(model, support)
    start = prefix["prompt"]
    # All arms use the same explicit causal mask path. The older span is physically selected.
    with s.HistoryMask(model, start, start, force=True):
        groups = s.ingest(model, bank, recent, meta, logical_start=logical_start)
    encoded = s.snapshot(model)
    stop = start + 196 * len(selected)
    expected = logical_start + 196 * len(recent)
    if any(int(pos[-1]) + 1 != expected or int(pos[stop]) != logical_start for pos in encoded["positions"]):
        raise ValueError("Context construction changed logical positions")
    return encoded, groups, stop


def question(model, analyzer, bank, meta, video, qi, history, recent, old):
    sid = video["sample_id"]
    p.sync()
    started = time.perf_counter()
    p.reset(model)
    s.ingest(model, bank, history + recent, meta)
    original = s.snapshot(model)
    start = original["prompt"]
    logical_start = start + 196 * len(history)
    prefix = s.prefix_state(original, logical_start)
    p.sync()
    shared_prefix_s = time.perf_counter() - started

    before = time.perf_counter()
    chosen, mass, trace = score_history(model, bank, prefix, recent, 2) if history else ([], [], {})
    p.sync()
    scoring_s = time.perf_counter() - before if history else 0.0
    selections = dict(selected_two=chosen, latest_two=list(range(max(0, len(history) - 2), len(history))),
                      attention_mass_two=mass, no_history=[], all_history=list(range(len(history))), compact_recent4=[])
    base_identity = d.question_identity(video, qi, ARMS[0])
    prompt = analyzer.format_mcqa_prompt(base_identity["question"], base_identity["choices"])
    order = list(ARMS)
    random.Random(p.SEED + sid * 31 + qi).shuffle(order)
    rows = {}
    for arm in order:
        selected = selections[arm]
        p.sync()
        before = time.perf_counter()
        if arm == "compact_recent4":
            p.reset(model)
            groups = s.ingest(model, bank, recent, meta)
            encoded, stop = s.snapshot(model), start
        elif not history:
            encoded, groups, stop = original, s.c.input_groups(recent, "original", meta), start
        else:
            encoded, groups, stop = construction(model, bank, meta, prefix, recent, selected, logical_start)
        p.sync()
        encoding_s = time.perf_counter() - before
        before = time.perf_counter()
        state = q.make_short(encoded, start, stop) if stop > start else encoded
        p.sync()
        extraction_s = time.perf_counter() - before if stop > start else 0.0
        answer, answer_s, positions = q.position_trace(model, state, prompt, (0, 0))
        choice = analyzer.extract_characters_regex(answer)
        expected = old.get((sid, qi, arm))
        row = d.question_identity(video, qi, arm)
        row.update(pred_choice=choice, pred_answer=answer.replace("\n", ""), correct=int(choice == row["gold_choice"]),
                   feature_fingerprint=d.fingerprint(meta),
                   history_indices=history, recent_indices=recent, selected_history=[history[i] for i in sorted(selected)],
                   recent_groups=groups, position_trace=positions, query_kv_bytes=q.cache_bytes(state),
                   construction_history_slots=196 * len(selected), construction_s=encoding_s,
                   score_s=scoring_s if arm in ("selected_two", "attention_mass_two") else 0.0,
                   extraction_s=extraction_s, answer_s=answer_s,
                   reference_choice=expected, reference_matches=choice == expected if expected is not None else None)
        rows[arm] = row
        model.clear_cache()
    if len({row["query_kv_bytes"] for row in rows.values()}) != 1:
        raise ValueError("Answer cache budget differs between arms")
    if len({rows[arm]["position_trace"][0]["logical_start"] for arm in ARMS[:-1]}) != 1:
        raise ValueError("Experimental arms have different query positions")
    if len({tuple(map(tuple, row["recent_groups"])) for row in rows.values()}) != 1:
        raise ValueError("Recent construction groups differ between arms")
    choices_by_subset = {}
    for arm in ARMS[:-1]:
        row = rows[arm]
        subset = tuple(row["selected_history"])
        if choices_by_subset.setdefault(subset, row["pred_choice"]) != row["pred_choice"]:
            raise ValueError("Same construction subset produced different choices")
    return [rows[arm] for arm in ARMS], dict(history_indices=history, recent_indices=recent,
        shared_prefix_s=shared_prefix_s, score_s=scoring_s, score_trace=trace, answer_order=order)


def export(rows, work):
    p.write_csv(work / "exports/answers.csv", rows)
    grouped = {}
    for row in rows:
        grouped.setdefault((row["sample_id"], row["qa_index"]), {})[row["method"]] = row
    summary, paired = [], []
    for arm in ARMS:
        subset = [group[arm] for group in grouped.values()]
        summary.append(dict(arm=arm, answered=len(subset), correct=sum(r["correct"] for r in subset),
                            median_query_kv_bytes=statistics.median(r["query_kv_bytes"] for r in subset),
                            median_score_and_construction_s=statistics.median(r["score_s"]+r["construction_s"] for r in subset)))
        if arm == "selected_two":
            continue
        pairs = [(g["selected_two"], g[arm]) for g in grouped.values()]
        paired.append(dict(candidate="selected_two", baseline=arm, questions=len(pairs),
                           candidate_only_correct=sum(a["correct"] > b["correct"] for a, b in pairs),
                           baseline_only_correct=sum(a["correct"] < b["correct"] for a, b in pairs),
                           choice_changes=sum(a["pred_choice"] != b["pred_choice"] for a, b in pairs),
                           median_score_and_construction_s_saved=statistics.median(
                               b["score_s"] + b["construction_s"] - a["score_s"] - a["construction_s"] for a, b in pairs)))
    p.write_csv(work / "exports/summary.csv", summary)
    p.write_csv(work / "exports/paired.csv", paired)
    lines = ["# Construction selection development run", "", "| Arm | Correct | Answered |", "|---|---:|---:|"]
    lines.extend(f"| {r['arm']} | {r['correct']} | {r['answered']} |" for r in summary)
    lines.extend(["", "All arms have the same answer K/V budget. The shared history build is excluded from paired construction timing.",
                  "Attention-mass timing conservatively charges the shared scorer. Its standalone cost may be lower."])
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def execute(args):
    import torch
    import transformers

    ref_path = args.work / "reference.json"
    ref = json.loads(ref_path.read_text(encoding="utf-8"))
    d.check_reference(ref, args.repo)
    old = {(r["sample_id"], r["qa_index"], r["arm"]): r["choice"] for r in ref["baseline_choices"]}
    if (len(old) != 80 or args.model.name != p.MODEL_SHA or not torch.cuda.is_available()
            or torch.__version__ != "2.10.0+cu128" or transformers.__version__ != "4.45.0.dev0"):
        raise ValueError("Expected pinned model, runtime and 80 reference choices")
    config = dict(reference_sha256=p.digest(ref_path), sources={n: p.digest(args.work/n) for n in SOURCE_PATHS},
                  model=p.MODEL_SHA, gpu=torch.cuda.get_device_name(0), arms=ARMS, history_budget_frames=2, seed=p.SEED,
                  torch=torch.__version__, transformers=transformers.__version__,
                  short_query_archive_sha256=ref["short_query_archive_sha256"])
    config_hash = d.fingerprint(config)
    path = args.work / "exports/run_config.json"
    if path.exists() and json.loads(path.read_text())["config_hash"] != config_hash:
        raise d.RunConflict("Construction selection config changed; use new WORK")
    p.atomic_json(path, dict(**config, config_hash=config_hash))
    videos = copy.deepcopy(ref["videos"])
    d.acquire_videos(videos, args.reuse, args.work)
    p.atomic_json(args.work / "exports/replay_annotations.json", videos)
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA

    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.manual_seed(p.SEED)
    p.report_progress("loading_selection_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    rows, done, started = [], 0, time.perf_counter()
    videos.sort(key=lambda v: (v["sample_id"] != 204, v["sample_id"]))
    with torch.inference_mode():
        for video in videos:
            directory, meta, _ = d.acquire_features(model, video, args.reuse, args.work,
                                                   d.fingerprint(dict(config=config_hash, video=video["video_sha256"])))
            bank = s.c.FeatureBank(directory, meta)
            sid = video["sample_id"]
            if sid == 204:
                p.preflight(model, video, args.work)
                ids = processor.tokenizer("What is visible?", return_tensors="pt").input_ids.to(model.device)
                q.preflight(model, bank, meta, ids, args.work)
            for qi in range(len(video["conversations"])):
                if done and args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                    break
                h, r = s.split_frames(video, qi)
                key = d.fingerprint(dict(config=config_hash, feature=d.fingerprint(meta), sample=sid, qa=qi, h=h, r=r))
                dest = args.work / "checkpoints" / f"sample_{sid}_q{qi}.json"
                if dest.exists():
                    saved = json.loads(dest.read_text())
                    if saved["key"] != key or d.fingerprint(saved["rows"]) != saved["rows_sha256"]:
                        raise d.RunConflict("Construction selection checkpoint differs; use new WORK")
                    block = saved["rows"]
                else:
                    p.report_progress("selection_question", sample_id=sid, qa_index=qi)
                    block, plan = question(model, analyzer, bank, meta, video, qi, h, r, old)
                    p.atomic_json(args.work / "preparation" / dest.name, plan)
                    p.atomic_json(dest, dict(key=key, rows=block, rows_sha256=d.fingerprint(block)))
                if any(row["reference_matches"] is False for row in block):
                    raise ValueError("Historical all-history or recent-four choice changed; inspect saved checkpoint")
                if not h and len({row["pred_choice"] for row in block}) != 1:
                    raise ValueError("Empty-history null control disagreed")
                rows.extend(block)
                done += 1
                export(rows, args.work)
                print(f"QUESTION_BLOCK_DONE {done}/40", flush=True)
            del bank
            if args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                break
    status = "complete" if done == 40 else "partial_time_budget"
    if len(rows) != len(ARMS) * done:
        raise ValueError("Construction selection matrix incomplete")
    p.atomic_json(args.work / "exports/completion.json", dict(status=status, main_questions=done, main_answers=len(rows),
                   config_hash=config_hash, elapsed_post_model_s=time.perf_counter() - started))
    p.report_progress(status, main_questions=done)
    package(args.work, status)
    print(f"CONSTRUCTION_SELECTION_{status.upper()}", flush=True)


def package(work, status):
    original = s.ARTIFACT
    try:
        s.ARTIFACT = ARTIFACT
        s.package(work, status)
    finally:
        s.ARTIFACT = original


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--" + name, type=lambda value: Path(value).resolve(), required=True)
    parser.add_argument("--max-hours", type=float, default=1.0)
    args = parser.parse_args()
    if args.max_hours < 0:
        parser.error("Invalid soft budget")
    d.validate_workspaces(args.work, args.reuse)
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work / "exports/failure.json", dict(type=type(exc).__name__, detail=str(exc)[-1500:]))
            p.atomic_json(args.work / "exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(SELECTION_BASE_SOURCE)
assert hashlib.sha256(SELECTION_BASE_SOURCE.encode()).hexdigest() == 'cff497c924236bdb2148c6f259c8e5eec2c930435cbc37845989c5c265541a9b'
RUNNER_SOURCE = r'''"""Evaluate target/context verification under a fixed four-frame answer budget."""
import argparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import statistics
import sys
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote
from zipfile import BadZipFile, ZipFile

try:
    from provenance import confirmation_base as base
except ModuleNotFoundError as exc:
    if exc.name != "provenance":
        raise
    import autodl_short_query_confirmation as base

p, s, d, q = base.p, base.s, base.d, base.q
ARTIFACT = "HERMES_target_binding_evidence.zip"
SOURCE_PATHS = (*base.SOURCE_PATHS, "provenance/confirmation_base.py")
ARMS = ("recent", "previous", "global_relevance", "target_only", "mean", "conjunction")
PROBES = {
    "relevance": "Do these frames visibly contain information relevant to answering the question?",
    "target": "Is the specific subject named or referred to in the question visually identifiable in these frames?",
    "binding": "Do these frames show the requested action, measurement, or condition for that exact subject? "
               "Choose No if it is shown only for a different subject or setting.",
}
ARM_PROBES = {"recent": (), "previous": (), "global_relevance": ("relevance",), "target_only": ("target",),
              "mean": ("target", "binding"), "conjunction": ("target", "binding")}
TEXT_TASK = "Text-Rich Understanding"


def frozen_videos(manifest, repo):
    raw = (repo / manifest["annotation_path"]).read_bytes().replace(b"\r\n", b"\n")
    if hashlib.sha256(raw).hexdigest() != manifest["annotation_sha256"]:
        raise ValueError("Annotation digest differs from the frozen manifest")
    source = {p.sample_id(v): v for v in json.loads(raw)}
    videos = []
    for shard in manifest["shards"]:
        for clip in shard["clips"]:
            sid = clip["sample_id"]
            video = dict(source[sid], sample_id=sid, shard=shard["shard"])
            questions = video["conversations"]
            if (video["video_id"] != clip["video_id"] or Path(video["video_path"]).name != clip["video_basename"]
                    or len(questions) != clip["question_count"]
                    or sum(d.prefix_end(video, i) >= 8 for i in range(len(questions))) != clip["primary_count"]
                    or any(not 2 <= len(a["choices"]) <= 8 or a["choices"].count(a["answer"]) != 1 for a in questions)):
                raise ValueError(f"Frozen identity differs for sample {sid}")
            videos.append(video)
    ids = {v["sample_id"] for v in videos}
    if len(videos) != manifest["clips"] or len(ids) != len(videos) or ids.intersection(manifest["excluded_diagnostic_sample_ids"]):
        raise ValueError("Frozen cohort has missing or reused clips")
    return videos


def check_media_space(shards, work, reserve=4 * 1024**3):
    """Read ZIP indexes to budget selected members plus a work-space reserve."""
    import requests

    remaining = 0
    for shard in shards:
        receipt = work / "provenance" / "media" / f"{shard['shard']}.json"
        metadata = json.loads(receipt.read_text()) if receipt.exists() else {}
        wanted = {c["sample_id"] for c in shard["clips"]}
        for clip in metadata.get("videos", []):
            path = work / "videos" / f"sample_{clip['sample_id']}_real.mp4"
            if path.is_file() and path.stat().st_size == clip["bytes"]:
                wanted.discard(clip["sample_id"])
        if not wanted:
            continue
        for host in ("huggingface.co", "hf-mirror.com"):
            url = f"https://{host}/datasets/mjuicem/StreamingBench/resolve/main/{quote(shard['archive'])}?download=true"
            try:
                with requests.Session() as session:
                    accessor = p.HTTPRangeFile(url, shard["archive_bytes"], session)
                    with ZipFile(accessor) as archive:
                        members = defaultdict(list)
                        for item in archive.infolist():
                            if item.is_dir() or Path(item.filename).name.startswith("._"):
                                continue
                            match = re.search(r"sample_(\d+)", item.filename)
                            if match and int(match[1]) in wanted and item.filename.lower().endswith(".mp4"):
                                members[int(match[1])].append(item.file_size)
                        if set(members) != wanted or any(len(sizes) != 1 for sizes in members.values()):
                            raise ValueError(f"Missing or ambiguous video names in {shard['shard']}")
                        remaining += sum(sizes[0] for sizes in members.values())
                break
            except (OSError, BadZipFile, p.RangeUnsupported, requests.RequestException) as exc:
                if host == "hf-mirror.com":
                    raise RuntimeError(f"Cannot read ZIP index for {shard['shard']}") from exc
                print(f"MEDIA_INDEX_RETRY {shard['shard']} {type(exc).__name__}", flush=True)
    free = shutil.disk_usage(work).free
    print(f"MEDIA_PLAN remaining_GiB={remaining / 1024**3:.2f} free_GiB={free / 1024**3:.2f}", flush=True)
    if remaining + reserve > free:
        raise RuntimeError("Insufficient free space for selected video members and the 4 GiB work-space reserve")


def candidate_windows(history, recent):
    """Return [newest, ...] complete disjoint windows. Keep short starts as recent-only."""
    if not recent:
        raise ValueError("No causal frame at this question time")
    return [recent] + [history[end-4:end] for end in range(len(history), 3, -4)]


def select(scores):
    """Ties go to the newest window. All selectors share the candidate pool."""
    if len(scores) == 1:
        return dict.fromkeys(ARMS, 0)
    if not scores or any(not math.isfinite(row[k]) for row in scores for k in PROBES):
        raise ValueError("Each selectable window requires finite scores")
    relevance = [row["relevance"] for row in scores]
    target = [row["target"] for row in scores]
    mean = [(row["target"] + row["binding"]) / 2 for row in scores]
    conjunction = [min(row["target"], row["binding"]) for row in scores]
    return dict(zip(ARMS, (0, 1, relevance.index(max(relevance)), target.index(max(target)),
                           mean.index(max(mean)), conjunction.index(max(conjunction))), strict=True))


def probe_window(model, state, question, criterion):
    """Average signed Yes-minus-No logit margins after swapping the answer letters."""
    tokenizer = model.processor.tokenizer
    label_ids = [tokenizer.encode(letter, add_special_tokens=False) for letter in ("A", "B")]
    if any(len(ids) != 1 for ids in label_ids) or label_ids[0] == label_ids[1]:
        raise ValueError("A/B must map to distinct single tokens")
    margins = []
    p.sync()
    started = time.perf_counter()
    for yes_first in (True, False):
        first, second = ("Yes", "No") if yes_first else ("No", "Yes")
        text = model.get_prompt(
            f"Question about the video: {question}\nVerification: {PROBES[criterion]}\n"
            f"Options:\n(A) {first}\n(B) {second}\nReply with only the letter A or B.") + "Answer letter:\n"
        ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        prefix_ids = ids[0].tolist()
        for letter, suffix in zip(("A", "B"), label_ids, strict=True):
            if tokenizer(text + letter).input_ids != prefix_ids + suffix:
                raise ValueError(f"Tokenizer merges the probe continuation {letter}")
        logits, _, _ = s.probe(model, state, ids, 0, 0, steps=0)
        a, b = logits[0][0, -1, [label_ids[0][0], label_ids[1][0]]].float().tolist()
        margins.append(a - b if yes_first else b - a)
        s.assert_state(model, state)
    p.sync()
    if not all(math.isfinite(value) for value in margins):
        raise ValueError("Nonfinite probe margin")
    return statistics.mean(margins), margins, time.perf_counter() - started


def evaluate_question(model, analyzer, bank, meta, video, qi):
    qa = video["conversations"][qi]
    history, recent = s.split_frames(video, qi)
    windows = candidate_windows(history, recent)
    order = list(range(len(windows)))
    random.Random(p.SEED + 31 * video["sample_id"] + qi).shuffle(order)
    scores, candidates = [{} for _ in windows], [{} for _ in windows]
    for i in order:
        indices = windows[i]
        p.sync()
        started = time.perf_counter()
        p.reset(model)
        groups = s.ingest(model, bank, indices, meta)
        state = s.snapshot(model)
        d.assert_fresh_cache(model, len(indices))
        p.sync()
        construction = time.perf_counter() - started
        raw, probe_times = {}, {}
        if len(windows) > 1:
            for criterion in PROBES:
                scores[i][criterion], raw[criterion], probe_times[criterion] = probe_window(
                    model, state, qa["question"], criterion)
        prompt = analyzer.format_mcqa_prompt(qa["question"], qa["choices"])
        answer, answer_time, trace = q.position_trace(model, state, prompt, (0, 0))
        candidates[i] = dict(window=i, indices=indices, groups=groups, scores=scores[i], order_margins=raw,
                             query_kv_bytes=q.cache_bytes(state), position_trace=trace,
                             construction_s=construction, probe_s=probe_times, answer_s=answer_time,
                             pred_choice=analyzer.extract_characters_regex(answer), pred_answer=answer)
        del state
        model.clear_cache()
    chosen = select(scores)
    if len({(c["query_kv_bytes"], c["position_trace"][0]["logical_start"]) for c in candidates}) != 1:
        raise ValueError("Candidate answer cache sizes or starting positions differ")
    identity = d.question_identity(video, qi, "window")
    for c in candidates:
        c["correct"] = int(c["pred_choice"] == identity["gold_choice"])
    rows = []
    for arm in ARMS:
        i = chosen[arm]
        c = candidates[i]
        scanned = candidates if ARM_PROBES[arm] and len(windows) > 1 else [c]
        component_s = (sum(x["construction_s"] for x in scanned)
                       + sum(x["probe_s"].get(probe, 0.0) for x in scanned for probe in ARM_PROBES[arm])
                       + c["answer_s"])
        rows.append(dict(identity, method=arm, shard=video["shard"], eligible=len(windows) > 1,
                         window=i, indices=c["indices"], pred_choice=c["pred_choice"], correct=c["correct"],
                         query_kv_bytes=c["query_kv_bytes"], component_s=component_s))
    return dict(rows=rows, windows=candidates, execution_order=order, oracle_correct=max(c["correct"] for c in candidates))


def compare(blocks, candidate, baseline):
    deltas = defaultdict(list)
    wins = losses = switches = 0
    for block in blocks:
        rows = {r["method"]: r for r in block["rows"]}
        a, b = rows[candidate], rows[baseline]
        delta = a["correct"] - b["correct"]
        deltas[a["sample_id"]].append(delta)
        wins += delta == 1
        losses += delta == -1
        switches += a["window"] != b["window"]
    return dict(candidate=candidate, baseline=baseline, wins=wins, losses=losses, window_changes=switches,
                questions=len(blocks), videos=len(deltas), **base.paired_interval(deltas))


def write_blind_review(blocks, work):
    """Queue differing selections with selector and correctness kept in a separate key."""
    rng = random.Random(p.SEED)
    review, key = [], []
    for block in blocks:
        rows = {r["method"]: r for r in block["rows"]}
        a, b = rows["conjunction"], rows["mean"]
        if a["window"] == b["window"]:
            continue
        pair = [a, b]
        rng.shuffle(pair)
        rid = f"{a['sample_id']}-q{a['qa_index']}"
        review.append(dict(review_id=rid, sample_id=a["sample_id"], qa_index=a["qa_index"], question=a["question"],
                           query_time_s=a["query_time_s"], left_indices=pair[0]["indices"], right_indices=pair[1]["indices"],
                           left_support="", right_support="", notes=""))
        key.append(dict(review_id=rid, left_method=pair[0]["method"], right_method=pair[1]["method"],
                        left_correct=pair[0]["correct"], right_correct=pair[1]["correct"]))
    if review:
        p.write_csv(work / "exports/blind_review.csv", review)
        p.write_csv(work / "exports/blind_review_key.csv", key)


def export(blocks, work, complete):
    rows = [row for block in blocks for row in block["rows"]]
    p.write_csv(work / "exports/answers.csv", rows)
    primary = [block for block in blocks if block["rows"][0]["eligible"]]
    lines = ["# Target binding under a fixed answer budget", "",
             "Within StreamingBench. Original source overlap is unverified.", "",
             "| Cohort | Selector | Correct / questions | Median component ms | p95 component ms |",
             "|---|---|---:|---:|---:|"]
    summary = []
    cohorts = {"causal_two_or_more_windows": primary, "all_questions": blocks,
               "text_rich_two_or_more_windows": [b for b in primary if b["rows"][0]["task"] == TEXT_TASK]}
    for name, cohort in cohorts.items():
        if not cohort:
            continue
        for i, arm in enumerate(ARMS):
            subset = [b["rows"][i] for b in cohort]
            times = sorted(r["component_s"] * 1000 for r in subset)
            item = dict(cohort=name, arm=arm, questions=len(subset), correct=sum(r["correct"] for r in subset),
                        older_selected=sum(r["window"] > 0 for r in subset), median_component_ms=statistics.median(times),
                        p95_component_ms=times[math.ceil(.95 * len(times)) - 1],
                        oracle_correct=sum(b["oracle_correct"] for b in cohort))
            summary.append(item)
            lines.append(f"| {name} | {arm} | {item['correct']} / {item['questions']} | "
                         f"{item['median_component_ms']:.1f} | {item['p95_component_ms']:.1f} |")
    p.write_csv(work / "exports/summary.csv", summary)
    lines.extend(["", "Component times sum serialized cache builds, required probes, and one answer from the shared replay.",
                  "They exclude frame decoding, feature extraction and stream maintenance. The oracle is diagnostic only."])
    if complete:
        effects = [compare(primary, "conjunction", baseline) for baseline in
                   ("mean", "recent", "global_relevance", "target_only", "previous")]
        p.atomic_json(work / "exports/paired.json", dict(primary=effects[0], secondary=effects[1:]))
        for result in effects:
            lines.append(f"\nConjunction vs {result['baseline']}: {result['difference_pp']:+.2f} pp, "
                         f"95% video bootstrap [{result['ci_low_pp']:+.2f}, {result['ci_high_pp']:+.2f}], "
                         f"{result['wins']} wins and {result['losses']} losses.")
        write_blind_review(primary, work)
    (work / "exports/report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def execute(args):
    import torch
    import transformers

    if (args.model.name != p.MODEL_SHA or torch.__version__ != "2.10.0+cu128"
            or transformers.__version__ != "4.45.0.dev0" or not torch.cuda.is_available()):
        raise ValueError("Pinned GPU model/runtime required")
    path = args.work / "reference.json"
    manifest = json.loads(path.read_text(encoding="utf-8"))
    videos = frozen_videos(manifest, args.repo)
    config = dict(manifest_sha256=p.digest(path), sources={n: p.digest(args.work / n) for n in SOURCE_PATHS},
                  model=p.MODEL_SHA, torch=torch.__version__, gpu=torch.cuda.get_device_name(0), arms=ARMS)
    fingerprint = d.fingerprint(config)
    config_path = args.work / "exports/run_config.json"
    if config_path.exists() and json.loads(config_path.read_text())["config_hash"] != fingerprint:
        raise d.RunConflict("Configuration changed. Preserve this run and use a new WORK directory.")
    p.atomic_json(config_path, dict(**config, config_hash=fingerprint))
    p.report_progress("indexing_selected_media")
    check_media_space(manifest["shards"], args.work)
    assets = base.acquire_media(manifest["shards"], args.work)
    for video in videos:
        video["video_path"] = assets[video["sample_id"]]["path"]
    sys.path.insert(0, str(args.repo))
    from inference.llavaov_hermes import load_model
    from video_qa.hermes_vqa import HermesVQA

    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    torch.manual_seed(p.SEED)
    p.report_progress("loading_target_binding_model")
    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=p.FPS)
    model.eval()
    analyzer = HermesVQA(anno=[], save_dir=str(args.work), sample_fps=p.FPS, qa_model=model, qa_processor=processor)
    blocks, started, warmed = [], time.perf_counter(), False
    with torch.inference_mode():
        p.preflight(model, max(videos, key=lambda v: float(v["conversations"][-1]["end_time"])), args.work)
        for i, video in enumerate(videos):
            if i and args.max_hours and time.perf_counter() - started > args.max_hours * 3600:
                break
            sid = video["sample_id"]
            directory, meta = base.prepare_confirmation_features(
                model, video, args.work, d.fingerprint(dict(config=fingerprint, source=assets[sid]["sha256"])))
            video["timestamps"] = p.read_csv(directory / "sampled_timestamps.csv")
            p.atomic_json(args.work / "provenance" / f"features_{sid}.json", dict(metadata=meta, timestamps=video["timestamps"]))
            bank = s.c.FeatureBank(directory, meta)
            for qi in range(len(video["conversations"])):
                dest = args.work / "checkpoints" / f"sample_{sid}_q{qi}.json"
                key = d.fingerprint(dict(config=fingerprint, features=d.fingerprint(meta), sample=sid, question=qi))
                if dest.exists():
                    saved = json.loads(dest.read_text(encoding="utf-8"))
                    if saved["key"] != key or d.fingerprint(saved["block"]) != saved["block_sha256"]:
                        raise d.RunConflict(f"Checkpoint differs: {dest}")
                    block = saved["block"]
                else:
                    p.report_progress("target_binding_question", sample_id=sid, qa_index=qi, completed=len(blocks))
                    if not warmed and d.prefix_end(video, qi) >= 8:
                        evaluate_question(model, analyzer, bank, meta, video, qi)
                        warmed = True
                    block = evaluate_question(model, analyzer, bank, meta, video, qi)
                    p.atomic_json(dest, dict(key=key, block=block, block_sha256=d.fingerprint(block)))
                blocks.append(block)
            del bank
            export(blocks, args.work, complete=False)
            print(f"VIDEO_BLOCK_DONE {i+1}/{len(videos)} questions={len(blocks)}/{manifest['questions']}", flush=True)
    complete = len(blocks) == manifest["questions"]
    if complete and sum(b["rows"][0]["eligible"] for b in blocks) != manifest["primary_questions"]:
        raise ValueError("Primary cohort count differs from the frozen manifest")
    status = "complete" if complete else "partial_time_budget"
    export(blocks, args.work, complete)
    p.atomic_json(args.work / "exports/completion.json", dict(status=status, main_answers=len(blocks) * len(ARMS),
                  main_questions=len(blocks), elapsed_post_model_s=time.perf_counter() - started, config_hash=fingerprint))
    p.report_progress(status, main_questions=len(blocks))
    package(args.work, status)
    print(f"TARGET_BINDING_{status.upper()}", flush=True)


def package(work, status):
    previous = s.ARTIFACT
    try:
        s.ARTIFACT = ARTIFACT
        s.package(work, status)
    finally:
        s.ARTIFACT = previous


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("repo", "work", "model", "reuse"):
        parser.add_argument("--" + name, type=lambda value: Path(value).resolve(), required=True)
    parser.add_argument("--max-hours", type=float, default=0.0)
    args = parser.parse_args()
    d.validate_workspaces(args.work, args.reuse)
    if not math.isfinite(args.max_hours) or args.max_hours < 0:
        parser.error("Invalid soft budget")
    with p.ProgressMonitor(args.work):
        try:
            execute(args)
        except d.RunConflict:
            raise
        except Exception as exc:
            p.atomic_json(args.work / "exports/failure.json", dict(type=type(exc).__name__, detail=str(exc)[-1600:]))
            p.atomic_json(args.work / "exports/completion.json", dict(status="failed"))
            package(args.work, "failed_partial")
            raise


if __name__ == "__main__":
    main()
'''
ast.parse(RUNNER_SOURCE)
assert hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest() == '0027b6c4048b0b66cbffb2124cb7a3ae35af347291b43e9d061e623ca71d2914'
REFERENCE_JSON = '{"annotation_path":"data/streamingbench/streamingbench_realtime.json","annotation_sha256":"6a2969953e11db44f5f8584d0d9fd3ec47c41dacf63807ef0003ccf446869f73","clips":60,"excluded_diagnostic_sample_ids":[35,53,54,57,59,62,63,66,69,71,72,77,80,81,82,86,89,90,91,92,93,108,109,111,112,113,114,118,120,121,123,127,131,133,135,140,146,180,189,212,218,219,222,232,277,302,305,342,349,370,381,385,427,429,431,480,487,489,493,499],"primary_questions":269,"questions":300,"scope":"New diagnostic clip IDs within StreamingBench; original source overlap unverified","seed":"HERMES-target-binding-2026-11-21-v1","selection":"Metadata-only SHA256 rank, reserved text-rich clips by shard, then rank-fill","shards":[{"archive":"Real-Time Visual Understanding_101-150.zip","archive_bytes":8218047488,"archive_sha256":"6fdc44e97d1888b471b2d7f55dddace16291c04fdfcaab10433b35fb315bb8b3","clips":[{"primary_count":5,"question_count":5,"sample_id":105,"text_rich_count":5,"video_basename":"sample_105_real.mp4","video_id":421},{"primary_count":4,"question_count":5,"sample_id":115,"text_rich_count":5,"video_basename":"sample_115_real.mp4","video_id":29},{"primary_count":5,"question_count":5,"sample_id":116,"text_rich_count":2,"video_basename":"sample_116_real.mp4","video_id":152},{"primary_count":4,"question_count":5,"sample_id":119,"text_rich_count":3,"video_basename":"sample_119_real.mp4","video_id":333},{"primary_count":5,"question_count":5,"sample_id":125,"text_rich_count":0,"video_basename":"sample_125_real.mp4","video_id":24},{"primary_count":4,"question_count":5,"sample_id":126,"text_rich_count":0,"video_basename":"sample_126_real.mp4","video_id":106},{"primary_count":5,"question_count":5,"sample_id":134,"text_rich_count":0,"video_basename":"sample_134_real.mp4","video_id":175},{"primary_count":5,"question_count":5,"sample_id":139,"text_rich_count":0,"video_basename":"sample_139_real.mp4","video_id":262},{"primary_count":5,"question_count":5,"sample_id":144,"text_rich_count":0,"video_basename":"sample_144_real.mp4","video_id":316},{"primary_count":4,"question_count":5,"sample_id":147,"text_rich_count":2,"video_basename":"sample_147_real.mp4","video_id":481},{"primary_count":4,"question_count":5,"sample_id":149,"text_rich_count":2,"video_basename":"sample_149_real.mp4","video_id":309},{"primary_count":4,"question_count":5,"sample_id":150,"text_rich_count":1,"video_basename":"sample_150_real.mp4","video_id":274}],"shard":"101-150"},{"archive":"Real-Time Visual Understanding_151-200.zip","archive_bytes":17984397733,"archive_sha256":"ce4c47e4161a5502f25e9fe9b7ee3929aa839b6916777e3614a81fa31def4472","clips":[{"primary_count":4,"question_count":5,"sample_id":153,"text_rich_count":0,"video_basename":"sample_153_real.mp4","video_id":208},{"primary_count":3,"question_count":5,"sample_id":156,"text_rich_count":1,"video_basename":"sample_156_real.mp4","video_id":420},{"primary_count":4,"question_count":5,"sample_id":166,"text_rich_count":0,"video_basename":"sample_166_real.mp4","video_id":100},{"primary_count":5,"question_count":5,"sample_id":169,"text_rich_count":0,"video_basename":"sample_169_real.mp4","video_id":362},{"primary_count":4,"question_count":5,"sample_id":170,"text_rich_count":0,"video_basename":"sample_170_real.mp4","video_id":429},{"primary_count":5,"question_count":5,"sample_id":173,"text_rich_count":0,"video_basename":"sample_173_real.mp4","video_id":168},{"primary_count":4,"question_count":5,"sample_id":174,"text_rich_count":0,"video_basename":"sample_174_real.mp4","video_id":49},{"primary_count":4,"question_count":5,"sample_id":176,"text_rich_count":0,"video_basename":"sample_176_real.mp4","video_id":454},{"primary_count":5,"question_count":5,"sample_id":177,"text_rich_count":0,"video_basename":"sample_177_real.mp4","video_id":123},{"primary_count":4,"question_count":5,"sample_id":179,"text_rich_count":0,"video_basename":"sample_179_real.mp4","video_id":76},{"primary_count":5,"question_count":5,"sample_id":190,"text_rich_count":2,"video_basename":"sample_190_real.mp4","video_id":214},{"primary_count":5,"question_count":5,"sample_id":197,"text_rich_count":1,"video_basename":"sample_197_real.mp4","video_id":204}],"shard":"151-200"},{"archive":"Real-Time Visual Understanding_201-250.zip","archive_bytes":2230277873,"archive_sha256":"b7946e8b678fbf704b9816fa708e6febb77f5c37a7ad884084adf858f5d20dbf","clips":[{"primary_count":4,"question_count":5,"sample_id":206,"text_rich_count":0,"video_basename":"sample_206_real.mp4","video_id":173},{"primary_count":5,"question_count":5,"sample_id":209,"text_rich_count":0,"video_basename":"sample_209_real.mp4","video_id":118},{"primary_count":4,"question_count":5,"sample_id":210,"text_rich_count":1,"video_basename":"sample_210_real.mp4","video_id":289},{"primary_count":5,"question_count":5,"sample_id":213,"text_rich_count":0,"video_basename":"sample_213_real.mp4","video_id":447},{"primary_count":5,"question_count":5,"sample_id":214,"text_rich_count":0,"video_basename":"sample_214_real.mp4","video_id":91},{"primary_count":4,"question_count":5,"sample_id":215,"text_rich_count":0,"video_basename":"sample_215_real.mp4","video_id":171},{"primary_count":5,"question_count":5,"sample_id":217,"text_rich_count":0,"video_basename":"sample_217_real.mp4","video_id":427},{"primary_count":5,"question_count":5,"sample_id":220,"text_rich_count":0,"video_basename":"sample_220_real.mp4","video_id":302},{"primary_count":4,"question_count":5,"sample_id":221,"text_rich_count":0,"video_basename":"sample_221_real.mp4","video_id":3},{"primary_count":5,"question_count":5,"sample_id":223,"text_rich_count":0,"video_basename":"sample_223_real.mp4","video_id":402},{"primary_count":4,"question_count":5,"sample_id":224,"text_rich_count":0,"video_basename":"sample_224_real.mp4","video_id":367},{"primary_count":4,"question_count":5,"sample_id":225,"text_rich_count":0,"video_basename":"sample_225_real.mp4","video_id":376},{"primary_count":4,"question_count":5,"sample_id":229,"text_rich_count":0,"video_basename":"sample_229_real.mp4","video_id":222},{"primary_count":5,"question_count":5,"sample_id":230,"text_rich_count":0,"video_basename":"sample_230_real.mp4","video_id":266},{"primary_count":5,"question_count":5,"sample_id":231,"text_rich_count":0,"video_basename":"sample_231_real.mp4","video_id":334},{"primary_count":5,"question_count":5,"sample_id":233,"text_rich_count":0,"video_basename":"sample_233_real.mp4","video_id":89},{"primary_count":5,"question_count":5,"sample_id":234,"text_rich_count":0,"video_basename":"sample_234_real.mp4","video_id":220},{"primary_count":5,"question_count":5,"sample_id":239,"text_rich_count":0,"video_basename":"sample_239_real.mp4","video_id":146},{"primary_count":5,"question_count":5,"sample_id":240,"text_rich_count":0,"video_basename":"sample_240_real.mp4","video_id":23},{"primary_count":5,"question_count":5,"sample_id":243,"text_rich_count":0,"video_basename":"sample_243_real.mp4","video_id":326},{"primary_count":4,"question_count":5,"sample_id":244,"text_rich_count":0,"video_basename":"sample_244_real.mp4","video_id":292},{"primary_count":5,"question_count":5,"sample_id":247,"text_rich_count":0,"video_basename":"sample_247_real.mp4","video_id":141},{"primary_count":5,"question_count":5,"sample_id":248,"text_rich_count":0,"video_basename":"sample_248_real.mp4","video_id":306},{"primary_count":5,"question_count":5,"sample_id":249,"text_rich_count":0,"video_basename":"sample_249_real.mp4","video_id":216}],"shard":"201-250"},{"archive":"Real-Time Visual Understanding_251-300.zip","archive_bytes":18838513104,"archive_sha256":"776f78a9338aaa72ed2e8c99a10eeced813c8499988bda29f4205c542a8a577a","clips":[{"primary_count":5,"question_count":5,"sample_id":259,"text_rich_count":1,"video_basename":"sample_259_real.mp4","video_id":411},{"primary_count":4,"question_count":5,"sample_id":261,"text_rich_count":1,"video_basename":"sample_261_real.mp4","video_id":270},{"primary_count":4,"question_count":5,"sample_id":265,"text_rich_count":1,"video_basename":"sample_265_real.mp4","video_id":236},{"primary_count":4,"question_count":5,"sample_id":266,"text_rich_count":3,"video_basename":"sample_266_real.mp4","video_id":290},{"primary_count":5,"question_count":5,"sample_id":278,"text_rich_count":0,"video_basename":"sample_278_real.mp4","video_id":69},{"primary_count":4,"question_count":5,"sample_id":282,"text_rich_count":0,"video_basename":"sample_282_real.mp4","video_id":140},{"primary_count":5,"question_count":5,"sample_id":283,"text_rich_count":0,"video_basename":"sample_283_real.mp4","video_id":41},{"primary_count":4,"question_count":5,"sample_id":284,"text_rich_count":0,"video_basename":"sample_284_real.mp4","video_id":484},{"primary_count":4,"question_count":5,"sample_id":287,"text_rich_count":0,"video_basename":"sample_287_real.mp4","video_id":164},{"primary_count":4,"question_count":5,"sample_id":291,"text_rich_count":0,"video_basename":"sample_291_real.mp4","video_id":300},{"primary_count":4,"question_count":5,"sample_id":294,"text_rich_count":1,"video_basename":"sample_294_real.mp4","video_id":272},{"primary_count":4,"question_count":5,"sample_id":300,"text_rich_count":0,"video_basename":"sample_300_real.mp4","video_id":255}],"shard":"251-300"}],"text_rich_questions":32}'
assert hashlib.sha256(REFERENCE_JSON.encode()).hexdigest() == '9dda93d48764f147d340e9842b49a8b8c7f974cfaaf8c557d21bb7b07109f665'
if not VALIDATE_ONLY:
    import fcntl
    session_lock=(WORK/'.notebook.lock').open('a')
    try:
        fcntl.flock(session_lock,fcntl.LOCK_EX | fcntl.LOCK_NB)
        for name,source in [('stage_runner.py',RUNNER_SOURCE),('provenance/confirmation_base.py',CONFIRMATION_BASE_SOURCE),('provenance/stage_base.py',STAGE_BASE_SOURCE),('provenance/matched_base.py',MATCHED_BASE_SOURCE),('provenance/short_base.py',SHORT_BASE_SOURCE),('provenance/selection_base.py',SELECTION_BASE_SOURCE),('stage_context_utils.py',CONTEXT_SOURCE),('context_input_utils.py',INPUT_SOURCE),('diagnostic_pilot_utils.py',UTILS_SOURCE)]:
            target=WORK/name
            target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists() and target.read_text(encoding='utf-8')!=source:
                raise RuntimeError('Embedded source changed: use a new WORK directory')
            target.write_text(source,encoding='utf-8',newline='\n')
        target=WORK/'reference.json'
        if target.exists() and target.read_text(encoding='utf-8')!=REFERENCE_JSON:
            raise RuntimeError('Reference changed: use a new WORK directory')
        target.write_text(REFERENCE_JSON,encoding='utf-8',newline='\n')
    except Exception:
        session_lock.close()
        raise
print('EMBEDDED_STAGE_SOURCES_PASS')


## 3. Acquire the frozen clips and run the six selectors


In [ ]:
if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        repository_revision=BASE_REVISION,runner_sha256=hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest(),
        reuse_candidate=str(REUSE),max_hours=MAX_HOURS,gpu_driver=smi.stdout.strip()),indent=2),encoding='utf-8')
    try:
        logged([PYTHON,'-u',RUNNER,'--repo',REPO,'--work',WORK,'--model',MODEL_PATH,'--reuse',REUSE,
                '--max-hours',str(MAX_HOURS)],'target_binding.log',ENV,REPO)
    finally:
        try:
            if (WORK/'exports/completion.json').is_file() and (WORK/'HERMES_target_binding_evidence.zip').is_file():
                repack="import sys,json; from pathlib import Path; sys.path.insert(0,sys.argv[1]); import stage_runner as s; w=Path(sys.argv[1]); s.package(w,json.loads((w/'exports/completion.json').read_text())['status'])"
                result=probe([PYTHON,'-c',repack,WORK],ENV,REPO)
                print(result.stdout,flush=True)
                assert result.returncode==0,result.stderr[-2000:]
        finally:
            fcntl.flock(session_lock,fcntl.LOCK_UN)
            session_lock.close()
else:
    print('VALIDATION_PASS: every stage notebook cell executed; network/install/GPU and filesystem mutations skipped')


## 4. View results and download the evidence ZIP
If the run stops, run this cell manually to download the failure evidence.


In [ ]:
if not VALIDATE_ONLY:
    from IPython.display import display, Markdown, FileLink
    completion = json.loads((WORK/'exports/completion.json').read_text(encoding='utf-8'))
    report = WORK/'exports/report.md'
    if report.exists():
        display(Markdown(report.read_text(encoding='utf-8')))
    print('Status:', completion['status'], '| Main answers:', completion.get('main_answers', 'see checkpoints'))
    if completion['status'] != 'complete':
        print('Partial results are preserved. Time-budget stops can resume. Inspect failed gates before restarting.')
    artifact = WORK/'HERMES_target_binding_evidence.zip'
    link = Path.cwd()/artifact.name
    if link.resolve() != artifact.resolve():
        if link.is_symlink():
            link.unlink()
        # Always refresh the small evidence copy, never link/copy videos or weights here.
        shutil.copy2(artifact, link)
    display(FileLink(artifact.name))
    print('Evidence:', artifact, '| MiB:', round(artifact.stat().st_size/1024**2, 2))
    print('Send this ZIP and the saved executed notebook back for analysis. Preserve asset caches for reuse.')
